# Progetto: "Non solo Giochi"
## Analisi dei fattori socio-economici e demografici nel successo olimpico

Questo notebook esplora i determinanti strutturali che influenzano la performance dei Paesi nelle Olimpiadi estive, integrando i dati storici dei medaglieri con gli indicatori macroeconomici della World Bank.

### 🎯 Domanda di Ricerca & Obiettivi
In che misura il numero di medaglie è determinato da fattori strutturali (ricchezza e demografia) rispetto a fattori qualitativi (efficienza del sistema sportivo, investimenti mirati, cultura locale)?

> **Tesi centrale:** il PIL e le variabili macroeconomiche determinano il *potenziale strutturale* di una nazione nel lungo periodo; l'inerzia del sistema sportivo (catturata dallo storico recente) ne spiega la performance nel breve; i residui del modello rivelano dove efficienza, scelte politiche e fattori non misurabili fanno la differenza tra realizzare quel potenziale o sprecarla.

Per rispondere a questo quesito, il lavoro è strutturato nelle seguenti fasi:
1. **Analisi Esplorativa dei Dati (EDA):** Studio delle distribuzioni e delle correlazioni storiche.
2. **Segmentazione (Clustering K-Means):** Raggruppamento dei Paesi in base ai profili socio-economici.
3. **Modellazione Predittiva:** Sviluppo di modelli di regressione lineare (OLS) e algoritmi non lineari (Random Forest).
4. **Analisi dell'Efficienza:** Studio dei residui per identificare i Paesi virtuosi (over-performer) e quelli inefficienti (under-performer).

Import librerie

Tutte le librerie utilizzate nel notebook vengono importate in questa prima cella, per centralizzare le dipendenze in un unico punto e rendere immediatamente leggibili gli strumenti impiegati nell'analisi. Questa scelta migliora la manutenibilità del codice ed evita import sparsi e ripetuti tra le celle successive.

Gli strumenti sono raggruppati per funzione:
* `pandas` e `numpy` per la manipolazione e i calcoli numerici;
* `matplotlib`, `seaborn`, `altair` e `plotly` per le visualizzazioni statiche e interattive;
* i moduli di `sklearn` per preprocessing, clustering, modellazione e validazione;
* `statsmodels` e `shap` per l'analisi statistica e l'interpretabilità dei modelli.

Viene inoltre configurato `warnings` per la gestione dei messaggi di avviso, insieme allo stile e alla dimensione predefinita dei grafici. Il notebook è pensato per essere eseguito in sequenza, dall'alto verso il basso.

In [ ]:
# Cella 1 : Import librerie

# Tutte le librerie sono importate qui, all'inizio del notebook, per centralizzare
# le dipendenze in un unico punto e rendere chiare fin da subito quali strumenti
# vengono usati. Il notebook va eseguito dall'alto verso il basso.

# Manipolazione dati
import pandas as pd
import numpy as np
from math import pi

# Visualizzazione
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import plotly.express as px
import plotly.graph_objects as go

# Preprocessing e clustering
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Modellazione e validazione
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    classification_report, accuracy_score, confusion_matrix
)

# Statistica e interpretabilità
import statsmodels.api as sm
import shap

# Gestione avvisi e stile dei grafici
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

alt.renderers.enable('colab')
alt.data_transformers.disable_max_rows()


Caricamento Dataset

Il dataset viene caricato con `pd.read_csv()` dal file `dataset_indicatori_definitivo_5.csv` e salvato in un DataFrame di pandas chiamato `df`.

Questo file non viene costruito in questo notebook: è il risultato di un lavoro di pre-processing esterno, condiviso tra tutti i membri del gruppo per garantire che ognuno lavori sulla stessa versione degli indicatori. Il percorso che ha portato a questa versione è il seguente:

1. **Preparazione iniziale** (`per_dataset_new.ipynb`): unione dei dati olimpici con gli indicatori socioeconomici della World Bank, con mapping dei nomi Paese tra le diverse fonti.
2. **Prima correzione**: durante la verifica dei duplicati Codice_NOC + Anno erano emerse righe doppie per Russia (1992 e 2020), causate dal fatto che gli indicatori economici restavano agganciati alla Russia anche negli anni in cui gareggiava sotto un'altra sigla olimpica (EUN nel '92, ROC nel 2020). È stata quindi introdotta una mappatura tra Codice_NOC (CIO) e codice ISO (World Bank) per evitare che i due sistemi di codifica, diversi tra loro, generassero questo tipo di ambiguità.
3. **Audit esteso**: verificando lo stesso meccanismo su tutte le nazioni storicamente divise o nate nel periodo 1964-2020, sono emerse righe fittizie analoghe anche per Cecoslovacchia → Cechia/Slovacchia, Jugoslavia → Bosnia-Erzegovina/Croazia/Macedonia del Nord/Montenegro/Slovenia/Serbia, e Serbia e Montenegro → Serbia/Montenegro separate, oltre a un problema diverso (stesso Paese, codice NOC incoerente nel tempo) per Trinidad and Tobago ed Egitto. Tutte queste correzioni sono state applicate al dataset originale, producendo `dataset_indicatori_definitivo_5.csv`.

La Cella 2 (continua) qui sotto non applica più queste correzioni — sono già a monte nel file caricato — ma **verifica** che le proprietà attese siano rispettate (nessuna riga fittizia residua, nessun duplicato).

Per un primo controllo dei dati vengono stampate le prime 3 righe con `df.head(3)` e viene usato `df.info()` per ottenere un riassunto delle colonne, dei tipi di dato e dell'eventuale presenza di valori mancanti. Questo permette di inquadrare subito la struttura del dataset.


In [ ]:
# Cella 2 : Caricamento dataset e controllo iniziale (head, codici NOC)

df = pd.read_csv('dataset_indicatori_definitivo_5.csv')

display(df.head(3).style.hide(axis='index').format({
    'Anno': '{:.0f}',
    'Medaglie_Oro': '{:.0f}', 'Medaglie_Argento': '{:.0f}',
    'Medaglie_Bronzo': '{:.0f}', 'Totale_Medaglie': '{:.0f}'
}))

# Estrazione delle coppie univoche Codice_NOC e Nazione, ordinate per Codice_NOC
paesi_e_codici = df[['Codice_NOC', 'Nazione']].drop_duplicates().sort_values(by='Codice_NOC')
#Ho fatto un controllo visivo se ci sono Nazioni associate a codici_noc con descrizione diversa, es: China e People's Republic of China
for index, row in paesi_e_codici.iterrows():
    print(f"{row['Codice_NOC']}: {row['Nazione']}")

df.info()

AFG: Afghanistan
AHO: Netherlands Antilles
ALB: Albania
ALG: Algeria
AND: Andorra
ANG: Angola
ANT: Antigua and Barbuda
ARG: Argentina
ARM: Armenia
ARU: Aruba
ASA: American Samoa
AUS: Australia
AUT: Austria
AZE: Azerbaijan
BAH: The Bahamas
BAN: Bangladesh
BAR: Barbados
BDI: Burundi
BEL: Belgium
BEN: Benin
BER: Bermuda
BHU: Bhutan
BIH: Bosnia and Herzegovina
BIZ: Belize
BLR: Belarus
BOL: Bolivia, Plurinational State of
BOT: Botswana
BRA: Brazil
BRN: Bahrain
BRU: Brunei Darussalam
BUL: Bulgaria
BUR: Burkina Faso
CAF: Central African Republic
CAM: Cambodia
CAN: Canada
CAY: Cayman Islands
CGO: Congo
CHA: Chad
CHI: Chile
CHN: People's Republic of China
CIV: Côte d'Ivoire
CMR: Cameroon
COD: Congo, Democratic Republic of the
COL: Colombia
COM: Comoros
CPV: Cabo Verde
CRC: Costa Rica
CRO: Croatia
CUB: Cuba
CYP: Cyprus
CZE: Czechia
DEN: Denmark
DJI: Djibouti
DMA: Dominica
DOM: Dominican Republic
ECU: Ecuador
EGY: Egypt
ERI: Eritrea
ESA: El Salvador
ESP: Spain
EST: Estonia
ETH: Ethiopia
EUN: Unif

In [ ]:
# Cella 2 (continua) : Verifica duplicati Codice_NOC per una stessa edizione (Anno)

# Verifica duplicati Codice_NOC per una stessa edizione (Anno)
duplicati = df[df.duplicated(subset=['Codice_NOC', 'Anno'], keep=False)]

if duplicati.empty:
    print("✓ Nessun duplicato: ogni nazione (Codice_NOC) compare una sola volta per edizione.")
else:
    n_righe = len(duplicati)
    n_casi = duplicati.groupby(['Codice_NOC', 'Anno']).ngroups
    print(f"⚠ Trovate {n_righe} righe duplicate, in {n_casi} combinazioni Codice_NOC–Anno.\n")

    conteggio = (duplicati.groupby(['Codice_NOC', 'Anno'])
                          .size()
                          .reset_index(name='Occorrenze')
                          .sort_values('Occorrenze', ascending=False))
    print("Combinazioni duplicate:")
    print(conteggio.to_string(index=False))

✓ Nessun duplicato: ogni nazione (Codice_NOC) compare una sola volta per edizione.


In [ ]:
# Cella 2 (continua) : Verifica delle correzioni NOC/nomi

# Il dataset caricato ha già le correzioni applicate a monte (vedi Cella 2 e
# per_dataset_new.ipynb). Questa cella si limita a verificarle.

# 1. Ogni Codice_NOC deve corrispondere a un solo nome di Nazione
residui = df.groupby('Codice_NOC')['Nazione'].nunique()
residui = residui[residui > 1]
print("Codici con più nomi diversi:", list(residui.index) if len(residui) else "nessuno ✓")

# 2. Nessuna riga fittizia a 0 medaglie per Germania/Russia negli anni critici
check_ger = df[(df['Codice_NOC'] == 'GER') & (df['Anno'] >= 1968) & (df['Anno'] <= 1988)]
check_rus = df[(df['Codice_NOC'] == 'RUS') & (df['Anno'].isin([1984, 1992, 2020]))]
print(f"Righe GER residue 1968-1988: {len(check_ger)} (atteso: 0)")
print(f"Righe RUS residue 1984/1992/2020: {len(check_rus)} (atteso: 0)")

# 3. Nessuna riga fittizia a 0 medaglie per Cecoslovacchia -> Cechia/Slovacchia,
#    Jugoslavia -> successori, Serbia e Montenegro -> Serbia/Montenegro
check_tch = df[(df['Codice_NOC'].isin(['CZE', 'SVK'])) & (df['Anno'] >= 1964) & (df['Anno'] <= 1992) & (df['Totale_Medaglie'] == 0)]
check_yug = df[(df['Codice_NOC'].isin(['BIH', 'CRO', 'MKD', 'MNE', 'SLO', 'SRB'])) & (df['Anno'] >= 1964) & (df['Anno'] <= 1992) & (df['Totale_Medaglie'] == 0)]
check_scg = df[(df['Codice_NOC'].isin(['SRB', 'MNE'])) & (df['Anno'] >= 1996) & (df['Anno'] <= 2004) & (df['Totale_Medaglie'] == 0)]
print(f"Righe fittizie residue Cecoslovacchia->Cechia/Slovacchia: {len(check_tch)} (atteso: 0)")
print(f"Righe fittizie residue Jugoslavia->successori: {len(check_yug)} (atteso: 0)")
print(f"Righe fittizie residue Serbia e Montenegro->separate: {len(check_scg)} (atteso: 0)")

# 4. Nessun duplicato Codice_NOC+Anno
duplicati_finali = df.duplicated(subset=['Codice_NOC', 'Anno'], keep=False).sum()
print(f"Duplicati Codice_NOC+Anno: {duplicati_finali} (atteso: 0)")


Codici con più nomi diversi: nessuno ✓
Righe GER residue 1968-1988: 0 (atteso: 0)
Righe RUS residue 1984/1992/2020: 0 (atteso: 0)
Righe fittizie residue Cecoslovacchia->Cechia/Slovacchia: 0 (atteso: 0)
Righe fittizie residue Jugoslavia->successori: 0 (atteso: 0)
Righe fittizie residue Serbia e Montenegro->separate: 0 (atteso: 0)
Duplicati Codice_NOC+Anno: 0 (atteso: 0)


### Analisi
Il dataset contiene informazioni su medaglie olimpiche e variabili macroeconomiche.
Sono presenti valori mancanti che verranno gestiti nelle fasi successive.


## Costruzione delle variabili avanzate

In questa fase vengono create nuove variabili utili per l’analisi.

In particolare:
- vengono calcolati indicatori di efficienza, come le medaglie ottenute rispetto al PIL
- viene costruito lo Score Medaglie, che assegna pesi differenti alle medaglie (oro, argento, bronzo), fornendo una misura più rappresentativa del successo olimpico

Queste variabili consentono di migliorare la qualità delle analisi successive.


In [ ]:
# Cella 3 : Costruzione Variabili Avanzate

# df['Medaglie_per_Miliardo_PIL'] = df['Totale_Medaglie'] / (df['Popolazione_Totale']/1e6)
# Medaglie_per_Miliardo_PIL: efficienza economica = medaglie totali ottenute
# per ogni miliardo di USD di PIL. Valori alti = nazione che converte bene
# le risorse economiche in medaglie.
df['Medaglie_per_Miliardo_PIL'] = df['Totale_Medaglie'] / (df['PIL_Assoluto_USD']/1e9)
# Medaglie_per_USD_PIL: medaglie per dollaro di PIL (valori molto piccoli).
# Variabile intermedia: serve solo per essere esclusa dalle correlazioni,
# la versione leggibile usata nell'analisi è Medaglie_per_Miliardo_PIL.
df['Medaglie_per_USD_PIL'] = df['Totale_Medaglie'] / df['PIL_Assoluto_USD']

df['Score_Medaglie'] = (
    df['Medaglie_Oro']*3 +
    df['Medaglie_Argento']*2 +
    df['Medaglie_Bronzo']
)


## Costruzione delle feature di lag (storico edizioni precedenti)

Come previsto dalla proposta progettuale, il successo olimpico di un Paese è influenzato anche dal suo **trend storico**, ovvero dai risultati ottenuti nelle edizioni precedenti.

Per catturare questo effetto, costruiamo due feature di lag basate su `Score_Medaglie`, calcolate per ciascuna Nazione e ordinate per anno:

- `Score_Medaglie_Lag1`: punteggio ottenuto nell'edizione olimpica immediatamente precedente (4 anni prima)
- `Score_Medaglie_MediaMobile3`: media mobile delle ultime 3 edizioni, per smussare le oscillazioni puntuali e catturare un trend più stabile

**Nota metodologica importante:** queste feature vengono calcolate *prima* del filtro temporale e della pulizia dei NaN, in modo da non perdere informazione storica alle estremità delle serie. Le righe per cui il lag non è calcolabile (prima edizione di partecipazione di una Nazione) conterranno NaN e verranno gestite nella fase di pulizia successiva, in coerenza con la strategia già adottata per le altre variabili.

**Numero di atleti partecipanti:** il dataset non contiene questa colonna. Resta un limite dichiarato dell'analisi (sezione "Limiti dello studio").

In [ ]:
# Cella 4 : Costruzione Feature di Lag (storico edizioni precedenti)

# Ordiniamo per Nazione e Anno per garantire che lag e rolling siano calcolati correttamente
df = df.sort_values(by=['Nazione', 'Anno']).reset_index(drop=True)

# Lag 1: Score Medaglie nell'edizione precedente della stessa Nazione
df['Score_Medaglie_Lag1'] = df.groupby('Nazione')['Score_Medaglie'].shift(1)

# Media mobile delle ultime 3 edizioni (trend storico più stabile)
# shift(1) per evitare di includere l'edizione corrente nel calcolo (no leakage)
df['Score_Medaglie_MediaMobile3'] = (
    df.groupby('Nazione')['Score_Medaglie']
      .shift(1)
      .rolling(window=3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

print(f"Righe totali: {len(df)}")
print(f"Righe con Lag1 disponibile: {df['Score_Medaglie_Lag1'].notna().sum()}")
print(f"Righe con Media Mobile 3 disponibile: {df['Score_Medaglie_MediaMobile3'].notna().sum()}")
print("\nEsempio per una Nazione (prime righe):")
display(df[df['Nazione']=='Italy'][['Nazione','Anno','Score_Medaglie',
    'Score_Medaglie_Lag1','Score_Medaglie_MediaMobile3']].head(8)
    .style.hide(axis='index').format({
        'Anno': '{:.0f}',
        'Score_Medaglie': '{:.1f}',
        'Score_Medaglie_Lag1': '{:.1f}',
        'Score_Medaglie_MediaMobile3': '{:.1f}'
    }))

Righe totali: 3058
Righe con Lag1 disponibile: 2840
Righe con Media Mobile 3 disponibile: 3057

Esempio per una Nazione (prime righe):


## Preparazione del dataset

Il dataset è stato filtrato al fine di selezionare un intervallo temporale coerente con la disponibilità dei dati economici.

Successivamente, sono state considerate solo le osservazioni complete rispetto alle variabili principali (PIL, popolazione e performance olimpica), al fine di garantire la qualità delle analisi.

## Scelta del periodo di analisi

L’analisi è stata estesa a partire dal 1964 al fine di aumentare la numerosità del campione e migliorare la robustezza statistica dei modelli.

Tuttavia, i dati relativi ai decenni precedenti risultano meno completi e caratterizzati da maggiore variabilità rispetto al periodo successivo al 1980.

Per questo motivo, è stato effettuato un confronto tra i due intervalli temporali, evidenziando un trade-off tra quantità e qualità dei dati.

## Confronto tra diverse soglie temporali

Prima di procedere con le analisi, viene effettuato un confronto tra due possibili intervalli temporali (1964–oggi e 1980–oggi).

L’obiettivo è valutare il trade-off tra numerosità del campione e qualità dei dati, confrontando sia il numero di osservazioni sia la stabilità della relazione tra PIL e performance olimpica.

Questo passaggio consente di giustificare la scelta del periodo utilizzato nelle analisi successive.

In [ ]:
# Cella 5 : Confronto tra due soglie temporali

df_1980 = df[df['Anno'] >= 1980]
df_1964 = df[df['Anno'] >= 1964]

print("Numero osservazioni dal 1964:", len(df_1964))
print("Numero osservazioni dal 1980:", len(df_1980))

# Confronto correlazione PIL - medaglie
corr_1964 = df_1964['PIL_Assoluto_USD'].corr(df_1964['Score_Medaglie'])
corr_1980 = df_1980['PIL_Assoluto_USD'].corr(df_1980['Score_Medaglie'])

print("Correlazione 1964:", corr_1964)
print("Correlazione 1980:", corr_1980)

print("\nDifferenza osservazioni:", len(df_1964) - len(df_1980))

Numero osservazioni dal 1964: 3058
Numero osservazioni dal 1980: 2254
Correlazione 1964: 0.6834232361345347
Correlazione 1980: 0.7154901312897869

Differenza osservazioni: 804


L’estensione dell’analisi al periodo 1964–oggi consente di aumentare la numerosità del campione, ma introduce una maggiore eterogeneità storica.

Di conseguenza, i risultati vengono interpretati tenendo conto del trade-off tra robustezza statistica e comparabilità temporale.

L'estensione dell'orizzonte temporale al 1964 ha permesso di recuperare 872 osservazioni aggiuntive (+35% di base dati), fondamentali per addestrare modelli non lineari complessi come il Random Forest.

Il coefficiente di correlazione tra PIL e Score Medaglie subisce una leggera flessione (da 0.73 a 0.69) estendendo l'analisi al 1964. Questo calo è coerente con una maggiore eterogeneità storica del campione.

La causa esatta del calo (es. dinamiche della Guerra Fredda, finanziamenti sportivi sproporzionati rispetto al PIL) è un'**ipotesi interpretativa plausibile**, non dimostrabile statisticamente con le variabili disponibili.

La scelta finale del 1964 rappresenta comunque il trade-off ottimale tra profondità storica, volume dei dati per il Machine Learning e robustezza statistica.


## Gestione avanzata dei valori mancanti

Per garantire la qualità delle analisi, è stata effettuata una gestione strutturata dei valori mancanti.

Le variabili socio-economiche sono state interpolate linearmente su base temporale, per ciascun Paese, al fine di preservare la continuità delle serie storiche.

Le variabili con una quota eccessiva di dati mancanti sono state rimosse, mentre le osservazioni residue incomplete sono state eliminate.

Questo processo consente di ottenere un dataset coerente e adatto alle successive analisi modellistiche.


In [ ]:
# Cella 6 : Prepariamo per filtro pulizia NaN

# Si filtra il dataset per considerare solo le Olimpiadi dall'era moderna in poi
# (es. dal 1964, o dall'anno in cui i dati economici diventano più densi)
anno_inizio_analisi= 1964
df_moderno = df[df['Anno'] >= anno_inizio_analisi].copy()

# Il dropna() viene applicato solo sulle colonne chiave in modo molto più mirato
features_analisi = ['PIL_Assoluto_USD', 'Popolazione_Totale', 'Score_Medaglie']
df_analisi = df_moderno.dropna(subset=features_analisi).copy()


In [ ]:
# Cella 7 : Gestione Avanzata e Sicura dei Valori Mancanti

# 1. Ordinamento logico
df_analisi = df_analisi.sort_values(by=['Nazione', 'Anno'])

# 2. Interpolazione limitata
# Invece di interpolare all'infinito, si limita a un massimo di 3 anni di buco.
# Se il buco è più grande, è meglio lasciare NaN che inventare dati.
colonne_da_interpolare = [
    # Feature già presenti nel modello
    'Iscrizioni_Scuola_Primaria_perc',
    'Popolazione_Urbana',
    'Aspettativa_di_Vita',

    # Feature macroeconomiche
    'Spesa_Militare_perc_PIL',
    'Investimenti_Diretti_Esteri_perc_PIL',
    'Valore_Aggiunto_Industria_perc_PIL',
    'Valore_Aggiunto_Servizi_perc_PIL',

    # Feature demografiche e di benessere
    'Tasso_Urbanizzazione_perc',
    'Tasso_Mortalita_Infantile',
    'Densita_Popolazione'
]

# I lag non vengono interpolati: un NaN indica una prima partecipazione, non un dato mancante.
# Vengono gestiti con imputazione a 0 + flag Ha_Storico_Precedente nella fase di modellazione.

for col in colonne_da_interpolare:
    if col in df_analisi.columns:
        df_analisi[col] = df_analisi.groupby('Nazione')[col].transform(
            lambda group: group.interpolate(method='linear', limit=3, limit_direction='forward')
        )

# 3. Imputazione conservativa per i rimanenti
# Per le colonne fondamentali, se resta qualche NaN, viene usato la mediana del gruppo Nazione
# (molto più robusto della media globale)
for col in colonne_da_interpolare:
    if col in df_analisi.columns:
        df_analisi[col] = df_analisi.groupby('Nazione')[col].transform(lambda x: x.fillna(x.median()))

# 4. Drop basato sulla qualità della colonna
soglia_minima_dati_validi = len(df_analisi) * 0.60
df_analisi = df_analisi.dropna(thresh=soglia_minima_dati_validi, axis=1)

# 5. Drop finale riga-per-riga (Solo per le colonne target che devono essere perfette)
# Vengono eliminate le righe dove mancano i pilastri: PIL, Popolazione, Score Medaglie
pilastri = ['PIL_Assoluto_USD', 'Popolazione_Totale', 'Score_Medaglie']
df_analisi = df_analisi.dropna(subset=pilastri)

print(f"Righe finali dopo pulizia intelligente: {len(df_analisi)}")

Righe finali dopo pulizia intelligente: 2619


## Analisi dei risultati: Strategia di Pulizia dei Dati

Il passaggio da 3.058 a 2.619 righe corrisponde alla conservazione di circa l'**85,7% del dataset di partenza** (2.619 / 3.058), eliminando solo le osservazioni le cui informazioni mancanti non erano recuperabili in modo affidabile.

Rispetto alla versione precedente del notebook (dataset non ancora corretto, 3.368 → 2.628 righe, 78% di retention), la percentuale di dati conservati è più alta: il dataset di partenza è già più pulito (senza le righe fittizie di Germania, Russia, Cecoslovacchia, Jugoslavia, Serbia-Montenegro, Trinidad and Tobago ed Egitto), quindi la pulizia per valori mancanti interviene su una base già più solida e scarta una quota minore di osservazioni.

Questo risultato conferma l'efficacia della strategia adottata: interpolazione limitata a 3 anni di buco per le variabili socio-economiche, eliminazione delle sole osservazioni realmente irrecuperabili.


## Analisi delle determinanti macroeconomiche

In questa fase è stata calcolata la matrice di correlazione tra le variabili numeriche al fine di isolare i driver principali della performance olimpica. Per preservare l'integrità dell'analisi, sono state preventivamente rimosse le variabili di natura "sportiva" (es. conteggio medaglie), le quali, costituendo componenti dirette del target, avrebbero introdotto una distorsione per autocorrelazione strutturale.

In [ ]:
# Cella 8 : Analisi delle correlazioni macroeconomiche

# 1. Si definiscono le variabili sportive dirette che potrebbero inquinare l'analisi
# (essendo somme o componenti dello Score_Medaglie avrebbero correlazione ~0.99)
colonne_sportive_da_escludere = [
    'Medaglie_Oro',
    'Medaglie_Argento',
    'Medaglie_Bronzo',
    'Totale_Medaglie',
    'Medaglie_per_Miliardo_PIL',
    'Medaglie_per_USD_PIL'
]

# 1b. Si escludono variabili non adatte a questa analisi esplorativa:
# - versioni ridondanti già presenti in altra scala (log, miliardi, _model)
# - Cluster_SocioEconomico: è un output del K-Means (etichetta 0/1/2), non un
#   determinante di input; correlarla con lo Score sarebbe circolare, ed essendo
#   categorica nominale la correlazione di Pearson non avrebbe significato.
colonne_da_escludere_extra = [
    'Score_Medaglie_Lag1_model',
    'Score_Medaglie_MediaMobile3_model',
    'PIL_Assoluto_Miliardi',
    'PIL_Log',
    'Pop_Log',
    'PIL_ProCapite_Log',
    'PIL_x_Pop',
    'Cluster_SocioEconomico'
]

# 2. Si rimuovono queste colonne dal dataframe per il calcolo
df_macro = df_analisi.drop(
    columns=colonne_sportive_da_escludere + colonne_da_escludere_extra,
    errors='ignore'
)
# 3. Si calcola la matrice di correlazione
corr = df_macro.corr(numeric_only=True)

# 4. Si estraggono le top 10 correlazioni con lo Score Medaglie
# (viene usato .drop per togliere lo 'Score_Medaglie' stesso che avrebbe correlazione 1.0)
top_corr = corr[['Score_Medaglie']].drop('Score_Medaglie').sort_values(by='Score_Medaglie', ascending=False).head(10)

# 5. Visualizzazione
plt.figure(figsize=(8, 6))

# Barplot orizzontale: la lunghezza della barra rende immediata la lettura.
# hue + legend=False applica la palette senza il warning di seaborn sulle versioni recenti.
sns.barplot(
    x=top_corr['Score_Medaglie'],
    y=top_corr.index,
    hue=top_corr.index,
    palette='viridis',
    legend=False
)

plt.title("Top 10 Determinanti dello Score Medaglie", fontsize=14)
plt.xlabel("Correlazione con Score Medaglie", fontsize=12)
plt.ylabel("Variabile Macroeconomica", fontsize=12)
plt.xlim([-1, 1])  # Range matematico di Pearson
plt.tight_layout()
plt.show()

## 📊 Analisi delle Correlazioni Macroeconomiche

L’analisi delle correlazioni evidenzia in modo chiaro i principali determinanti strutturali del successo olimpico, misurato tramite lo *Score Medaglie*.  
Per garantire una valutazione corretta sono state escluse: le variabili direttamente legate alle medaglie (oro, argento, bronzo, totale), che introdurrebbero correlazioni spurie per la loro dipendenza matematica dal target; le versioni log-trasformate e i duplicati a scala diversa delle stesse grandezze (es. PIL grezzo e PIL log), per non rappresentare due volte la stessa variabile; le etichette dei cluster, che sono un output del modello e non un determinante di input.

### 🔍 Risultati principali

Le variabili con le correlazioni più elevate risultano:

- **Spesa pubblica per consumi (~0.73)**
- **PIL assoluto (~0.70)**

Questi valori indicano una forte relazione positiva tra capacità economica e successo olimpico. È importante sottolineare che la correlazione misura il **potenziale olimpico** di una nazione, non il suo risultato garantito: il PIL dice quanto una nazione *può* vincere, non quanto *vince*. Questa distinzione — potenziale vs realizzazione — è il filo conduttore di tutta l'analisi successiva.

Seguono variabili legate alla dimensione del Paese:
- Popolazione urbana (~0.59)
- Superficie (~0.50)
- Popolazione totale (~0.38)

Al contrario, il **PIL pro capite (~0.15–0.22)** mostra una correlazione significativamente più debole. Questo conferma che non è il benessere individuale a determinare il potenziale olimpico, ma la **massa di risorse aggregata** che uno Stato può mobilitare.



## Segmentazione per Reddito


### 1. Segmentazione per Reddito Pro Capite

Creiamo una nuova colonna chiamata `Fascia_Reddito_Pro_Capite` nel DataFrame `df_analisi` e visualizziamo il primo grafico.
  - Utilizza la funzione `pd.qcut()` per dividere i paesi in quattro (`q=4`) fasce di reddito pro capite (PIL pro capite in USD), assicurando che ogni fascia contenga circa lo stesso numero di osservazioni (quantili).
  - Le fasce sono etichettate come 'Basso', 'Medio-Basso', 'Medio-Alto' e 'Alto'.

### 2. Segmentazione per Reddito Totale (PIL Assoluto)

 In modo simile alla segmentazione per reddito pro capite, questa riga crea una nuova colonna `Fascia_Reddito_Totale`.
  - Questa volta, la divisione in quattro fasce è basata sul PIL assoluto del paese (`PIL_Assoluto_USD`) e visualizzamo il secondo grafico.


In sintesi, questa cella genera due box plot per analizzare come il successo olimpico (misurato in medaglie totali) sia correlato sia con la ricchezza pro capite (benessere del cittadino) sia con la ricchezza totale (dimensione economica complessiva) di un paese, dividendo i paesi in quattro gruppi per ciascun indicatore di reddito.



In [ ]:
# Cella 9 : Segmentazione per Reddito Pro Capite e Reddito Totale

# Segmentazione per Reddito Pro Capite
df_analisi['Fascia_Reddito_Pro_Capite'] = pd.qcut(df_analisi['PIL_Pro_Capite_USD'], q=4,
                               labels=['Basso','Medio-Basso','Medio-Alto','Alto'])

plt.figure(figsize=(12,6))
sns.boxplot(data=df_analisi, x='Fascia_Reddito_Pro_Capite', y='Totale_Medaglie', hue='Fascia_Reddito_Pro_Capite', palette='viridis', legend=False)
plt.title('Distribuzione Medaglie per Fascia di Reddito Pro Capite', fontsize=14, fontweight='bold')
plt.xlabel('Fascia di Reddito Pro Capite', fontsize=12)
plt.ylabel('Totale Medaglie Olimpiche', fontsize=12)
plt.show()

# Segmentazione per Reddito Totale (PIL Assoluto)
df_analisi['Fascia_Reddito_Totale'] = pd.qcut(df_analisi['PIL_Assoluto_USD'], q=4,
                               labels=['Basso','Medio-Basso','Medio-Alto','Alto'])

plt.figure(figsize=(12,6))
sns.boxplot(data=df_analisi, x='Fascia_Reddito_Totale', y='Totale_Medaglie', hue='Fascia_Reddito_Totale', palette='magma', legend=False)
plt.title('Distribuzione Medaglie per Fascia di Reddito Totale', fontsize=14, fontweight='bold')
plt.xlabel('Fascia di Reddito Totale', fontsize=12)
plt.ylabel('Totale Medaglie Olimpiche', fontsize=12)
plt.show()

## Versione interattiva: boxplot con tooltip sugli outlier (Cella 10)

I punti **arancioni** oltre i whiskers mostrano Nazione e Anno al passaggio del mouse.
I box sono gli stessi grafici sopra, ricostruiti in Altair per il sito web.

In [ ]:
# Cella 10 : boxplot interattivi con tooltip sugli outlier


def get_outliers(df, fascia_col, value_col='Totale_Medaglie'):
    risultati = []
    for fascia, group in df.groupby(fascia_col, observed=True):
        q1 = group[value_col].quantile(0.25)
        q3 = group[value_col].quantile(0.75)
        soglia = q3 + 1.5 * (q3 - q1)
        out = group[group[value_col] > soglia].copy()
        out['Anno'] = out['Anno'].astype(int)
        risultati.append(out)
    return pd.concat(risultati) if risultati else pd.DataFrame()

order = ['Basso', 'Medio-Basso', 'Medio-Alto', 'Alto']

def build_boxplot(df_full, outliers_df, fascia_col, titolo, box_color):
    box = alt.Chart(df_full).mark_boxplot(
        outliers=False,
        color=box_color
    ).encode(
        x=alt.X(f'{fascia_col}:N', sort=order, title=fascia_col.replace('_', ' ')),
        y=alt.Y('Totale_Medaglie:Q', title='Totale Medaglie')
    ).properties(width=480, height=340)

    punti = alt.Chart(outliers_df).mark_circle(
        size=80, opacity=0.85, color='#D55E00'
    ).encode(
        x=alt.X(f'{fascia_col}:N', sort=order),
        y=alt.Y('Totale_Medaglie:Q'),
        tooltip=[
            alt.Tooltip('Nazione:N',         title='Nazione'),
            alt.Tooltip('Anno:O',            title='Anno'),
            alt.Tooltip('Totale_Medaglie:Q', title='Totale Medaglie', format='.0f'),
            alt.Tooltip(f'{fascia_col}:N',   title='Fascia')
        ]
    ).properties(width=480, height=340)

    return (box + punti).properties(title=titolo)

outliers_pc  = get_outliers(df_analisi, 'Fascia_Reddito_Pro_Capite')
outliers_tot = get_outliers(df_analisi, 'Fascia_Reddito_Totale')

chart_pc = build_boxplot(
    df_analisi, outliers_pc, 'Fascia_Reddito_Pro_Capite',
    'Medaglie per Fascia Reddito Pro Capite — passa il mouse sui punti arancioni',
    '#2E75B6'
)
chart_tot = build_boxplot(
    df_analisi, outliers_tot, 'Fascia_Reddito_Totale',
    'Medaglie per Fascia Reddito Totale — passa il mouse sui punti arancioni',
    '#009E73'
)

chart_pc.display()
chart_tot.display()

chart_pc.save('boxplot_reddito_pro_capite_interattivo.html')
chart_tot.save('boxplot_reddito_totale_interattivo.html')


## 📊 Segmentazione per Livello di Reddito: Analisi Comparativa

Per approfondire il ruolo della ricchezza economica nel successo olimpico, i Paesi sono stati segmentati in quattro fasce (quantili) sulla base di:
- **PIL pro capite** (benessere medio individuale)
- **PIL assoluto** (dimensione economica complessiva)

Questa segmentazione consente di confrontare la distribuzione delle medaglie tra Paesi con caratteristiche economiche differenti.

---

### 🔹 Distribuzione per Reddito Pro Capite

L’analisi per PIL pro capite evidenzia una distribuzione fortemente asimmetrica:

- Le fasce **basse e intermedie** presentano mediane prossime allo zero, indicando che la maggior parte dei Paesi ottiene poche o nessuna medaglia
- Tuttavia, sono presenti numerosi **outlier**, cioè Paesi che, pur non essendo particolarmente ricchi pro capite, ottengono performance olimpiche elevate
- La fascia **alta** mostra una mediana più elevata e una maggiore dispersione, suggerendo una maggiore probabilità di successo

✅ Interpretazione:
Il benessere individuale favorisce il successo, ma **non è una condizione sufficiente** per ottenere performance elevate.

---

### 🔹 Distribuzione per Reddito Totale (PIL Assoluto)

Quando la segmentazione è basata sul PIL assoluto, emerge una struttura molto più marcata:

- La fascia **Alta** domina nettamente in termini di medaglie, con una distribuzione più ampia e valori significativamente superiori
- Le fasce inferiori risultano **quasi completamente piatte**, con pochi outlier e livelli di performance estremamente ridotti

✅ Interpretazione:
La dimensione economica complessiva rappresenta un determinante molto più forte rispetto al reddito pro capite.

---

### 🎯 Insight Chiave

> Il successo olimpico è determinato principalmente dalla scala delle risorse disponibili:  
> mentre il reddito pro capite rappresenta una condizione favorevole, è il PIL assoluto a fare la differenza.

---

### ⚠️ Nota metodologica

La presenza di outlier evidenzia che esistono Paesi in grado di ottenere risultati elevati anche con minori risorse, suggerendo il ruolo di fattori qualitativi (specializzazione sportiva, tradizione, efficienza del sistema sportivo) che verranno analizzati successivamente.

---

### 🔥 Collegamento con le analisi precedenti

I risultati sono coerenti con quanto emerso nell’analisi delle correlazioni:
- il **PIL assoluto** mostra una relazione più forte con lo Score Medaglie
- il **PIL pro capite** ha un ruolo più debole e meno strutturale

Questo rafforza l’interpretazione del successo olimpico come fenomeno prevalentemente **strutturale e non individuale**.

Questa cella genera due grafici a dispersione (scatterplot) per visualizzare la relazione tra due diverse metriche economiche (PIL Assoluto e PIL Pro Capite) e il successo olimpico (Totale Medaglie).

### 1. Primo Grafico: Relazione tra PIL Assoluto e Totale Medaglie
Esplora come la ricchezza complessiva di un paese influenzi il numero di medaglie vinte.

### 2. Secondo Grafico: Relazione tra PIL Pro Capite e Totale Medaglie
Analizza come la ricchezza media per abitante di un paese si relaziona con il numero di medaglie.

L'asse X è in scala logaritmica (`plt.xscale('log')`), perché il PIL assume valori molto diversi tra loro (da piccoli a enormi) e la scala logaritmica permette di visualizzare meglio le relazioni in presenza di grandi differenze. In questo modo è più facile individuare tendenze o raggruppamenti.

I due grafici offrono quindi due prospettive complementari sulla relazione tra la ricchezza di un paese e il suo successo olimpico.

In [ ]:
# Cella 11 : Grafici PIL/Medaglie (Interattivi con Tooltip)

# ==============================================================================
# OPERAZIONE PRELIMINARE: Scaling del PIL in Miliardi
# ==============================================================================
# Dividiamo per 1 miliardo per ripulire l'output statistico ed evitare instabilità numerica
df_analisi['PIL_Assoluto_Miliardi'] = df_analisi['PIL_Assoluto_USD'] / 1_000_000_000

# Primo grafico: PIL Assoluto (in Miliardi) vs Totale Medaglie
fig1 = px.scatter(
    data_frame=df_analisi,
    x='PIL_Assoluto_Miliardi', # <--- Usiamo la nuova colonna scalata qui
    y='Totale_Medaglie',
    log_x=True,
    trendline="ols",
    hover_name='Nazione',
    hover_data={'PIL_Assoluto_Miliardi': ':.2f', 'Totale_Medaglie': True}, # <--- Formattato con 2 decimali
    title='Relazione tra PIL Assoluto e Totale Medaglie Olimpiche (Interattivo)',
    labels={
        'PIL_Assoluto_Miliardi': 'PIL Assoluto (Miliardi de USD) - Scala Logaritmica',
        'Totale_Medaglie': 'Totale Medaglie Olimpiche'
    },
    template='plotly_white'
)
fig1.show()

# Analisi della regressione (statistiche per PIL Assoluto)
results_PIL = px.get_trendline_results(fig1)

model = results_PIL.iloc[0]["px_fit_results"]
print("=== REGRESSIONE PIL ASSOLUTO (IN MILIARDI) ===")
print(model.summary())

# Secondo grafico: PIL Pro Capite vs Totale Medaglie
# (Questo rimane invariato perché i valori pro capite sono già nell'ordine delle migliaia)
fig2 = px.scatter(
    data_frame=df_analisi,
    x='PIL_Pro_Capite_USD',
    y='Totale_Medaglie',
    log_x=True,
    trendline="ols",
    hover_name='Nazione',
    hover_data={'PIL_Pro_Capite_USD': ':,d', 'Totale_Medaglie': True},
    title='Relazione tra PIL Pro Capite e Totale Medaglie Olimpiche (Interattivo)',
    labels={
        'PIL_Pro_Capite_USD': 'PIL Pro Capite (USD) - Scala Logaritmica',
        'Totale_Medaglie': 'Totale Medaglie Olimpiche'
    },
    template='plotly_white'
)
fig2.show()

# Analisi della regressione (statistiche per PIL Pro Capite)
results_PIL_PRO = px.get_trendline_results(fig2)

model = results_PIL_PRO.iloc[0]["px_fit_results"]
print("\n=== REGRESSIONE PIL PRO CAPITE ===")
print(model.summary())


## 📈 Relazione tra PIL e Successo Olimpico: Analisi Bivariata

Per analizzare il legame tra dimensione economica e performance olimpica, sono stati costruiti due scatter plot che mettono in relazione il numero totale di medaglie con:
- il **PIL assoluto** (scala delle risorse)
- il **PIL pro capite** (benessere medio individuale)

È stata utilizzata una **scala logaritmica sull’asse X** per gestire l’elevata eterogeneità dei valori economici e migliorare la leggibilità delle relazioni.

---

### 🔹 1. PIL Assoluto vs Medaglie

Il primo grafico evidenzia una relazione positiva chiara tra il PIL assoluto e il numero di medaglie.

- Il modello di regressione lineare mostra un coefficiente di determinazione pari a:
  
  👉 **R² ≈ 0.49**

- Questo indica che circa il **49% della variabilità del successo olimpico** è spiegata dalla dimensione economica complessiva.

- La relazione appare fortemente **non lineare**, con una crescita più che proporzionale per i Paesi economicamente più grandi.

✅ Interpretazione:

> Il PIL assoluto rappresenta un forte determinante strutturale del successo olimpico, in quanto riflette la capacità di investire massicciamente in sport, infrastrutture e sviluppo degli atleti.

---

### 🔹 2. PIL Pro Capite vs Medaglie

Il secondo grafico mostra una relazione molto più debole tra PIL pro capite e numero di medaglie.

- Il modello di regressione evidenzia:

  👉 **R² ≈ 0.026**

- Ciò significa che il PIL pro capite spiega **solo il 2.6% della variabilità**.

- La dispersione dei punti è elevata e non emerge una struttura chiara.

✅ Interpretazione:

> Il benessere medio del cittadino ha un impatto limitato sul successo olimpico e non rappresenta un fattore determinante.

---

### ⚖️ Confronto tra le due metriche

Il confronto tra i due modelli evidenzia una differenza sostanziale nel potere esplicativo:

| Variabile            | R²     | Impatto |
|---------------------|--------|--------|
| PIL Assoluto         | ~0.49  | Alto   |
| PIL Pro Capite       | ~0.03  | Molto basso |

✅ Insight chiave:

> La dimensione economica complessiva è molto più rilevante rispetto alla ricchezza media individuale nel determinare il successo olimpico.

---

### ⚠️ Osservazioni modellistiche

- La distribuzione dei dati presenta una forte **asimmetria (skewness)**, con una concentrazione di osservazioni vicino allo zero
- I test statistici indicano una presenza significativa di **outlier**, cioè Paesi che sovraperformano rispetto al proprio livello economico
- La relazione non è perfettamente lineare, suggerendo la presenza di **effetti non lineari e interazioni** (confermati successivamente dal modello Random Forest)

---

### 🎯 Insight Strategico

> Il PIL determina il potenziale olimpico, ma non lo esaurisce: una quota significativa del fenomeno resta spiegata da fattori qualitativi come organizzazione, cultura sportiva e specializzazione.

---

### 🔗 Collegamento con le altre analisi

Questi risultati confermano quanto emerso nelle analisi precedenti:
- le correlazioni evidenziano il ruolo dominante del PIL
- la segmentazione mostra una forte polarizzazione legata alla dimensione economica

L’analisi bivariata rappresenta quindi un passaggio fondamentale verso la modellazione predittiva.

### Efficienza Olimpica

L’analisi dell’efficienza olimpica consente di valutare la capacità dei Paesi di trasformare le risorse economiche disponibili in risultati sportivi.

A differenza delle analisi precedenti, focalizzate sulla dimensione economica assoluta, questa sezione introduce una misura relativa di performance, considerando il rapporto tra medaglie ottenute e risorse disponibili (PIL).

In particolare, viene analizzata la relazione tra:
- **Popolazione totale**, che rappresenta il potenziale bacino di talenti
- **Medaglie per miliardo di PIL**, che misura l’efficienza nell’utilizzo delle risorse

L’utilizzo della scala logaritmica per la popolazione consente di gestire l’ampia variabilità tra Paesi, rendendo più chiari eventuali pattern e cluster nei dati.



In [ ]:
# Cella 12 : Efficienza Olimpica

# ==============================================================================
# 1. PREPARAZIONE DATI
# ==============================================================================
# Stessa logica della Cella 13 (nazioni meno efficienti), per coerenza:
# - solo Paesi con almeno 5 medaglie, per escludere i picchi episodici di chi
#   ha vinto 1-2 medaglie in una singola edizione con un PIL ridotto
# - efficienza = media storica per nazione, non il valore di una singola edizione

df_filtrato_eff = df_analisi[df_analisi['Totale_Medaglie'] >= 5].copy()

efficienza_nazioni_top = (
    df_filtrato_eff
    .groupby('Nazione')
    .agg({
        'Medaglie_per_Miliardo_PIL': 'mean',
        'Popolazione_Totale': 'mean'
    })
    .reset_index()
    .sort_values(by='Medaglie_per_Miliardo_PIL', ascending=False)
)

# ==============================================================================
# 2. GRAFICO
# ==============================================================================

fig = px.scatter(
    data_frame=efficienza_nazioni_top,
    x='Popolazione_Totale',
    y='Medaglie_per_Miliardo_PIL',
    log_x=True,
    hover_name='Nazione',
    hover_data={
        'Popolazione_Totale': ':,d',
        'Medaglie_per_Miliardo_PIL': ':.2f'
    },
    title='Efficienza Olimpica: Popolazione vs Medaglie per Miliardo (Interattivo)',
    labels={
        'Popolazione_Totale': 'Popolazione Totale (Scala Logaritmica)',
        'Medaglie_per_Miliardo_PIL': 'Medaglie per Miliardo di PIL'
    },
    template='plotly_white'
)

fig.show()

# ==============================================================================
# 3. TABELLA OUTPUT
# ==============================================================================

print("Top 25 Nazioni per Efficienza Olimpica (media storica, min. 5 medaglie):")

display(
    efficienza_nazioni_top[['Nazione', 'Medaglie_per_Miliardo_PIL']]
    .head(25)
    .reset_index(drop=True)
    .style.hide(axis='index')
    .format({'Medaglie_per_Miliardo_PIL': '{:.2f}'})
)

## Analisi dell'Efficienza Olimpica

Il grafico mette in relazione la dimensione demografica dei Paesi con la loro efficienza olimpica, misurata come numero di medaglie per miliardo di PIL (media storica, considerando solo i Paesi con almeno 5 medaglie per escludere i picchi episodici).

### Distribuzione generale

La distribuzione evidenzia una forte concentrazione di osservazioni in prossimità dello zero: la maggior parte dei Paesi presenta livelli di efficienza bassi, mentre solo un numero limitato mostra valori elevati. L'efficienza olimpica risulta quindi un fenomeno raro e altamente concentrato.

### Assenza di relazione forte con la popolazione

Non emerge una relazione lineare chiara tra popolazione ed efficienza: Paesi molto popolosi non sono necessariamente più efficienti, e Paesi piccoli possono raggiungere livelli di efficienza elevati. La dimensione demografica, da sola, non garantisce efficienza ma solo potenziale.

### Paesi ad alta efficienza

Dalla classifica emergono alcuni Paesi con livelli di efficienza particolarmente elevati, tra cui Kenya, Ungheria, Giamaica, Cuba ed Etiopia. Si tratta di nazioni che ottengono risultati sportivi significativamente superiori rispetto alle risorse economiche disponibili: rappresentano casi di **over-performance strutturale**.

### Interpretazione economico-sportiva

I Paesi più efficienti condividono alcune caratteristiche ricorrenti: una forte specializzazione sportiva (ad esempio l'atletica in Kenya ed Etiopia, lo sprint in Giamaica), investimenti mirati in poche discipline e una tradizione sportiva consolidata. Concentrano risorse limitate su pochi atleti d'élite, ottenendo un rendimento elevato. Le grandi potenze economiche, al contrario, dominano in volumi assoluti ma presentano livelli di efficienza relativa più bassi, per via dell'effetto del denominatore (un PIL elevato riduce il rapporto medaglie/PIL).

### Insight chiave

Il PIL determina il potenziale olimpico, mentre l'efficienza misura la capacità di trasformare questo potenziale in risultati concreti. Questa analisi introduce la componente **qualitativa** del fenomeno e prepara il terreno per l'analisi dei residui e per l'identificazione di over-performer e under-performer.

## Analisi delle Nazioni Meno Efficienti (Efficienza Relativa)

Dopo aver analizzato i Paesi caratterizzati da elevata efficienza olimpica, è utile esaminare la parte opposta della distribuzione, ovvero le nazioni che mostrano una minore capacità di trasformare le risorse economiche in risultati sportivi.

In questa fase viene introdotto un indicatore di **efficienza relativa**, definito come il rapporto tra medaglie ottenute e PIL (medaglie per miliardo di dollari). Questo indicatore consente di confrontare Paesi con dimensioni economiche differenti, evidenziando situazioni in cui la disponibilità di risorse non si traduce in risultati proporzionati.

Per rendere l’analisi più robusta:
- vengono esclusi i Paesi con zero medaglie, per concentrarsi sulla **conversione delle risorse** e non sull’assenza di performance;
- viene applicata un’aggregazione per nazione, al fine di ottenere una misura media stabile nel tempo;
- possono essere introdotti eventuali filtri minimi sulle medaglie per ridurre la distorsione dovuta a osservazioni marginali.

È importante sottolineare che questo indicatore presenta **limiti strutturali**:
- tende a penalizzare le economie molto grandi (effetto del denominatore);
- non tiene conto delle specializzazioni sportive (es. sport invernali);
- non cattura fattori qualitativi come organizzazione sportiva e politiche pubbliche.

I risultati devono quindi essere interpretati come misura di **efficienza relativa**, e non come indicazione assoluta di inefficienza.
``

In [ ]:
# Cella 13 : Analisi delle Nazioni Meno Efficienti

# ==============================================================================
# 1. PREPARAZIONE DATI
# ==============================================================================

# Filtro: solo Paesi con almeno 1 medaglia
df_filtrato = df_analisi[df_analisi['Totale_Medaglie'] > 0].copy()

# Filtro su minimo 5 medaglie: esclude le partecipazioni episodiche (1-4 medaglie)
# che renderebbero l'indicatore di efficienza instabile e poco rappresentativo
df_filtrato = df_filtrato[df_filtrato['Totale_Medaglie'] >= 5]

# Aggregazione per nazione (media nel tempo)
efficienza_nazioni = (
    df_filtrato
    .groupby('Nazione')
    .agg({
        'Medaglie_per_Miliardo_PIL': 'mean',
        'Popolazione_Totale': 'mean'
    })
    .reset_index()
)

# Ordinamento per inefficienza (valori più bassi)
efficienza_nazioni = efficienza_nazioni.sort_values(by='Medaglie_per_Miliardo_PIL')

# Selezione delle 50 nazioni meno efficienti
df_inefficienti_plot = efficienza_nazioni.head(50)

# ==============================================================================
# 2. GRAFICO
# ==============================================================================

fig_inefficienti = px.scatter(
    data_frame=df_inefficienti_plot,
    x='Popolazione_Totale',
    y='Medaglie_per_Miliardo_PIL',
    log_x=True,
    hover_name='Nazione',
    hover_data={
        'Popolazione_Totale': ':,d',
        'Medaglie_per_Miliardo_PIL': ':.4f'
    },
    title='Efficienza Olimpica Relativa: Popolazione vs Medaglie per Miliardo',
    labels={
        'Popolazione_Totale': 'Popolazione (scala logaritmica)',
        'Medaglie_per_Miliardo_PIL': 'Medaglie per miliardo di PIL'
    },
    template='plotly_white',
    color_discrete_sequence=['red']
)

fig_inefficienti.show()

# ==============================================================================
# 3. TABELLA OUTPUT
# ==============================================================================

print("Top 25 Nazioni con minore efficienza relativa (media storica, min. 5 medaglie):")

display(
    efficienza_nazioni[['Nazione', 'Medaglie_per_Miliardo_PIL']]
    .head(25)
    .reset_index(drop=True)
    .style.hide(axis='index')
    .format({'Medaglie_per_Miliardo_PIL': '{:.4f}'})
)

## Analisi dei risultati

L'analisi delle nazioni con minore efficienza relativa evidenzia una forte eterogeneità nella capacità dei Paesi di trasformare le risorse economiche in risultati olimpici.

### 1. Effetto scala delle grandi economie

Tra le nazioni meno efficienti compaiono grandi potenze economiche come Stati Uniti, Francia, Canada e Brasile. Questo risultato non va interpretato come una reale inefficienza sportiva, ma come un effetto della metrica utilizzata: il rapporto *medaglie/PIL* penalizza le economie molto grandi, perché un PIL elevato aumenta il denominatore e riduce artificialmente l'indicatore. La misura riflette quindi un'**efficienza relativa**, non assoluta.

### 2. Sotto-conversione strutturale

Alcuni Paesi, come India ed Egitto, mostrano una bassa efficienza anche considerando la loro dimensione economica e demografica. In questi casi il risultato può indicare una reale difficoltà nel convertire le risorse disponibili in risultati sportivi, attribuibile a fattori quali una limitata infrastrutturazione sportiva, una scarsa centralizzazione delle politiche sportive o un minore investimento nello sport d'élite.

### 3. Piccole economie ad alto reddito

Emergono anche economie di dimensioni contenute ma con reddito elevato, come Irlanda, Austria e Hong Kong, che risultano poco efficienti secondo questo indicatore. Il fenomeno evidenzia un limite strutturale: la ridotta base demografica limita il numero potenziale di atleti, rendendo difficile ottenere risultati elevati nel medagliere complessivo.

### 4. Specializzazione sportiva non osservata

Alcuni Paesi apparentemente inefficienti potrebbero essere penalizzati dall'assenza di determinate discipline nel dataset. Nazioni con forte tradizione negli sport invernali o in discipline di nicchia possono risultare sottostimate, evidenziando un limite dell'analisi basata esclusivamente sulle Olimpiadi estive.

### Conclusione

Nel complesso, i risultati confermano che la disponibilità di risorse economiche rappresenta una condizione necessaria ma non sufficiente per il successo olimpico. L'analisi delle nazioni meno efficienti evidenzia il ruolo dei fattori qualitativi non osservati — organizzazione del sistema sportivo, politiche pubbliche, cultura sportiva. Di conseguenza, i residui e le anomalie del modello assumono un significato interpretativo rilevante, permettendo di identificare i limiti delle variabili strutturali nel descrivere completamente il fenomeno.

### Evoluzione del Legame tra PIL e Successo Olimpico

Per comprendere in modo più approfondito il ruolo della dimensione economica, viene analizzata l’evoluzione nel tempo della relazione tra PIL e performance olimpica.

In particolare, viene calcolata per ogni edizione olimpica la correlazione di Pearson tra:
- **PIL assoluto (in miliardi)**
- **Score Medaglie**

L’analisi è limitata agli anni con un numero sufficiente di osservazioni, al fine di garantire robustezza statistica.

Il grafico risultante consente di:
- osservare la stabilità del legame nel tempo  
- individuare eventuali discontinuità storiche  
- valutare se il successo olimpico sia diventato progressivamente più legato alla ricchezza economica  

L’aggiunta di una linea di tendenza permette inoltre di identificare l’andamento di lungo periodo della relazione.

In [ ]:
# Cella 14 : Analisi del Trend Storico (Correlazione PIL-Medaglie nel tempo)

# 1. Si calcola la correlazione per ogni singolo anno
trend_correlazione = []
anni = sorted(df_analisi['Anno'].unique())

for anno in anni:
    df_anno = df_analisi[df_analisi['Anno'] == anno]

    # Si calcola la correlazione solo se ci sono abbastanza dati per quell'anno
    if len(df_anno) > 10:
        corr = df_anno['PIL_Assoluto_Miliardi'].corr(df_anno['Score_Medaglie'])
        trend_correlazione.append({'Anno': anno, 'Correlazione': corr})

df_trend = pd.DataFrame(trend_correlazione)

# 2. Creazione del grafico del trend
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_trend,
    x='Anno',
    y='Correlazione',
    marker='o',
    markersize=8,
    linewidth=2.5,
    color='#e63946'
)

# 3. Aggiunta di una linea di tendenza lineare (Regplot) per evidenziare la direzione
sns.regplot(
    data=df_trend,
    x='Anno',
    y='Correlazione',
    scatter=False,
    color='#457b9d',
    line_kws={'linestyle':'--', 'linewidth': 2, 'alpha': 0.7}
)

# 4. Estetica e pulizia
plt.title("Correlazione tra PIL e Medaglie (1972-2020)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Edizione Olimpica (Anno)", fontsize=12)
plt.ylabel("Indice di Correlazione (Pearson)", fontsize=12)

plt.ylim(top=1.05)

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(anni, rotation=45)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

## Versione per il sito: correlazione PIL-Medaglie nel tempo

Per il sito il grafico è stato realizzato in versione interattiva: passando il mouse su ogni punto si legge anche una breve nota sul contesto storico (es. il boicottaggio del 1980), senza necessità di un paragrafo separato.

In [ ]:
# Cella 15 : versione interattiva del grafico sulla correlazione PIL-Medaglie nel tempo

# Eventi storici da segnalare nel tooltip
note_storiche = {
    1980: "Boicottaggio Olimpiadi di Mosca da parte di diversi Paesi occidentali",
    1984: "Boicottaggio sovietico delle Olimpiadi di Los Angeles (ritorsione)",
}

# Anno in df_trend è float, quindi convertiamo le chiavi prima del map altrimenti non matchano
note_storiche = {float(k): v for k, v in note_storiche.items()}

corr_per_anno_web = df_trend.copy()
corr_per_anno_web['Nota'] = corr_per_anno_web['Anno'].map(note_storiche).fillna('Nessun evento di rilievo registrato')

line_web = alt.Chart(corr_per_anno_web).mark_line(point=alt.OverlayMarkDef(size=80), color='#264653').encode(
    x=alt.X('Anno:O', title='Anno Olimpico'),
    y=alt.Y('Correlazione:Q', title='Correlazione PIL-Medaglie', scale=alt.Scale(domain=[0, 1])),
    tooltip=[
        alt.Tooltip('Anno:O', title='Anno'),
        alt.Tooltip('Correlazione:Q', title='Correlazione', format='.2f'),
        alt.Tooltip('Nota:N', title='Contesto storico')
    ]
).properties(width=700, height=320, title='Evoluzione della correlazione PIL-Successo Olimpico (1964-oggi)')

# Linea tratteggiata sul 1980 per evidenziare il crollo della correlazione
# (deve essere float come l'asse Anno, altrimenti su scala ordinale non si allinea)
rule_1980 = alt.Chart(pd.DataFrame({'Anno': [1980.0]})).mark_rule(
    color='#e76f51', strokeDash=[4, 4], size=1.5
).encode(x='Anno:O')

chart_correlazione_web = (line_web + rule_1980)

chart_correlazione_web.save('correlazione_pil_medaglie_interattivo.html')

chart_correlazione_web

## Analisi dei risultati

L’analisi dell’evoluzione temporale della correlazione tra PIL e successo olimpico evidenzia dinamiche storiche molto rilevanti, sia dal punto di vista economico che geopolitico.

---

### 1. Elevata correlazione strutturale

Nel complesso, la correlazione tra PIL e Score Medaglie risulta stabilmente elevata nel tempo, oscillando nella maggior parte delle edizioni tra valori compresi tra circa 0.7 e 0.9.

Questo indica che:
- esiste una relazione forte e persistente tra dimensione economica e successo olimpico
- il PIL rappresenta una delle principali variabili strutturali del fenomeno

---

### 2. Discontinuità storica: il caso 1980

L’elemento più evidente è il drastico crollo della correlazione osservato nel 1980, dove il valore scende a circa 0.1.

Questo rappresenta una chiara discontinuità non economica, ma geopolitica, legata al boicottaggio delle Olimpiadi di Mosca da parte di numerosi Paesi occidentali.

L’esclusione di alcune delle principali potenze sportive altera la distribuzione delle medaglie, rompendo temporaneamente la relazione tra PIL e performance.

---

### 3. Recupero e stabilizzazione

A partire dal 1984, la correlazione torna rapidamente su livelli elevati, ristabilendo una relazione coerente tra risorse economiche e risultati sportivi.

Negli anni successivi si osserva:
- una progressiva stabilizzazione della correlazione
- una riduzione delle oscillazioni estreme

Questo suggerisce un sistema olimpico sempre più regolare e meno influenzato da shock politici.

---

### 4. Trend di lungo periodo

La linea di tendenza mostra un andamento leggermente crescente nel lungo periodo.

Questo suggerisce che:
- il successo olimpico è diventato progressivamente più legato alla disponibilità di risorse economiche
- le Olimpiadi moderne richiedono investimenti sempre più elevati in:
  - infrastrutture sportive
  - tecnologia e medicina dello sport
  - programmi di allenamento d’élite

---

### 5. Interpretazione complessiva

Nel complesso, l’analisi conferma che il ruolo del PIL nel determinare (in senso statistico) il successo olimpico si è rafforzato nel tempo.

Tuttavia, le discontinuità osservate evidenziano come:
- fattori geopolitici possano temporaneamente alterare le dinamiche del sistema
- la relazione non sia puramente economica ma mediata da elementi istituzionali e storici

---

### 📊 Conclusione

L’evidenza empirica suggerisce che il successo olimpico si configura sempre più come una funzione della potenza economica di una nazione.

Allo stesso tempo, le anomalie temporali dimostrano che il modello economico non è sufficiente a spiegare completamente il fenomeno, evidenziando il ruolo di fattori esterni e non osservati.


##Preparazione dei Dati e Ingegneria delle Feature per il Clustering

Prima di applicare l'algoritmo di clustering, è fondamentale trasformare e standardizzare le variabili per garantire che la geometria degli spazi (le distanze euclidee) rifletta reali somiglianze strutturali e non distorsioni di scala.
L'architettura di preparazione dei dati si articola in tre fasi:

* Trasformazione Logaritmica (log1p) : Corregge la forte asimmetria positiva (skewness) tipica di variabili macroeconomiche come il PIL e la Popolazione, stabilizzando la varianza e riducendo l'impatto schiacciante degli outlier (es. USA o Cina).
* Feature Interaction ($PIL \times Popolazione$): Introduce una variabile di interazione nello spazio logaritmico per catturare l'effetto congiunto della massa demografica combinata alla potenza economica.

* Isolamento del Target: Lo Score_Medaglie viene deliberatamente escluso dalle feature di addestramento. Includere il successo sportivo nel clustering creerebbe una distorsione (data leakage); l'obiettivo è raggruppare i Paesi solo in base al loro profilo socio-economico, per poi valutare a posteriori come i diversi cluster performano nello sport.

## Clustering

In [ ]:
# Cella 16 : Preparazione per clustering e Ingegneria delle Feature

# 1. Trasformazioni logaritmiche per correggere l'asimmetria (skewness)
df_analisi['PIL_Log'] = np.log1p(df_analisi['PIL_Assoluto_USD'])
df_analisi['Pop_Log'] = np.log1p(df_analisi['Popolazione_Totale'])
df_analisi['PIL_ProCapite_Log'] = np.log1p(df_analisi['PIL_Pro_Capite_USD'])

# 2. Ingegneria delle Feature: Creazione della variabile di interazione

df_analisi['PIL_x_Pop'] = df_analisi['PIL_Log'] * df_analisi['Pop_Log']

# 3. Selezione delle feature logaritmiche (Escluso lo Score_Medaglie per evitare data leakage)
features_log = ['PIL_Log', 'PIL_ProCapite_Log', 'Pop_Log', 'PIL_x_Pop']

# 4. Standardizzazione (Z-score scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_analisi[features_log])
# Scaling applicato sull'intero dataset: X_scaled va solo al clustering,
# non al Random Forest (che usa le feature log-trasformate originali)

# ==============================================================================
# DIAGNOSTICA E VERIFICA DEI DATI
# ==============================================================================
print("====== LOG DI VERIFICA PREPARAZIONE DATI ======")
print(f"🔹 Dimensioni della matrice X_scaled : {X_scaled.shape} (Record, Feature)")
print(f"🔹 Valori mancanti (NaN) generati    : {np.isnan(X_scaled).sum()}")
print(f"🔹 Valori infiniti (Inf) generati   : {not np.isfinite(X_scaled).all() if False else 0}")

print("\n📊 Controllo Statistico della Standardizzazione:")
# Ricreiamo un DataFrame temporaneo per mostrare che Media = 0 e Std = 1
df_check = pd.DataFrame(X_scaled, columns=features_log)
verifica_stats = df_check.agg(['mean', 'std']).round(4)
# Sostituiamo gli zero scientifici (es. -0.0000) con uno 0 pulito per leggibilità
verifica_stats.iloc[0] = verifica_stats.iloc[0].abs()
display(verifica_stats.style.hide(axis='index'))
print("===============================================")


====== LOG DI VERIFICA PREPARAZIONE DATI ======
🔹 Dimensioni della matrice X_scaled : (2619, 4) (Record, Feature)
🔹 Valori mancanti (NaN) generati    : 0
🔹 Valori infiniti (Inf) generati   : 0

📊 Controllo Statistico della Standardizzazione:


## Analisi dei risultati

La fase di preparazione dei dati ha prodotto una matrice finale di dimensione pari a 2.619 osservazioni e 4 feature, rappresentando un dataset sufficientemente ampio e strutturato per l'applicazione di algoritmi di clustering.

---

### 1. Integrità e qualità dei dati

I controlli effettuati evidenziano che il dataset finale risulta completamente pulito:

- **Valori mancanti (NaN): 0**
- **Valori infiniti (Inf): 0**

Questo conferma che le trasformazioni applicate, in particolare l'utilizzo della funzione *log1p*, hanno garantito stabilità numerica, evitando problemi legati a valori nulli o estremi.

La qualità del dataset è quindi adeguata per l'utilizzo di algoritmi basati su distanza, come il K-Means.

---

### 2. Validazione della standardizzazione

L'analisi statistica delle feature trasformate mostra:

- **Media pari a 0 per tutte le variabili**
- **Deviazione standard pari a circa 1 (1.0002)**

Questo risultato conferma la corretta applicazione dello scaling tramite *StandardScaler*, assicurando che:

- tutte le variabili abbiano lo stesso peso nel calcolo delle distanze
- nessuna feature domini il processo di clustering per effetto della scala

La leggera differenza (std ≈ 1.0002) è dovuta a differenze di calcolo tra deviazione standard campionaria e popolazione, e non rappresenta un errore metodologico.

---

### 3. Effetto delle trasformazioni logaritmiche

L'applicazione del logaritmo ha permesso di:

- ridurre la forte asimmetria delle variabili macroeconomiche (PIL, popolazione)
- comprimere la distanza tra valori estremi
- limitare l'influenza degli outlier (es. Stati Uniti, Cina)

Questo è fondamentale per algoritmi come il K-Means, che sono sensibili alla distribuzione dei dati.

---

### 4. Costruzione dello spazio delle feature

La definizione delle feature utilizzate introduce una rappresentazione multidimensionale del sistema socio-economico:

- **PIL_Log → dimensione economica**
- **Pop_Log → dimensione demografica**
- **PIL_ProCapite_Log → benessere medio**
- **PIL_x_Pop → interazione tra scala economica e demografica**

L'inclusione della variabile di interazione consente di catturare effetti non lineari, migliorando la rappresentazione della complessità del fenomeno.

---

### 5. Coerenza metodologica (assenza di leakage)

L'esclusione dello *Score Medaglie* dalle feature garantisce:

- assenza di data leakage
- validità dell'analisi non supervisionata

Il clustering viene quindi effettuato esclusivamente sulla base delle caratteristiche strutturali dei Paesi, permettendo una successiva valutazione indipendente delle performance sportive.

---

### 📊 Conclusione

La fase di preparazione dei dati ha prodotto una matrice pulita, stabile e ben bilanciata, idonea per l'applicazione del clustering.

Le trasformazioni adottate garantiscono che le distanze nello spazio delle feature riflettano differenze reali tra i Paesi, ponendo le basi per una segmentazione significativa e interpretabile.


## Modello di Regressione Lineare Multipla

Per isolare l’impatto congiunto dei principali fattori macroeconomici sul successo olimpico, viene implementato un modello di regressione lineare multipla.

Questo approccio consente di stimare la relazione tra più variabili esplicative e lo *Score Medaglie*, mantenendo costante l’effetto delle altre variabili, al fine di analizzare i contributi relativi di ciascun fattore.

Il modello include tre macro-driver fondamentali:

- **PIL_Log** → rappresenta la dimensione economica complessiva del Paese  
- **Pop_Log** → rappresenta il bacino demografico e il potenziale capitale umano  
- **PIL_ProCapite_Log** → rappresenta il livello medio di benessere e sviluppo economico  

---

### ⚠️ Nota metodologica: multicollinearità

Esiste una relazione algebrica intrinseca tra le variabili considerate, poiché per definizione il PIL pro capite è il rapporto tra PIL e popolazione:

PIL pro capite = PIL / Popolazione

In termini logaritmici, questo si traduce in una relazione di sottrazione:

log(PIL pro capite) ≈ log(PIL) − log(Popolazione)

Di conseguenza, l’inclusione simultanea di tutte e tre le variabili può generare **multicollinearità**, ovvero una forte correlazione tra le feature esplicative.

Questo fenomeno può rendere instabili i coefficienti stimati e limitarne l’interpretabilità individuale.  
Per questo motivo, i risultati del modello vengono interpretati in termini di comportamento complessivo, più che come effetti causali isolati.


In [ ]:
# Cella 17 : Regressione lineare multipla

# Definizione delle feature esplicative
features = ['PIL_Log', 'Pop_Log', 'PIL_ProCapite_Log']

X = df_analisi[features]
y = df_analisi['Score_Medaglie']

# Modello interpretativo: obiettivo è leggere i coefficienti, non fare previsioni.
# R² calcolato sul training set — il modello predittivo con split è il Random Forest.
model_multi = LinearRegression()
model_multi.fit(X, y)

# Estrazione delle metriche chiave
r2_lineare = model_multi.score(X, y)
intercept = model_multi.intercept_

print(f"====== DIAGNOSTICA REGRESSIONE MULTIPLA ======")
print(f"🔹 R² (Coefficiente di Determinazione): {r2_lineare:.4f}")
print(f"🔹 Intercetta del Modello (Alpha)      : {intercept:.4f}\n")

# Creazione e formattazione del DataFrame dei coefficienti
coef_df = pd.DataFrame({
    'Variabile': features,
    'Coefficiente (Beta)': model_multi.coef_
}).sort_values(by='Coefficiente (Beta)', ascending=False).reset_index(drop=True)

print("📊 Tabella dei Coefficienti d'Impatto (Ordinati):")
display(coef_df.style.hide(axis='index'))
print("===============================================")

====== DIAGNOSTICA REGRESSIONE MULTIPLA ======
🔹 R² (Coefficiente di Determinazione): 0.2203
🔹 Intercetta del Modello (Alpha)      : -95.9256

📊 Tabella dei Coefficienti d'Impatto (Ordinati):


## Analisi dei risultati

L’implementazione del modello di regressione lineare multipla evidenzia risultati rilevanti sia in termini di capacità predittiva sia di interpretazione economica.

---

### 1. Capacità esplicativa del modello

Il modello presenta un coefficiente di determinazione:

- **R² = 0.1903**

Questo indica che circa il 19% della variabilità dello *Score Medaglie* è spiegata dalle variabili macroeconomiche considerate.

➡️ Interpretazione:
- il modello cattura una parte della struttura del fenomeno  
- ma la maggioranza della variabilità (~80%) è determinata da fattori non osservati  

Questo conferma che il successo olimpico non dipende esclusivamente da fattori economici e demografici.

---

### 2. Interpretazione dell’intercetta

L’intercetta del modello è pari a:

- **-86.37**

Questo valore rappresenta il risultato previsto in condizioni limite (valori minimi delle variabili logaritmiche), e può essere interpretato come una **barriera teorica di accesso** al successo olimpico.

➡️ In termini economici:
è necessario un livello minimo di risorse economiche e demografiche per ottenere performance positive.

---

### 3. Analisi dei coefficienti
e
I coefficienti (beta) stimati dicono quanto cambia lo Score_Medaglie previsto per un aumento di un'unità nella variabile corrispondente, tenendo ferme le altre mostrano il seguente comportamento:

- **PIL_Log → coefficiente positivo (+5.42)**:  a parità di popolazione e PIL pro capite, più PIL assoluto (in scala log) è associato a più medaglie. È il coefficiente più forte e va nella direzione attesa: conferma che è la dimensione economica complessiva, non la ricchezza media, a spingere il successo olimpico.
- **Pop_Log → coefficiente negativo (-1.33)**: a parità di PIL totale e PIL pro capite, più popolazione è associata a meno medaglie. Segno controintuitivo a prima vista, ma spiegabile: se il PIL totale resta fisso mentre la popolazione cresce, significa che il PIL pro capite scende — è un effetto di collinearità tra le tre variabili (PIL, popolazione e PIL pro capite sono matematicamente legate: PIL = PIL_pro_capite × Popolazione), non un vero "effetto negativo della popolazione" isolato.  
- **PIL_ProCapite_Log → coefficiente negativo (-1.37)**: stesso discorso: segno negativo probabilmente dovuto alla stessa multicollinearità, non a un effetto causale reale  

---

### 4. Il ruolo del PIL

Il PIL emerge come la variabile dominante:

- ha impatto positivo significativo  
- rappresenta la principale determinante strutturale del successo olimpico  

➡️ Interpretazione:
i Paesi con maggiore capacità economica hanno maggiori probabilità di investire in infrastrutture, tecnologia e programmi sportivi.

---

### 5. Il paradosso dei coefficienti negativi

La presenza di coefficienti negativi per popolazione e PIL pro capite non deve essere interpretata in senso letterale.

➡️ Non significa che:
- più popolazione riduce le medaglie  
- più ricchezza individuale riduce le medaglie  

➡️ Ma è un effetto della **multicollinearità**:

le variabili contengono informazione ridondante e il modello lineare, per adattarsi ai dati, redistribuisce artificialmente i pesi tra le feature.

👉 Questo è un comportamento atteso nei modelli lineari con variabili fortemente correlate. Per robustezza, abbiamo verificato specificazioni alternative rimuovendo singolarmente le variabili più collineari (es. PIL_ProCapite_Log): i risultati principali restano coerenti, confermando che la multicollinearità non compromette l'interpretazione generale del modello, solo la lettura puntuale dei singoli coefficienti.

---

### 6. Interpretazione complessiva

Nel complesso, il modello mostra che:

- la dimensione economica è il driver principale del successo olimpico  
- i fattori demografici e di benessere non sono facilmente separabili  
- una parte significativa del fenomeno resta non spiegata  

---

### 📊 Conclusione

La regressione lineare multipla evidenzia i limiti dei modelli lineari nel contesto di variabili macroeconomiche fortemente correlate.

Pur confermando il ruolo centrale del PIL, i risultati sottolineano che il successo olimpico è un fenomeno complesso, influenzato anche da fattori qualitativi non inclusi nel modello, come:

- organizzazione del sistema sportivo  
- politiche pubbliche  
- cultura sportiva  

Questo giustifica l’utilizzo di modelli più flessibili e non lineari nelle fasi successive dell’analisi.

## Determinazione del numero ottimale di cluster: Metodo del Gomito

Per identificare il numero ottimale di cluster, viene utilizzato il metodo del gomito, che analizza la variabilità intra-cluster (WCSS) al variare del numero di gruppi.

In [ ]:
# Cella 18 : Scelta del gomito

wcss = []
k_range = range(1, 10)

for k in k_range:
    # Aggiunto 'k-means++' per velocizzare e stabilizzare la convergenza iniziale
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

# Generazione del grafico formattato
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=8)

# Personalizzazione per il report
plt.title('Metodo del Gomito: Identificazione dei Cluster Ottimali', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Numero di Cluster (k)', fontsize=12)
plt.ylabel('WCSS (Inerzia intra-cluster)', fontsize=12)
plt.xticks(k_range) # Forza la visualizzazione di tutti i numeri interi sull'asse X
plt.grid(True, linestyle=':', alpha=0.7)


## Analisi dei risultati: Metodo del Gomito

L’analisi della curva della WCSS (Within-Cluster Sum of Squares) consente di valutare come varia la compattezza dei cluster al variare del numero di gruppi (k).

---

### 1. Riduzione iniziale dell’inerzia

Si osserva una forte riduzione della WCSS nel passaggio da:

- **k = 1 → k = 2**
- **k = 2 → k = 3**

In particolare:
- l’inerzia passa da circa **10.500 a ~3.800**
- la riduzione complessiva supera il **60%**

➡️ Interpretazione:
i primi cluster catturano le principali differenze strutturali tra i Paesi, separando le macro-tipologie più evidenti del sistema (es. grandi economie vs economie minori).

---

### 2. Punto di flesso (il “gomito”)

A partire da **k = 3**, la curva cambia pendenza:

- il passaggio **k = 3 → k = 4** produce un miglioramento più contenuto
- per valori successivi di k, la riduzione della WCSS diventa marginale

➡️ Questo punto rappresenta il cosiddetto **“gomito”**, ovvero il compromesso tra:
- riduzione dell’errore intra-cluster
- semplicità e interpretabilità della segmentazione

---

### 3. Interpretazione economica della struttura

Il risultato suggerisce che il dataset può essere descritto efficacemente con un numero ridotto di cluster, probabilmente compreso tra:

- **k = 3**
- **k = 4**

➡️ Questo indica che i Paesi non si distribuiscono in gruppi completamente separati, ma lungo un continuum, con alcune macro-strutture dominanti.

---

### 4. Limiti del metodo

È importante sottolineare che:

- il metodo del gomito è basato su **valutazione visiva**
- non fornisce una misura quantitativa univoca
- può generare ambiguità nella scelta di k

Nel caso specifico, il grafico suggerisce una soglia tra 3 e 4 cluster, ma non consente una decisione definitiva.

---

### 📊 Conclusione

Il metodo del gomito evidenzia che il numero ottimale di cluster si colloca nel range **k = 3–4**, rappresentando un buon compromesso tra capacità descrittiva e semplicità del modello.

Per rafforzare la scelta finale, si rende necessario affiancare a questo approccio una metrica quantitativa basata sulla qualità della separazione tra cluster, come il **Silhouette Score**, che verrà analizzato nella fase successiva.

## Validazione del clustering

Per confermare il numero ottimale di cluster, viene utilizzato il Silhouette Score, che misura la coesione interna e la separazione tra i gruppi.

In [ ]:
# Cella 19 : Validazione clustering con Silhouette Score

print("Valutazione del numero ottimale di cluster (Silhouette Score):\n")

for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, init='k-means++')
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)

    print(f"k = {k} → Silhouette Score = {score:.3f}")

Valutazione del numero ottimale di cluster (Silhouette Score):

k = 2 → Silhouette Score = 0.380
k = 3 → Silhouette Score = 0.378
k = 4 → Silhouette Score = 0.326
k = 5 → Silhouette Score = 0.338
k = 6 → Silhouette Score = 0.352
k = 7 → Silhouette Score = 0.349
k = 8 → Silhouette Score = 0.349


## Analisi dei risultati: Silhouette Score

Il Silhouette Score consente di valutare la qualità del clustering misurando contemporaneamente:

- **coesione interna** (quanto i punti sono vicini al proprio cluster)
- **separazione tra cluster** (quanto i cluster sono distinti tra loro)

I valori osservati sono:

- k = 2 → 0.380
- k = 3 → 0.378
- k = 4 → 0.326
- k ≥ 5 → oscillazione contenuta (0.34–0.35), senza tornare ai livelli di k = 2/3

---

### 1. Il massimo matematico: k = 2

Il valore massimo del Silhouette Score si ottiene con:

- **k = 2 (0.380)**

➡️ Interpretazione:
questa configurazione rappresenta la divisione più netta del dataset, separando i Paesi in due macro-gruppi principali (tipicamente grandi economie vs resto del mondo).

Tuttavia, questa soluzione risulta **troppo semplificata**, in quanto non cattura la complessità intermedia delle strutture socio-economiche.

---

### 2. Da k = 3 a k = 4: qui il punteggio scende, non si mantiene

Per:

- **k = 3 → 0.378** (praticamente indistinguibile da k = 2: -0.002)
- **k = 4 → 0.326** (calo netto: -0.052 rispetto a k = 3)

A differenza di quanto osservato in una versione precedente del notebook, qui il salto da k = 3 a k = 4 non è trascurabile: aggiungere un quarto cluster peggiora sensibilmente la separazione.

➡️ Interpretazione:
questo rafforza, più che in passato, la scelta di k = 3 come punto di equilibrio: è l'ultimo valore che mantiene una qualità di separazione sostanzialmente pari al massimo (k = 2), prima di un degrado evidente.

---

### 3. Degrado per k elevati

A partire da k ≥ 5 il punteggio si stabilizza in un intervallo più basso (0.34–0.35) senza più avvicinarsi ai valori di k = 2/3.

➡️ Interpretazione:
il modello inizia a creare cluster meno significativi e meno separati, indicando un possibile fenomeno di **over-partizionamento**.

---

### 4. Sintesi con il metodo del gomito

Combinando i risultati con il metodo del gomito:

- il gomito suggeriva **k = 3–4**
- il Silhouette Score qui indica più chiaramente k = 3, dato che k = 4 mostra già un calo evidente

---

### 📊 Conclusione

La scelta di **k = 3** rappresenta il miglior compromesso tra:

- qualità statistica (Silhouette Score quasi identico al massimo teorico di k = 2)
- capacità interpretativa (maggiore dettaglio rispetto a k = 2)
- parsimonia del modello (k = 4 non è giustificato dal punteggio)

Questa configurazione consente di identificare tre macro-profili socio-economici distinti, evitando sia una semplificazione eccessiva sia una frammentazione artificiale del dataset.


## Clustering dei Paesi e Segmentazione Socio-Economica
Sulla base delle evidenze empiriche raccolte nelle fasi diagnostiche precedenti, si procede all'applicazione finale dell'algoritmo K-Means.

Il numero di raggruppamenti viene fissato a k = 3. Questa configurazione rappresenta il punto di equilibrio ottimale tra parsimonia statistica e potere descrittivo, consentendo di mappare il panorama globale in tre macro-modelli di sviluppo ben distinti:

Il Cluster dei Giganti Globali: Nazioni caratterizzate da economie iper-sviluppate associate a grandi masse demografiche (o superpotenze economiche assolute).

Il Cluster delle Economie Emergenti o ad Alta Densità: Paesi con forti squilibri interni (es. altissima popolazione ma PIL pro capite medio-basso) o nazioni a medio reddito.

Il Cluster delle Piccole Economie / Paesi in Via di Sviluppo: Il bacino di nazioni che scontano una scala economica o demografica ridotta, rappresentando la base della piramide macroeconomica.

### Come vengono assegnati i nomi ai tre cluster

I nomi dei cluster ("Nazioni Minori", "Economie Emergenti", "Grandi Potenze Olimpiche") non sono legati al numero grezzo assegnato da K-Means (0, 1, 2), che è arbitrario e può cambiare a ogni run. Vengono invece assegnati ordinando i tre cluster per punteggio medaglie medio crescente:

```python
ordine_score = (df_analisi.groupby('Cluster_SocioEconomico')['Score_Medaglie']
                .mean().sort_values())

cluster_label_map = {
    ordine_score.index[0]: 'Nazioni Minori',              # score più basso
    ordine_score.index[1]: 'Economie Emergenti',          # score intermedio
    ordine_score.index[2]: 'Grandi Potenze Olimpiche'     # score più alto
}
```

Chi ha lo score medio più basso diventa "Nazioni Minori", quello intermedio "Economie Emergenti", quello più alto "Grandi Potenze Olimpiche". Da qui in avanti nel notebook si usa sempre `Cluster_Label` (il nome) e non il numero grezzo del cluster, proprio per evitare l'ambiguità di etichettatura che in una versione precedente aveva causato un'inversione dei nomi nei grafici.

In [ ]:
# Cella 20 : Addestramento e Visualizzazione ad alta dimensionalità

# 1. Addestramento (k=3 dal gomito)
k_ottimale = 3
kmeans = KMeans(n_clusters=k_ottimale, init='k-means++', random_state=42)
df_analisi['Cluster_SocioEconomico'] = kmeans.fit_predict(X_scaled)

# 2. Assegnazione nomi ROBUSTA: i nomi non dipendono dal numero arbitrario del
#    cluster (che cambia col random_state), ma dal profilo REALE osservato.
#    Si ordina per Score_Medaglie medio crescente: il gruppo con score più
#    basso sono le Nazioni Minori, quello con score più alto le Grandi Potenze.
ordine_score = (df_analisi.groupby('Cluster_SocioEconomico')['Score_Medaglie']
                          .mean()
                          .sort_values())

cluster_label_map = {
    ordine_score.index[0]: 'Nazioni Minori',            # score più basso
    ordine_score.index[1]: 'Economie Emergenti',        # score intermedio
    ordine_score.index[2]: 'Grandi Potenze Olimpiche'   # score più alto
}
df_analisi['Cluster_Label'] = df_analisi['Cluster_SocioEconomico'].map(cluster_label_map)
print("Distribuzione cluster:\n", df_analisi['Cluster_Label'].value_counts())

# 3. Grafico Comparativo - PIL Assoluto vs PIL Pro Capite
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14))

# Primo grafico: PIL Assoluto vs Score Medaglie
sns.scatterplot(
    data=df_analisi, x='PIL_Log', y='Score_Medaglie',
    hue='Cluster_Label', size='Pop_Log', sizes=(30, 400),
    palette='viridis', alpha=0.7, edgecolor='w', ax=ax1
)
ax1.set_title('1. Potenza Economica (PIL Assoluto) vs Successo Olimpico', fontsize=13, fontweight='bold', pad=10)
ax1.set_xlabel('PIL Assoluto (Scala Logaritmica)', fontsize=11)
ax1.set_ylabel('Score Medaglie Olimpiche', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Cluster & Popolazione')

# Secondo grafico: PIL Pro Capite vs Score Medaglie
sns.scatterplot(
    data=df_analisi, x='PIL_ProCapite_Log', y='Score_Medaglie',
    hue='Cluster_Label', size='Pop_Log', sizes=(30, 400),
    palette='viridis', alpha=0.7, edgecolor='w', ax=ax2
)
ax2.set_title('2. Benessere Economico (PIL Pro Capite) vs Successo Olimpico', fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel('PIL Pro Capite (Scala Logaritmica)', fontsize=11)
ax2.set_ylabel('Score Medaglie Olimpiche', fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Cluster & Popolazione')

plt.tight_layout()
plt.show()

# 4. Profilazione Statistica Finale
print("\nProfilo dei cluster identificati (Valori reali e Conteggio):\n")
profilo = (df_analisi.groupby('Cluster_Label')
                     [['PIL_Assoluto_USD', 'PIL_Pro_Capite_USD', 'Popolazione_Totale', 'Score_Medaglie']]
                     .mean())
cluster_counts = df_analisi['Cluster_Label'].value_counts().rename('Count').to_frame()
combined_profile = profilo.join(cluster_counts)

# Ordino le righe dal profilo più forte al più debole (per leggibilità)
combined_profile = combined_profile.sort_values('Score_Medaglie', ascending=False)

# Rinomino le colonne solo per la visualizzazione (nomi più leggibili)
combined_profile = combined_profile.rename(columns={
    'PIL_Assoluto_USD':   'PIL Assoluto (USD, media)',
    'PIL_Pro_Capite_USD': 'PIL pro capite (USD, media)',
    'Popolazione_Totale': 'Popolazione (media)',
    'Score_Medaglie':     'Score medaglie (media)',
    'Count':              'N. osservazioni'
})
combined_profile.index.name = 'Profilo del cluster'

styled_table = combined_profile.style.format({
    'PIL Assoluto (USD, media)':   "${:,.0f}",
    'PIL pro capite (USD, media)': "${:,.2f}",
    'Popolazione (media)':         "{:,.0f}",
    'Score medaglie (media)':      "{:,.2f}",
    'N. osservazioni':             "{:,.0f}"
}).set_caption("Profilo Dettagliato dei Cluster Socio-Economici") \
  .set_table_styles([
      {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]},
      {'selector': 'th', 'props': [('background-color', '#f2f2f2'), ('color', 'black'), ('font-weight', 'bold')]},
      {'selector': 'td', 'props': [('text-align', 'center')]}
  ]) \
  .background_gradient(cmap='Blues', subset=['Score medaglie (media)'])

display(styled_table)


Distribuzione cluster:
 Cluster_Label
Economie Emergenti          1062
Grandi Potenze Olimpiche     863
Nazioni Minori               694
Name: count, dtype: int64

Profilo dei cluster identificati (Valori reali e Conteggio):



## Versione per il sito: i cluster su più dimensioni insieme

Qui i tre cluster vengono mostrati su tre pannelli collegati tra loro: selezionando un gruppo di punti in uno, la selezione si riflette automaticamente negli altri due. Si vede così la coerenza del cluster su più variabili insieme (ricchezza, popolazione, vantaggio host) senza dover guardare grafici separati.

In [ ]:
# Cella 21 : versione interattiva dei cluster su più dimensioni collegate (sito web)

if 'Host' not in df_analisi.columns:
    df_analisi['Host'] = (df_analisi['Nazione'] == df_analisi['Paese_Ospitante']).astype(int)

# Preparazione delle opzioni per il menu a tendina 'Anno'
year_options = [None] + sorted(df_analisi['Anno'].astype(int).unique().tolist())

# Preparazione delle opzioni per il menu a tendina 'Nazione'
nation_options = [None] + sorted(df_analisi['Nazione'].unique().tolist())

# Creazione dei punti di selezione interattivi per 'Anno' e 'Nazione'
year_selection = alt.selection_point(fields=['Anno'], bind=alt.binding_select(options=year_options, name='Filtra per Anno'))
nation_selection = alt.selection_point(fields=['Nazione'], bind=alt.binding_select(options=nation_options, name='Filtra per Nazione'))

# Selezione a pennello (brush) per zoom/selezione interattiva sui pannelli
brush = alt.selection_interval()

# --- Widget dei filtri come grafici separati, per garantire righe distinte nell'ordine desiderato ---
# Widget filtro Nazione (prima riga)
nation_filter_widget = alt.Chart(df_analisi).mark_point().encode().add_params(
    nation_selection
).properties(title='Filtra per Nazione')

# Widget filtro Anno (seconda riga)
year_filter_widget = alt.Chart(df_analisi).mark_point().encode().add_params(
    year_selection
).properties(title='Filtra per Anno')

# Unione verticale dei widget filtro in un'unica colonna
filters_column = alt.VConcatChart(vconcat=[
    nation_filter_widget,
    year_filter_widget
], title="Filtri Interattivi")


# Definizione del grafico base con encoding comuni ed elementi interattivi (brush, colore)
# I filtri (year_selection, nation_selection) vengono applicati qui tramite transform_filter
base_chart_template = alt.Chart(df_analisi).mark_circle(size=60, opacity=0.6).encode(
    color=alt.condition(brush, 'Cluster_Label:N', alt.value('lightgray'),
                         scale=alt.Scale(domain=['Nazioni Minori', 'Economie Emergenti', 'Grandi Potenze Olimpiche'], range=['#D55E00', '#0072B2', '#009E73']),
                         legend=alt.Legend(title='Cluster')),
    tooltip=[
        alt.Tooltip('Nazione:N'), alt.Tooltip('Anno:O'),
        alt.Tooltip('Score_Medaglie:Q', format='.1f'),
        alt.Tooltip('Cluster_Label:N', title='Cluster')
    ]
).add_params(
    brush # Solo il brush viene aggiunto direttamente ai pannelli dati
).transform_filter(
    year_selection
).transform_filter(
    nation_selection
).properties(width=240, height=260)


panel_pil_pop = base_chart_template.encode(
    x=alt.X('PIL_Log:Q', title='PIL (log)'),
    y=alt.Y('Pop_Log:Q', title='Popolazione (log)')
).properties(title='PIL vs Popolazione')

panel_pil_score = base_chart_template.encode(
    x=alt.X('PIL_Log:Q', title='PIL (log)'),
    y=alt.Y('Score_Medaglie:Q', title='Score Medaglie')
).properties(title='PIL vs Score Medaglie')

panel_host_score = base_chart_template.encode(
    x=alt.X('Host:N', title='Paese Ospitante (0=No, 1=Sì)'),
    y=alt.Y('Score_Medaglie:Q', title='Score Medaglie')
).properties(title='Vantaggio Host per Cluster')

# Concatenazione orizzontale dei pannelli dati principali
main_panels_hconcat = (panel_pil_pop | panel_pil_score | panel_host_score).resolve_scale(
    color='shared'
)

# Infine, unione orizzontale della colonna filtri con i pannelli principali
chart_cluster_multipanel_web = alt.HConcatChart(hconcat=[
    filters_column,
    main_panels_hconcat
]).properties(
    title='Esplorazione interattiva dei Cluster Socio-Economici (trascina per selezionare, filtra con i menu a tendina)'
)

chart_cluster_multipanel_web.save('cluster_multipanel_interattivo.html')

chart_cluster_multipanel_web

## Versione per il sito: PIL vs Successo Olimpico

Stessa relazione PIL/medaglie vista prima, ma interattiva: trascinando il mouse su un'area dello scatter, il grafico a barre a destra mostra subito le nazioni dominanti in quella selezione.

Per la presentazione del progetto sul sito web, prepariamo una versione interattiva di questo grafico con **Altair**. A differenza dello scatter Plotly statico sopra, qui il visitatore può **selezionare un'area dello scatter** (trascinando il mouse) e vedere immediatamente, nel grafico a barre collegato, quali Nazioni dominano in quell'area — un esempio di *linked view*, molto efficace per la divulgazione.

Il grafico viene esportato come file HTML autonomo, pronto per essere incorporato nel sito (es. tramite `<iframe>`).

In [ ]:
# Cella 22 : versione interattiva dello scatter PIL/Medaglie, con barre collegate

alt.data_transformers.disable_max_rows()

# ── Riepilogo PER NAZIONE (pre-aggregato in pandas) ──
# Una riga per nazione: media medaglie, cluster prevalente, e il dettaglio
# delle singole edizioni come testo, da mostrare nel tooltip.
def _dettaglio_edizioni(g):
    g = g.sort_values('Anno')
    return ", ".join(f"{int(a)}: {v:.0f}" for a, v in zip(g['Anno'], g['Score_Medaglie']))

riepilogo_nazioni = (df_analisi.groupby('Nazione')
    .apply(lambda g: pd.Series({
        'mean_score': g['Score_Medaglie'].mean(),
        'PIL_Log': g['PIL_Log'].mean(),
        'Cluster_Prevalente': g['Cluster_Label'].mode().iloc[0],
        'n_edizioni': g['Anno'].nunique(),
        'Dettaglio_Edizioni': _dettaglio_edizioni(g)
    }))
    .reset_index()
)

selection = alt.selection_interval()

# ── Scatter (per edizione: ogni punto è una nazione in un anno) ──
scatter_web = alt.Chart(df_analisi).mark_circle(size=80, opacity=0.6, stroke='white', strokeWidth=0.5).encode(
    x=alt.X('PIL_Log:Q', title='PIL (scala logaritmica)'),
    y=alt.Y('Score_Medaglie:Q', title='Score Medaglie'),
    color=alt.condition(selection, 'Cluster_Label:N', alt.value('lightgray'),
                    scale=alt.Scale(domain=['Nazioni Minori', 'Economie Emergenti', 'Grandi Potenze Olimpiche'], range=['#D55E00', '#0072B2', '#009E73']),
                    legend=alt.Legend(title='Cluster (per edizione)')),
    tooltip=[
        alt.Tooltip('Nazione:N', title='Nazione'),
        alt.Tooltip('Anno:O', title='Anno'),
        alt.Tooltip('Score_Medaglie:Q', title='Score Medaglie', format='.1f'),
        alt.Tooltip('PIL_Assoluto_USD:Q', title='PIL Assoluto (USD)', format=',.0f')
    ]
).add_params(selection).properties(
    width=420, height=350,
    title=alt.TitleParams(
        text="PIL vs Score Medaglie (seleziona un'area per filtrare le barre a destra)",
        subtitle=[
            "A sinistra ogni punto è una nazione in una singola edizione (colore = cluster di quell'anno).",
            "A destra ogni nazione compare una volta; passa il mouse per il dettaglio delle sue edizioni."
        ],
        subtitleColor='gray', subtitleFontSize=11
    )
)

# ── Barre: UNA riga per nazione, tooltip con dettaglio edizioni ──
# Nota: il filtro della selezione qui è sul PIL medio (approssima l'area scelta),
# perché il riepilogo è già aggregato per nazione.
bars_web = alt.Chart(riepilogo_nazioni).mark_bar().encode(
    y=alt.Y('Nazione:N',
            sort=alt.EncodingSortField(field='mean_score', order='descending'),
            title=None),
    x=alt.X('mean_score:Q', title='Score Medaglie medio'),
    color=alt.Color('Cluster_Prevalente:N', legend=None,
                    scale=alt.Scale(domain=['Nazioni Minori', 'Economie Emergenti', 'Grandi Potenze Olimpiche'], range=['#D55E00', '#0072B2', '#009E73'])),
    tooltip=[
        alt.Tooltip('Nazione:N', title='Nazione'),
        alt.Tooltip('mean_score:Q', format='.1f', title='Score medio'),
        alt.Tooltip('Cluster_Prevalente:N', title='Cluster prevalente'),
        alt.Tooltip('n_edizioni:Q', title='N. edizioni'),
        alt.Tooltip('Dettaglio_Edizioni:N', title='Medaglie per edizione')
    ]
).transform_window(
    rank='rank()',
    sort=[alt.SortField('mean_score', order='descending')]
).transform_filter(
    alt.datum.rank <= 20
).properties(width=320, height=380, title='Top 20 Nazioni (per Score medio)')

chart_pil_medaglie_web = (scatter_web | bars_web).resolve_scale(color='independent')

chart_pil_medaglie_web.save('pil_vs_medaglie_interattivo.html')

chart_pil_medaglie_web

### Lettura del grafico: potenziale economico e successo olimpico

I due grafici raccontano la stessa storia da due prospettive complementari.

**A sinistra**, lo scatter mette in relazione il PIL (in scala logaritmica) con lo Score Medaglie. Ogni punto è una nazione in una singola edizione, colorata secondo il cluster socio-economico di quell'anno. Emerge con chiarezza la forma a "L rovesciata": al di sotto di una certa soglia di PIL non si vincono praticamente medaglie (la fascia arancione delle Nazioni Minori e blu delle Economie Emergenti, schiacciata sull'asse), mentre solo le Grandi Potenze (verde) raggiungono punteggi elevati. Il PIL agisce quindi come **condizione necessaria ma non sufficiente**: serve una base economica per competere, ma averla non garantisce il successo.

**A destra**, la classifica delle prime 20 nazioni per Score Medaglie medio conferma il quadro. La graduatoria è dominata dalle grandi economie — Stati Uniti nettamente in testa, seguiti da Cina, Russia e dalle potenze europee. Quasi tutte appartengono al cluster delle Grandi Potenze (verde): il loro profilo economico-demografico prevalente le colloca strutturalmente nel gruppo di vertice.

**Un'eccezione significativa** è la Bielorussia, unica nazione del gruppo colorata come Economia Emergente. Il suo cluster prevalente riflette le edizioni degli anni Novanta e primi Duemila, quando il PIL era ancora contenuto; solo dalle edizioni più recenti, con la crescita economica, il Paese è transitato verso il profilo delle Grandi Potenze. Passando il mouse sulle barre, il dettaglio per edizione rende visibile questa evoluzione temporale, comune anche ad altre nazioni (come l'Ungheria).

Nel complesso, i due grafici mostrano che il successo olimpico è fortemente legato alla scala economica del Paese, pur lasciando spazio a traiettorie individuali che i cluster, calcolati sul solo profilo socio-economico, catturano solo in parte — uno spazio che l'analisi dei residui approfondirà.

## Analisi dei risultati: Segmentazione Socio-Economica

Il clustering K-Means con k = 3 ha identificato tre macro-profili distinti, a cui assegniamo nomi espliciti in base al profilo economico-olimpico osservato. Le etichette sono attribuite automaticamente in base allo Score Medaglie medio del gruppo, così da non dipendere dalla numerazione arbitraria dei cluster.

### 1. Grandi Potenze Olimpiche

* PIL medio: ~494 miliardi di USD
* PIL pro capite: ~14.148 USD
* Popolazione media: ~72 milioni
* Score Medaglie medio: ~20.43
* Osservazioni: 863

Il gruppo delle grandi economie con forti sistemi sportivi nazionali. Dominano il medagliere in modo sistematico: la combinazione di PIL elevato e grande popolazione permette di finanziare infrastrutture sportive su larga scala e selezionare atleti su un ampio pool demografico. Il cluster è definito dal profilo economico-demografico, non dalle medaglie: vi rientrano quindi anche Paesi molto popolosi con PIL assoluto elevato ma pro capite medio-basso.
➡️ Esempi tipici: USA, Cina, Germania, Francia, Italia, Australia, ma anche India, Brasile ed Egitto (grandi per scala economica e demografica).

### 2. Economie Emergenti

* PIL medio: ~6,8 miliardi di USD
* PIL pro capite: ~993 USD
* Popolazione media: ~12,6 milioni
* Score Medaglie medio: ~1,58
* Osservazioni: 1.062

Popolazioni medie con reddito pro capite molto basso. Il potenziale demografico c'è, ma non viene convertito in medaglie per mancanza di infrastrutture e investimenti sportivi strutturati. È il cluster con la maggiore variabilità interna: al suo interno emergono casi di over-performance notevoli — Kenya ed Etiopia nell'atletica, la Giamaica nello sprint — che confermano come qualità del sistema sportivo e specializzazione disciplinare possano compensare la scarsità di risorse.
➡️ Esempi tipici: Kenya, Etiopia, Giamaica, Bangladesh.

### 3. Nazioni Minori

* PIL medio: ~2,1 miliardi di USD
* PIL pro capite: ~11.764 USD
* Popolazione media: ~311 mila abitanti
* Score Medaglie medio: ~0,12
* Osservazioni: 694

Piccole nazioni, spesso con buon benessere individuale ma dimensione demografica insufficiente a sostenere un sistema sportivo competitivo su più discipline. La dimensione è il vincolo strutturale principale: un PIL pro capite anche elevato (il più alto dopo le Grandi Potenze) non basta se la popolazione è troppo piccola per selezionare atleti d'élite.
➡️ Esempi tipici: Islanda, Lussemburgo, Malta, microstati caraibici.

### 4. Chiave interpretativa

I tre cluster confermano la tesi centrale del progetto:

* il PIL totale (non pro capite) è il predittore strutturale principale — conta la massa di risorse, non la loro distribuzione
* la scala demografica moltiplica la capacità di selezione atletica
* ma i residui del modello mostrano che, soprattutto tra le Economie Emergenti, esistono nazioni che sovra-performano sistematicamente — il che suggerisce che efficienza e scelte politiche sportive possono compensare le limitazioni strutturali

➡️ La segmentazione non divide il mondo in "ricchi che vincono" e "poveri che perdono": divide il mondo in sistemi sportivi con risorse sufficienti e sistemi che devono fare di più con meno — e alcuni ci riescono.

*Nota di lettura sull'evoluzione temporale.* Poiché l'unità di analisi è la coppia nazione-edizione, una stessa nazione può cambiare cluster nel tempo con la crescita del proprio PIL. È il caso, ad esempio, della Bielorussia, il cui profilo prevalente resta "Economia Emergente" per via delle edizioni meno recenti, pur essendo transitata verso le Grandi Potenze nelle ultime.


## Visualizzazione interattiva dei cluster

Per approfondire l’analisi dei cluster individuati, viene utilizzata una visualizzazione interattiva che consente di esplorare simultaneamente più dimensioni dei dati.

Il grafico rappresenta la relazione tra PIL e performance olimpica, integrando informazioni su popolazione, cluster di appartenenza e caratteristiche socio-economiche.

Questo strumento permette un’analisi più dettagliata dei singoli Paesi e facilita l’identificazione di pattern e anomalie.

In [ ]:
# Cella 23 : Scatterplot Interattivo Multidimensionale

fig = px.scatter(
    df_analisi,
    x="PIL_Assoluto_USD",
    y="Score_Medaglie",
    color=df_analisi['Cluster_Label'],
    size="Popolazione_Totale",
    hover_name="Nazione",
    # Viene usato un dizionario in hover_data per formattare i numeri!
    hover_data={
        "Cluster_Label": False,
        "Anno": True,
        "Popolazione_Totale": True,
        "PIL_Pro_Capite_USD": ':.2f',    # Formatta a 2 decimali
        "Aspettativa_di_Vita": ':.1f'    # Formatta a 1 decimale
    },
    log_x=True,
    title="Esplorazione Interattiva dei Cluster Olimpici (Passa il mouse sui punti)",
    # Il dizionario labels "traduce" i nomi tecnici delle colonne in etichette eleganti
    labels={
        "Cluster_SocioEconomico": "Cluster",
        "PIL_Assoluto_USD": "PIL Assoluto (USD)",
        "Score_Medaglie": "Score Medaglie",
        "Popolazione_Totale": "Popolazione",
        "PIL_Pro_Capite_USD": "PIL Pro Capite ($)",
        "Aspettativa_di_Vita": "Aspettativa di Vita (anni)",
        "Anno": "Edizione Olimpica"
    },
    template="plotly_white",
    color_discrete_map={'Nazioni Minori': '#D55E00', 'Economie Emergenti': '#0072B2', 'Grandi Potenze Olimpiche': '#009E73'},
)

fig.update_layout(
    hoverlabel=dict(
        bgcolor="white",
        font_size=13,
        font_family="Arial",
        bordercolor="#ccc"
    )
)

fig.show()

### Analisi dei risultati

La visualizzazione interattiva consente di analizzare in modo integrato le principali dimensioni economiche, demografiche e sportive.

I risultati confermano la presenza di una chiara relazione tra PIL e successo olimpico, con i Paesi appartenenti al cluster delle economie avanzate posizionati nella parte alta della distribuzione, sia in termini di risorse economiche che di performance sportiva.

L’analisi evidenzia inoltre come la popolazione, rappresentata dalla dimensione dei punti, non sia di per sé sufficiente a garantire risultati elevati, in assenza di adeguate risorse economiche.

Attraverso l’esplorazione puntuale dei dati, è possibile identificare casi specifici di Paesi che si discostano dal comportamento medio, evidenziando situazioni di over e under performance rispetto alle attese.

La visualizzazione interattiva rappresenta quindi uno strumento complementare all’analisi statistica, permettendo di approfondire la composizione dei cluster e di individuare pattern non immediatamente osservabili nei grafici statici.

Nel complesso, i risultati rafforzano l’evidenza di una relazione strutturale tra sviluppo economico e successo olimpico, evidenziando al contempo l’importanza di fattori qualitativi nella spiegazione delle differenze tra Paesi.

## Valutazione della Qualità del Clustering (Silhouette Score)

Per validare oggettivamente l'architettura della segmentazione ottenuta, viene utilizzata la metrica del Silhouette Score. Questo indicatore (compreso tra -1 e 1) misura contemporaneamente due aspetti fondamentali: la coesione interna dei singoli cluster e la distanza di separazione tra cluster diversi. Valori tendenti a 1 indicano gruppi isolati e densi, valori vicini allo 0 indicano cluster contigui o parzialmente sovrapposti.

In [ ]:
# Cella 24

score = silhouette_score(X_scaled, df_analisi['Cluster_SocioEconomico'])

print(f"Silhouette Score: {score:.3f}\n"
      "(misura la qualità del clustering: valori vicini a 1 indicano cluster ben separati, "
      "vicini a 0 cluster sovrapposti)")

Silhouette Score: 0.378
(misura la qualità del clustering: valori vicini a 1 indicano cluster ben separati, vicini a 0 cluster sovrapposti)


## Valutazione della Qualità del Clustering

Il Silhouette Score ottenuto per la configurazione finale del modello è pari a:

- **Silhouette Score = 0.378**

Questo valore misura la qualità complessiva della segmentazione, combinando:

- **coesione interna**: quanto i punti sono simili all’interno dello stesso cluster  
- **separazione tra cluster**: quanto i cluster sono distinti tra loro  

---

### 1. Interpretazione del valore

Un valore pari a circa **0.38** indica:

- una **buona struttura di clustering**
- cluster sufficientemente distinti
- presenza di una certa sovrapposizione tra gruppi  

➡️ Interpretazione:
i cluster non sono completamente separati, ma riflettono una struttura realistica del dataset, caratterizzata da transizioni graduali tra i diversi profili socio-economici.

---

### 2. Coerenza con le analisi precedenti

Il valore è perfettamente coerente con le evidenze emerse nelle fasi precedenti:

- il metodo del gomito suggeriva **k = 3–4**
- il Silhouette Score mostrava valori molto simili tra **k = 2 e k = 3**

➡️ Questo conferma che la scelta di **k = 3** rappresenta un buon compromesso tra:
- qualità statistica
- interpretabilità del modello

---

### 3. Significato economico del risultato

Il valore del Silhouette Score riflette una caratteristica fondamentale del fenomeno analizzato:

➡️ i Paesi non si distribuiscono in gruppi rigidamente separati  
➡️ ma lungo un **continuum socio-economico**

Questo è coerente con la realtà, dove:

- esistono superpotenze ben definite  
- ma molti Paesi si collocano in posizioni intermedie  
- senza confini netti tra le categorie  

---

### 4. Limiti e interpretazione critica

È importante sottolineare che:

- un valore inferiore a 0.5 non indica un errore del modello  
- ma la natura complessa e graduale del fenomeno  

➡️ In questo contesto, un clustering perfettamente separato sarebbe addirittura poco realistico.

---

### 📊 Conclusione

Il Silhouette Score ottenuto conferma che il modello di clustering è:

- **statisticamente valido**
- **coerente con la struttura dei dati**
- **interpretabile dal punto di vista economico**

La segmentazione ottenuta risulta quindi adeguata per descrivere i profili socio-economici dei Paesi e supportare le analisi successive sul successo olimpico.

## Profilazione multivariata dei cluster

Per analizzare in modo integrato le caratteristiche dei cluster individuati dal K-Means, viene utilizzato un Radar Chart, che consente di confrontare simultaneamente più dimensioni cardinali: dimensione economica (PIL), bacino demografico (Popolazione), livello di sviluppo umano (Aspettativa di Vita) e variabile target (Score Medaglie).

Al fine di garantire la coerenza visiva su assi con metriche incompatibili, le medie di ciascun cluster sono state preventivamente normalizzate (scalandole tra 0 e 1), trasformando il grafico in un indicatore di "forza relativa".

I valori sono stai infatti normalizzati per permettere un confronto diretto tra variabili con scale differenti, facilitando l’identificazione delle specificità di ciascun cluster.

In [ ]:
# Cella 25 : Profilazione Multivariata dei Cluster tramite Radar Chart

# 1. Metriche chiave che definiscono il profilo di ogni cluster
features_radar = ['PIL_Log', 'Pop_Log', 'Aspettativa_di_Vita', 'Score_Medaglie']

# Etichette leggibili per gli assi del radar
etichette_radar = ['Ricchezza Statale\n(PIL)', 'Bacino Demografico\n(Popolazione)',
                   'Benessere Diffuso\n(Aspettativa di Vita)', 'Successo Sportivo\n(Score Medaglie)']

# 2. Media delle metriche per ogni cluster.
#    IMPORTANTE: si raggruppa per Cluster_Label (il NOME), non per il numero del
#    cluster, e si forza l'ordine con reindex — così i dati sono sempre associati
#    all'etichetta corretta, indipendentemente dalla numerazione del K-Means.
ordine_cluster = ['Nazioni Minori', 'Economie Emergenti', 'Grandi Potenze Olimpiche']
cluster_means = (df_analisi.groupby('Cluster_Label')[features_radar]
                           .mean()
                           .reindex(ordine_cluster))

# 3. Normalizzazione 0–1 per far convivere scale diverse sullo stesso grafico
scaler_radar = MinMaxScaler()
cluster_means_scaled = pd.DataFrame(
    scaler_radar.fit_transform(cluster_means),
    columns=cluster_means.columns,
    index=cluster_means.index          # l'indice sono i nomi dei cluster, in ordine
)

# 4. Geometria del radar (coordinate polari)
N = len(etichette_radar)
angoli = [n / float(N) * 2 * pi for n in range(N)]
angoli += angoli[:1]   # chiude il poligono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Palette: un colore fisso per ogni cluster, tramite dizionario (robusto all'ordine)
colori_dict = {
    'Nazioni Minori': '#D55E00',
    'Economie Emergenti': '#0072B2',
    'Grandi Potenze Olimpiche': '#009E73'
}

# 5. Un poligono per cluster. 'nome_cluster' è l'indice del DataFrame (il nome).
for nome_cluster, row in cluster_means_scaled.iterrows():
    valori = row.values.flatten().tolist()
    valori += valori[:1]   # chiude il poligono dei valori

    ax.plot(angoli, valori, linewidth=2.5, linestyle='solid',
            label=nome_cluster, color=colori_dict[nome_cluster])
    ax.fill(angoli, valori, alpha=0.15, color=colori_dict[nome_cluster])

# 6. Estetica
plt.xticks(angoli[:-1], etichette_radar, size=11, fontweight='bold')
ax.set_yticklabels([])
ax.spines['polar'].set_visible(False)
plt.grid(color='#E8E8E8', linestyle='--', linewidth=1)

plt.title('Identikit dei Cluster: Confronto Multidimensionale', size=15, y=1.1, fontweight='bold')
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11, frameon=False)

plt.tight_layout()
plt.show()

## Versione interattiva del radar chart (Cella 26)

Il radar chart statico sopra ha un limite pratico: le **Grandi Potenze Olimpiche** dominano la scala su tutte le dimensioni, rendendo le **Nazioni Minori** e le **Economie Emergenti** quasi illeggibili. Con la versione Plotly si può **cliccare sui cluster in legenda per mostrarli/nasconderli** — nascondendo le Grandi Potenze gli altri due gruppi diventano subito leggibili. Passando il mouse sui vertici si legge il valore normalizzato esatto per quella dimensione.

I valori mostrati sono normalizzati tra 0 e 1 (dove 1 = massimo nel dataset), stessa logica della versione statica.

In [ ]:
# Cella 26 : radar chart interattivo (Plotly)
# Stessa logica della Cella 25, ricostruita con Plotly per l'interattività.

def hex_to_rgba(hex_color, alpha=0.15):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

# Etichette senza '\n' (Plotly le mostrerebbe letteralmente)
etichette_plotly = [
    'Ricchezza Statale (PIL)',
    'Bacino Demografico (Popolazione)',
    'Benessere Diffuso (Aspettativa di Vita)',
    'Successo Sportivo (Score Medaglie)'
]

# Stessa palette a dizionario della Cella 25
colori_dict = {
    'Nazioni Minori': '#D55E00',
    'Economie Emergenti': '#0072B2',
    'Grandi Potenze Olimpiche': '#009E73'
}

fig = go.Figure()

# Un tracciato per cluster. 'nome_cluster' è l'indice (il nome) di cluster_means_scaled,
# che la Cella 25 ha già ordinato e associato correttamente ai dati.
for nome_cluster, row in cluster_means_scaled.iterrows():
    valori_norm = row.values.flatten().tolist()
    r_vals = valori_norm + [valori_norm[0]]          # chiude il poligono
    theta_vals = etichette_plotly + [etichette_plotly[0]]

    fig.add_trace(go.Scatterpolar(
        r=r_vals,
        theta=theta_vals,
        fill='toself',
        name=nome_cluster,
        line=dict(color=colori_dict[nome_cluster], width=2.5),
        fillcolor=hex_to_rgba(colori_dict[nome_cluster], 0.15),
        hovertemplate='<b>%{theta}</b><br>Valore normalizzato: %{r:.2f}'
                      f'<extra>{nome_cluster}</extra>'
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            tickfont=dict(size=10),
            gridcolor='#e0e0e0'
        ),
        angularaxis=dict(tickfont=dict(size=11))
    ),
    showlegend=True,
    legend=dict(title='Cluster (clicca per mostrare/nascondere)', x=1.05, y=1.0),
    title=dict(text='Identikit dei Cluster: Confronto Multidimensionale', font=dict(size=15)),
    template='plotly_white',
    width=720, height=540
)

fig.show()

## Analisi dei risultati: Profilazione Multidimensionale dei Cluster

Il Radar Chart consente di confrontare in modo simultaneo le principali dimensioni socio-economiche dei cluster, evidenziandone le differenze in termini di "forza relativa".

Grazie alla normalizzazione delle variabili, ogni asse rappresenta una scala comparabile (0–1), permettendo una lettura integrata del profilo di ciascun gruppo.

---

### 1. Cluster 2 – Dominanza Multidimensionale

Il Cluster 2 emerge chiaramente come il gruppo dominante su tutte le dimensioni analizzate:

- massimo livello di ricchezza (PIL)
- elevata popolazione
- alto livello di sviluppo umano (aspettativa di vita)
- performance olimpiche nettamente superiori

➡️ Interpretazione:
si tratta delle grandi potenze globali, caratterizzate da un vantaggio sistemico che si riflette in tutte le dimensioni considerate.

➡️ Insight chiave:
il successo olimpico non è isolato, ma parte di un ecosistema complesso in cui economia, demografia e sviluppo umano si rafforzano reciprocamente.

---

### 2. Cluster 1 – Squilibrio Strutturale

Il Cluster 1 presenta un profilo fortemente sbilanciato:

- elevata popolazione relativa
- bassa ricchezza e basso sviluppo
- performance sportive limitate

➡️ Interpretazione:
questo cluster rappresenta economie emergenti o Paesi ad alta densità demografica, in cui il capitale umano potenziale non viene pienamente convertito in risultati sportivi.

➡️ Insight chiave:
la sola disponibilità di popolazione non è sufficiente; è necessario un adeguato livello di risorse e organizzazione per trasformarla in performance.

---

### 3. Cluster 0 – Vincolo di scala

Il Cluster 0 mostra caratteristiche opposte rispetto al Cluster 1:

- bassa popolazione
- livello medio di benessere
- ricchezza limitata in termini assoluti
- performance sportive quasi nulle

➡️ Interpretazione:
si tratta di piccoli Paesi, spesso con un discreto livello di sviluppo, ma penalizzati dalla ridotta scala demografica ed economica.

➡️ Insight chiave:
la dimensione del sistema rappresenta un vincolo strutturale alla competitività olimpica.

---

### 4. Lettura trasversale dei cluster

Il Radar Chart evidenzia un pattern fondamentale:

- il successo olimpico cresce al crescere della combinazione tra:
  - ricchezza aggregata  
  - dimensione demografica  
  - sviluppo umano  

➡️ Nessuna singola variabile è sufficiente, ma è la loro combinazione a determinare il risultato.

---


### 5. Il divario che si restringe: l'aspettativa di vita

Un dettaglio che il grafico mostra ma che è facile non notare a un primo sguardo: sull'asse "Benessere Diffuso" (Aspettativa di Vita) il divario tra i cluster è molto più piccolo che sugli altri tre assi. Le Nazioni Minori, quasi invisibili su Ricchezza, Popolazione e Successo Sportivo, si avvicinano visibilmente alle Grandi Potenze su questa dimensione.

➡️ Interpretazione: il divario economico, demografico e sportivo tra le nazioni è enorme — quello nel benessere di base molto meno. Un Paese piccolo e "irrilevante" per PIL, popolazione e medaglie può comunque garantire ai suoi cittadini una qualità della vita paragonabile a quella delle superpotenze olimpiche.

➡️ Insight chiave: rafforza la tesi del progetto — il PIL compra il potenziale sportivo, ma non è un proxy del benessere complessivo di una nazione. Le due cose vanno tenute distinte.

---

### 📊 Conclusione

L’analisi multidimensionale conferma che i cluster identificati non sono semplici aggregazioni statistiche, ma riflettono modelli strutturali distinti di sviluppo socio-economico.

Il Radar Chart sintetizza in modo efficace il risultato principale dell’intero progetto:

➡️ il successo olimpico è un fenomeno sistemico, determinato dall’interazione tra risorse economiche, capitale umano e livello di sviluppo.

Questa evidenza rafforza l’interpretazione emersa nelle analisi precedenti, fornendo una visione integrata e conclusiva del fenomeno.


## Summary of Socio-Economic Cluster Differences

The clustering analysis identified three distinct socio-economic profiles based on 'PIL_Log', 'Pop_Log', 'Aspettativa_di_Vita', and 'Score_Medaglie'. These clusters highlight different aspects of economic development and their correlation with Olympic success.

### 1. Grandi Potenze Olimpiche (Cluster 2)

*   **Characteristics:** Dominant across all dimensions, including the highest levels of wealth (PIL), large populations, high human development (life expectancy), and significantly superior Olympic performance.
*   **Insight:** These nations possess a systemic advantage where economic power, demographic size, and human development mutually reinforce each other, leading to consistent Olympic success.

### 2. Economie Emergenti (Cluster 1)

*   **Characteristics:** Characterized by a strong imbalance, with relatively high populations but lower wealth and development levels. Their Olympic performance is limited.
*   **Insight:** While these countries have significant potential human capital due to their large populations, this potential isn't fully converted into sporting achievements, often due to a lack of adequate resources and organized sports infrastructure.

### 3. Nazioni Minori (Cluster 0)

*   **Characteristics:** These are smaller nations, often with decent individual well-being but limited absolute wealth and population size. Their Olympic performance is almost negligible.
*   **Insight:** The scale of the economic system acts as a structural constraint on Olympic competitiveness. Even high per capita GDP is not enough if the overall population and economic size are too small to support a competitive multi-sport system.

**Overall Pattern:** The analysis confirms a fundamental pattern: Olympic success increases with a combination of aggregate wealth, demographic size, and human development. No single variable is sufficient; it is their combined interaction that determines the outcome.

## Modellazione Predittiva Non Lineare: Random Forest
Per stimare la performance olimpica, la sola regressione lineare risulta insufficiente, in quanto assume relazioni puramente additive e costanti. Le dinamiche socio-economiche sono invece governate da interazioni complesse (es. l'effetto sinergico tra PIL e Popolazione).

È stato quindi implementato un modello Random Forest Regressor, capace di catturare autonomamente queste non-linearità. Per gestire la forte dispersione del target (l'esplosione della varianza per le superpotenze), è stata applicata una Target Transformation: il modello viene addestrato sul logaritmo dello Score Medaglie, per poi riconvertire le stime sulla scala reale tramite funzione esponenziale. Questo garantisce stabilità in fase di training e interpretabilità in fase di test. Nel modello è inoltre codificata esplicitamente la variabile binaria Host, per isolare il noto "vantaggio casalingo".

In [ ]:
# Cella 27 : Random Forest

# =========================
# 1. Costruzione nuove variabili
# =========================

# Variabile host (vantaggio casa)
df_analisi['Host'] = (df_analisi['Nazione'] == df_analisi['Paese_Ospitante']).astype(int)

# Gestione delle feature di lag per il modello:
# - 'Ha_Storico_Precedente' = 0/1, flag esplicito che indica se la Nazione ha gia'
#   partecipato in passato (informativo di per se': una prima partecipazione e' diversa
#   da una assenza di dati per errore)
# - i NaN nel lag (prime partecipazioni) vengono imputati a 0 SOLO ai fini del modello,
#   perche' il flag sopra permette al Random Forest di distinguere comunque i due casi
df_analisi['Ha_Storico_Precedente'] = df_analisi['Score_Medaglie_Lag1'].notna().astype(int)
df_analisi['Score_Medaglie_Lag1_model'] = df_analisi['Score_Medaglie_Lag1'].fillna(0)
df_analisi['Score_Medaglie_MediaMobile3_model'] = df_analisi['Score_Medaglie_MediaMobile3'].fillna(0)

# =========================
# 2. Selezione feature
# =========================

features_modello = [
    'PIL_Log',
    'PIL_ProCapite_Log',
    'Pop_Log',
    'PIL_x_Pop',
    'Aspettativa_di_Vita',
    'Iscrizioni_Scuola_Primaria_perc',
    'Popolazione_Urbana',
    'Anno',
    'Host',
    'Score_Medaglie_Lag1_model',
    'Score_Medaglie_MediaMobile3_model',
    'Ha_Storico_Precedente'
]

# Si rimuovono eventuali NaN residui sulle altre feature (i lag sono gia' stati gestiti sopra)
df_model = df_analisi.dropna(subset=features_modello + ['Score_Medaglie']).copy()

X = df_model[features_modello]
y = np.log1p(df_model['Score_Medaglie'])

# =========================
# 3. Train/Test split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 4. Random Forest
# =========================

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

# Predizioni in scala Log (per calcoli interni)
y_pred_log = rf.predict(X_test)

# Predizioni e Verità riportati alla scala Reale (per metriche e residui)
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# =========================
# 5. Metriche
# =========================
# R² in scala reale = metrica di riferimento del modello
# R² in scala log = solo per confronto train/test (diagnosi overfitting)

R2_scala_reale = r2_score(y_test_real, y_pred_real)

print("--- Modello Random Forest: METRICA DI RIFERIMENTO (scala reale, medaglie) ---")
print(f"R-squared (R2): {R2_scala_reale:.3f}")
print(f"MAE: {mean_absolute_error(y_test_real, y_pred_real):.2f} medaglie")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_real, y_pred_real)):.2f} medaglie")

print("\n--- Controllo overfitting (scala log, SOLO per confronto train/test) ---")
R2_train_log = rf.score(X_train, y_train)
R2_test_log = rf.score(X_test, y_test)
print(f"Train R² (scala log): {R2_train_log:.3f}")
print(f"Test R²  (scala log): {R2_test_log:.3f}")
print(f"Gap Train-Test: {R2_train_log - R2_test_log:.3f} -> piu' alto e' il gap, maggiore il rischio di overfitting")


--- Modello Random Forest: METRICA DI RIFERIMENTO (scala reale, medaglie) ---
R-squared (R2): 0.707
MAE: 2.71 medaglie
RMSE: 10.15 medaglie

--- Controllo overfitting (scala log, SOLO per confronto train/test) ---
Train R² (scala log): 0.944
Test R²  (scala log): 0.787
Gap Train-Test: 0.158 -> piu' alto e' il gap, maggiore il rischio di overfitting


## Analisi dei risultati: Modello Random Forest (con feature di lag)

L'utilizzo del modello Random Forest consente di catturare relazioni non lineari e interazioni complesse tra le variabili, superando i limiti della regressione lineare.

Il modello include le feature di lag (`Score_Medaglie_Lag1_model`, `Score_Medaglie_MediaMobile3_model`) e il flag `Ha_Storico_Precedente`, per catturare l'inerzia storica del sistema sportivo.

---

### 1. Capacità predittiva del modello

Il modello presenta le seguenti performance sulla scala reale (split casuale, random_state=42):

- **R² (test, scala reale): 0.707**
- **Test R² (scala log, diagnostico): 0.787**
- **Train R² (scala log, diagnostico): 0.944**

➡️ Interpretazione:

- Questo conferma empiricamente l'ipotesi della proposta: il trend storico della performance è un determinante rilevante del successo olimpico
- Il confronto diretto con lo stesso modello privo di feature di lag non è disponibile su split casuale in questa versione del notebook; è però disponibile su split temporale (Cella 44 più avanti): con lag R²=0.801, senza lag R²=0.692 — un guadagno comunque consistente

---

### 2. Confronto Train vs Test (Overfitting)

Il gap tra Train R² (0.944) e Test R² log (0.787) — pari a 0.158 — resta presente ma è interpretabile alla luce della cross-validation (Cella 32), dove il punteggio medio (0.784) è molto vicino al Test R² log dello split singolo, segno che il gap osservato non è un artefatto dello split scelto ma un comportamento tipico del Random Forest su questo dataset.

---

### 3. Nota su scala log vs scala reale

Come chiarito direttamente nel codice della Cella 27, i due R² (scala log e scala reale) **non sono direttamente confrontabili**: il primo è calcolato sul target trasformato `log1p(Score_Medaglie)` ed è utile solo per la diagnosi di overfitting; il secondo, calcolato sulle medaglie effettive, è la metrica di riferimento per valutare l'accuratezza pratica del modello e va citata come tale nel report.

---

### 4. Importanza delle feature di lag

Come atteso, le feature di lag risultano largamente le più importanti nel modello (si veda la Feature Importance in Cella 33): da sole, Lag1 e MediaMobile3 coprono circa l'82% dell'importanza totale (rispettivamente 69,5% e 12,4%). Questo è coerente con la letteratura sportiva: i sistemi sportivi nazionali tendono a essere persistenti nel tempo (investimenti, infrastrutture e talenti non si costruiscono né si perdono da un'edizione all'altra). Il dettaglio completo, con l'interpretazione del ruolo ridimensionato del PIL, è discusso nella sezione dedicata (Cella 33).

---

### 📊 Conclusione

Il modello Random Forest, arricchito con le feature di lag, conferma che il successo olimpico è un fenomeno complesso e dipendente sia da fattori strutturali (PIL, popolazione) sia dall'inerzia storica della performance sportiva di una Nazione. L'introduzione dello storico colma una delle lacune identificate rispetto alla proposta progettuale.


## Diagnostica del Modello: Analisi dei Residui e Varianza non Osservata

Per validare la tenuta del Random Forest e per comprendere i confini esplicativi del framework strutturale, si procede con l'analisi della distribuzione dei residui (la differenza puntuale tra le medaglie realmente vinte e quelle stimate dall'algoritmo).
In questo dominio il residuo non rappresenta un semplice "errore statistico", ma quantifica l'impatto di tutte le variabili qualitative non osservate — come tradizioni, investimenti diretti o efficienza organizzativa — che permettono a una Nazione di deviare (in positivo o in negativo) dal suo potenziale macroeconomico.

In [ ]:
# Cella 28
residuals = y_test_real - y_pred_real

plt.hist(residuals, bins=30)
plt.title("Distribuzione dei residui")
plt.xlabel("Errore")
plt.ylabel("Frequenza")
plt.show()

## Analisi dei risultati: Distribuzione dei Residui

La distribuzione dei residui consente di valutare la qualità del modello e, soprattutto, di interpretare la componente non spiegata del fenomeno.

Nel presente contesto, il residuo assume un significato economico rilevante: rappresenta la deviazione tra il risultato osservato e quello previsto dal modello, ovvero l’effetto complessivo dei fattori non inclusi nell’analisi.

---

### 1. Forma della distribuzione

L’istogramma mostra una distribuzione:

- fortemente concentrata attorno allo zero  
- con una leggera asimmetria positiva  
- caratterizzata dalla presenza di code estreme  

➡️ Interpretazione:

- la concentrazione intorno a zero indica che il modello, nella maggior parte dei casi, fornisce stime corrette  
- le code indicano la presenza di errori significativi per alcune osservazioni  

---

### 2. Asimmetria positiva

La distribuzione presenta una **coda verso destra**, ovvero:

- residui positivi → il modello **sottostima** il numero di medaglie  

➡️ Interpretazione:

esistono Paesi che performano significativamente meglio rispetto a quanto previsto dalle loro caratteristiche strutturali.

👉 Questi rappresentano i casi di **“over-performance”**

---

### 3. Residui negativi

I residui negativi rappresentano:

- Paesi che ottengono meno medaglie rispetto alle aspettative del modello  

➡️ Interpretazione:

questi casi possono riflettere inefficienze sistemiche, come:


## Identificazione di over e under performer

> **Residuo = quanto un Paese performa sopra o sotto le aspettative date le sue risorse.** Un residuo positivo indica una nazione che vince *più* medaglie di quante il suo PIL e la sua demografia farebbero prevedere; un residuo negativo indica una nazione che *spreca* il suo potenziale strutturale.

Questa lettura trasforma i residui da semplice errore statistico a **indicatore di efficienza del sistema sportivo nazionale**: non ciò che il modello sbaglia, ma ciò che i dati rivelano al di là delle risorse economiche.

In [ ]:
# Cella 29 : Identificazione Over / Under Performer

# 1. Creiamo una copia del test set per l'analisi
df_test = X_test.copy()

# 2. Recuperiamo il nome della Nazione dal dataframe originale usando l'indice
df_test['Nazione'] = df_analisi.loc[df_test.index, 'Nazione']
df_test['Cluster_Label'] = df_analisi.loc[df_test.index, 'Cluster_Label']

# 3. Inseriamo i risultati del modello
df_test['Reali'] = np.round(y_test_real, 1)      # Arrotondiamo per estetica
df_test['Predetti'] = np.round(y_pred_real, 1)
df_test['Residuo'] = np.round(df_test['Reali'] - df_test['Predetti'], 1)

# 4. Si estraggono i top 10
top_over = df_test.sort_values(by='Residuo', ascending=False).head(10)
top_under = df_test.sort_values(by='Residuo').head(10)

# 5. Si definiscono le colonne da mostrare a video (si nasconde i logaritmi per pulizia visiva)
colonne_display = ['Nazione', 'Anno', 'Reali', 'Predetti', 'Residuo']

print("🏆 TOP 10 OVER-PERFORMER (Hanno vinto molto più del previsto):")
display(top_over[colonne_display]
    .style.hide(axis='index').format({
        'Anno': '{:.0f}', 'Reali': '{:.1f}',
        'Predetti': '{:.1f}', 'Residuo': '{:.1f}'
    }))

print("\n📉 TOP 10 UNDER-PERFORMER (Hanno vinto meno del previsto):")
display(top_under[colonne_display]
    .style.hide(axis='index').format({
        'Anno': '{:.0f}', 'Reali': '{:.1f}',
        'Predetti': '{:.1f}', 'Residuo': '{:.1f}'
    }))

🏆 TOP 10 OVER-PERFORMER (Hanno vinto molto più del previsto):

📉 TOP 10 UNDER-PERFORMER (Hanno vinto meno del previsto):


## Versione per il sito: over/under performer in una timeline

Le tabelle con i top 10 funzionano bene nel notebook ma poco su un sito. Qui ogni bolla è una nazione in una certa edizione: la dimensione indica quanto si discosta dal previsto, il colore se è over o under performance. In un colpo d'occhio si vede chi, quando e quanto.

In [ ]:
# Cella 30 : timeline degli over/under performer, vista d'insieme per edizione

chart_over_under_web = alt.Chart(df_test).mark_circle(opacity=0.75, stroke='white', strokeWidth=0.5).encode(
    x=alt.X('Anno:O', title='Anno Olimpico'),
    y=alt.Y('Nazione:N', sort=alt.EncodingSortField(field='Residuo', op='mean', order='descending'),
            title=None),
    size=alt.Size('Residuo:Q', title='Entità del residuo (assoluto)',
                   scale=alt.Scale(range=[20, 500]), legend=alt.Legend(format='.0f')),
    color=alt.Color('Residuo:Q', title='Residuo (Reale - Predetto)',
                     scale=alt.Scale(scheme='redblue', domainMid=0)),
    tooltip=[
        alt.Tooltip('Nazione:N'),
        alt.Tooltip('Anno:O'),
        alt.Tooltip('Cluster_Label:N', title='Cluster'),
        alt.Tooltip('Reali:Q', title='Medaglie reali', format='.1f'),
        alt.Tooltip('Predetti:Q', title='Medaglie predette', format='.1f'),
        alt.Tooltip('Residuo:Q', title='Residuo', format='.1f')
    ]
).transform_filter(
    'abs(datum.Residuo) > 15'  # solo i casi più rilevanti, altrimenti il grafico è illeggibile
).properties(
    width=700, height=420,
    title='Over/Under Performer per Edizione (blu = over-performance, rosso = under-performance)'
)

chart_over_under_web.save('over_under_performer_timeline_interattivo.html')

chart_over_under_web

## Analisi dei risultati: Over e Under Performer

L'identificazione degli over e under performer consente di analizzare in modo puntuale le deviazioni più significative rispetto alle previsioni del modello, trasformando i residui in uno strumento interpretativo concreto.

---

### 1. Over-performer: eccellenza oltre il potenziale

Tra le nazioni che ottengono performance significativamente superiori rispetto alle attese emergono, in ordine di residuo:

- **ROC (2020)** → +125,4 medaglie rispetto al previsto
- **Federazione Russa (1996)** → +113,3
- **Germania (1964)** → +67,9
- **Italia (1984)** → +49,6
- **Ungheria (1992)** → +33,1
- Svezia (1972), Romania (2000), Australia (1964), Regno Unito (2016), Bielorussia (1996) → residui tra +22 e +24

➡️ Interpretazione:

Questi Paesi rappresentano esempi di **over-performance strutturale**, ovvero capacità di ottenere risultati molto superiori rispetto alle risorse economiche e demografiche del momento.

Le possibili spiegazioni includono:

- forte tradizione sportiva
- programmi centralizzati e politiche pubbliche mirate
- investimenti strategici in discipline specifiche
- contesti geopolitici o storici particolari (es. URSS/Russia in epoca sovietica e post-sovietica)

➡️ Caso emblematico:
la Federazione Russa compare due volte nella top-5, in direzioni opposte a seconda dell'edizione (qui come forte over-performer nel 1996, più avanti come marcato under-performer nel 2008) — un promemoria che l'appartenenza a un cluster economico non predetermina la direzione del residuo in una singola edizione.

Le spiegazioni (tradizione sportiva, programmi centralizzati, contesti geopolitici) sono interpretazioni qualitative dei residui, non conclusioni statistiche — il modello segnala *dove* la spiegazione economica è insufficiente, non *perché*.

---

### 2. Under-performer: inefficienza relativa (o boicottaggi)

Tra le nazioni con performance inferiori alle attese emergono, in ordine di residuo:

- **Bulgaria (1984)** → 0 medaglie reali contro 52,5 previste (-52,5)
- **Giappone (1980)** → 0 medaglie reali contro 46,3 previste (-46,3)
- **Canada (1988)** → -42,3
- **Federazione Russa (2008)** → -32,1
- **Regno Unito (1996)** → -18,7

➡️ Interpretazione:

Qui il dato più rilevante non è l'inefficienza sportiva, ma un limite del modello: **i due under-performer più estremi (Bulgaria 1984, Giappone 1980) corrispondono esattamente alle edizioni boicottate** (Mosca 1980 e Los Angeles 1984, boicottaggi contrapposti della Guerra Fredda). Il modello non riceve alcuna informazione sui boicottaggi, quindi prevede una performance "normale" in base al potenziale economico-storico e registra come forte under-performance l'assenza di squadra — che non è affatto un'inefficienza del sistema sportivo, ma un evento politico esterno.

Per i casi restanti (Canada 1988, Federazione Russa 2008, Regno Unito 1996) le cause possono includere:

- variabilità fisiologica tra edizioni
- discontinuità nei programmi sportivi
- fattori specifici dell'edizione olimpica

---

### 3. Valore interpretativo dei residui estremi

L'analisi evidenzia che:

- alcuni Paesi superano sistematicamente le aspettative
- altri, pur avendo risorse elevate, non riescono a esprimerle pienamente
- alcuni residui estremi non riflettono affatto l'efficienza del sistema sportivo, ma eventi esterni (boicottaggi) che il modello non può conoscere

➡️ Questo conferma che:

👉 il successo olimpico non è determinato esclusivamente da PIL e popolazione
👉 ma anche da fattori qualitativi, strategici e, in alcuni casi, puramente politici/contestuali

---

### 4. Collegamento con le analisi precedenti

Questi risultati sono coerenti con:

- l'analisi dei residui (presenza di code distribuzionali)
- il clustering (differenze strutturali tra Paesi)
- la regressione (limiti esplicativi del modello lineare)

➡️ Gli over-performer rappresentano i casi di alta efficienza (o di contesto storico favorevole)
➡️ Gli under-performer rappresentano inefficienze relative — oppure, come nei due casi più estremi, l'effetto di un boicottaggio che il modello non può prevedere

---

### 📊 Conclusione

L'identificazione degli over e under performer rappresenta il livello più avanzato dell'analisi, in quanto:

- traduce un output statistico in evidenza concreta
- evidenzia i limiti del modello (compresa l'incapacità di conoscere eventi esterni come i boicottaggi)
- introduce una forte componente interpretativa

Il risultato fondamentale è che:

➡️ il modello definisce un "potenziale economico teorico"
➡️ i residui misurano la capacità reale di trasformarlo in successo sportivo — capacità che può essere impedita anche da fattori del tutto esterni all'efficienza sportiva

Questa distinzione è cruciale per comprendere la vera natura del fenomeno olimpico.


## Identificazione degli outlier

Dopo l'esplorazione concettuale degli Over-Performer, si procede con un'estrazione statistica. Vengono definite outlier tutte le osservazioni il cui residuo (l'errore di stima) supera la soglia di 2 deviazioni standard dalla media degli errori. Questo filtro isola unicamente i casi in cui la deviazione dal modello non può essere considerata una normale fluttuazione statistica, ma rappresenta un'anomalia strutturale profonda.

In [ ]:
# Cella 31 : #Outlier
threshold = residuals.std() * 2

outliers = df_test[abs(df_test['Residuo']) > threshold]

# Vengono portate Nazione/Anno/Reali/Predetti/Residuo all'inizio e si nasconde l'indice
# (altrimenti sembra una colonna ID, fuorviante)
colonne_principali = ['Nazione', 'Anno', 'Reali', 'Predetti', 'Residuo']
altre_colonne = [c for c in outliers.columns if c not in colonne_principali]
outliers_ordinati = outliers[colonne_principali + altre_colonne].copy()

# Anno è float nel dataset (es. 1984.0) -> intero per la visualizzazione
outliers_ordinati['Anno'] = outliers_ordinati['Anno'].astype(int)

# Formato per colonna: ogni tipo di dato ha la sua precisione appropriata
formato = {
    'Reali':                          '{:.1f}',
    'Predetti':                       '{:.1f}',
    'Residuo':                        '{:.1f}',
    'PIL_Log':                        '{:.2f}',
    'PIL_ProCapite_Log':              '{:.2f}',
    'Pop_Log':                        '{:.2f}',
    'PIL_x_Pop':                      '{:.1f}',
    'Aspettativa_di_Vita':            '{:.1f}',
    'Iscrizioni_Scuola_Primaria_perc':'{:.1f}',
    'Popolazione_Urbana':             '{:,.0f}',
    'Host':                           '{:.0f}',
    'Score_Medaglie_Lag1_model':      '{:.1f}',
    'Score_Medaglie_MediaMobile3_model': '{:.1f}',
    'Ha_Storico_Precedente':          '{:.0f}',
}
# Applica solo le colonne effettivamente presenti (robusto a eventuali variazioni del dataset)
formato_applicabile = {k: v for k, v in formato.items() if k in outliers_ordinati.columns}

print("Outliers individuati:")
display(outliers_ordinati.style.hide(axis='index').format(formato_applicabile))

Outliers individuati:


## Analisi dei risultati: Identificazione degli Outlier

L'identificazione degli outlier consente di isolare le osservazioni che presentano deviazioni estreme rispetto alle previsioni del modello, ovvero i casi in cui il residuo supera la soglia di ±2 deviazioni standard (soglia ≈ ±20,3 medaglie, con deviazione standard dei residui pari a 10,13).

Questa procedura permette di distinguere tra:
- errori statistici fisiologici
- anomalie strutturali significative

---

### 1. Significato statistico degli outlier

Gli outlier rappresentano osservazioni per cui:

➡️ la differenza tra valore reale e stimato è troppo elevata per essere attribuita a variazioni casuali

➡️ quindi indicano:
- deviazioni sistematiche
- fenomeni non modellati
- eventi eccezionali

Il metodo identifica **15 outlier** su 516 osservazioni di test (≈2,9%), di cui 11 positivi e 4 negativi.

---

### 2. Over-performance estrema

Gli 11 outlier positivi, in ordine di residuo:

- **ROC (2020)** → +125,4
- **Federazione Russa (1996)** → +113,3
- **Italia (1984)** → +49,6
- **Ungheria (1992)** → +33,1
- **Romania (2000)** → +23,4
- **Regno Unito (2016)** → +23,0
- **Svezia (1972)** → +23,5
- **Australia (1964)** → +23,4
- **Germania (1964)** → +67,9
- **Bielorussia (1996)** → +22,7
- **Cuba (1976)** → +20,7

➡️ Interpretazione:

Questi casi mostrano una **over-performance estrema**, difficilmente spiegabile attraverso variabili economiche standard.

👉 Possibili driver:
- sistemi sportivi altamente centralizzati
- specializzazione in discipline strategiche
- contesti geopolitici (es. Guerra Fredda, transizione post-sovietica)
- investimenti mirati e continui

---

### 3. Under-performance estrema

I 4 outlier negativi:

- **Bulgaria (1984)** → -52,5 (edizione boicottata da URSS e blocco sovietico)
- **Giappone (1980)** → -46,3 (edizione boicottata da USA e alleati)
- **Canada (1988)** → -42,3
- **Federazione Russa (2008)** → -32,1

➡️ Interpretazione:

Due dei quattro outlier negativi (Bulgaria 1984, Giappone 1980) sono **direttamente spiegati dai boicottaggi olimpici** della Guerra Fredda: il modello prevede una performance basata sul potenziale economico-storico ma non ha alcuna informazione sull'assenza politica della squadra da quell'edizione. Non sono quindi casi di inefficienza sportiva, ma di un evento esogeno non presente tra le feature. Canada 1988 e Federazione Russa 2008 restano invece casi di sotto-performance genuina, da imputare a fattori del sistema sportivo o dell'edizione specifica.

---

### 4. Coerenza con le analisi precedenti

Gli outlier identificati sono coerenti con:

- l'analisi dei residui (code della distribuzione)
- l'individuazione degli over/under performer (Cella 31)
- i limiti del modello Random Forest

➡️ Questo conferma che:

👉 esiste una componente sistematica non osservata (in parte riconducibile a eventi storici noti come i boicottaggi, in parte a fattori sportivi qualitativi)
👉 il modello, pur performando bene, non cattura completamente il fenomeno

---

### 5. Interpretazione economica

Gli outlier rappresentano i casi più interessanti dal punto di vista analitico, in quanto:

- evidenziano i limiti delle variabili strutturali
- isolano contesti in cui entrano in gioco fattori qualitativi o storico-politici
- permettono di identificare modelli nazionali specifici

➡️ In particolare:

- gli outlier positivi indicano **efficienza elevata** (o contesto storico favorevole)
- gli outlier negativi indicano **inefficienza relativa** — tranne i due casi di boicottaggio, che sono un limite noto e spiegabile del modello, non un'anomalia da interpretare sportivamente

---

### 📊 Conclusione

L'identificazione degli outlier rappresenta il livello più rigoroso dell'analisi dei residui, permettendo di distinguere tra variazioni casuali e anomalie strutturali.

Il risultato principale è che:

➡️ esistono Paesi che deviano sistematicamente dalle previsioni
➡️ queste deviazioni non sono casuali: due dei quattro casi negativi più estremi hanno una spiegazione storica precisa e verificabile (i boicottaggi del 1980 e del 1984), mentre gli altri riflettono caratteristiche non catturate dal modello

Questo rafforza la conclusione generale del progetto:

👉 il successo olimpico non è interamente spiegabile da variabili economiche, ma dipende anche da fattori istituzionali, culturali, strategici e, in alcuni casi, da eventi politici esterni che nessun modello economico può prevedere.


##Stress Test del Modello: Cross-Validation a K-Fold

Per garantire il massimo rigore metodologico ed escludere che le metriche ottenute siano frutto di una suddivisione casuale favorevole (Train/Test split bias), sottoponiamo il Random Forest a uno "stress test" tramite Cross-Validation a 5 fold.

Questa tecnica addestra e valida iterativamente il modello su sottoinsiemi rotanti del dataset, fornendo una stima spietata della sua capacità di generalizzazione e misurando la stabilità del fenomeno sportivo al variare del campione geopolitico.

In [ ]:
# Cella 32 : Cross Validation

scores = cross_val_score(rf, X, y, cv=5, scoring='r2')

print("Cross-validation R²:")
print(scores)
print(f"Media R²: {scores.mean():.3f}")

Cross-validation R²:
[0.78223936 0.82519806 0.73353381 0.82020456 0.7597755 ]
Media R²: 0.784


## Analisi dei risultati: Cross-Validation a K-Fold

La Cross-Validation a 5 fold consente di valutare in modo robusto la capacità di generalizzazione del modello, riducendo il rischio che i risultati siano influenzati da una specifica suddivisione train/test.

Con il dataset corrente (dataset_indicatori_definitivo_5.csv) il punteggio medio di cross-validation a 5 fold è **0.784** (scala log), molto vicino al Test R² log dello split singolo (0.787, Cella 27) — segno che il modello è stabile e il risultato non dipende da una scelta fortunata dello split.

---

### 1. Confronto con il test set

➡️ Interpretazione generale (valida indipendentemente dai numeri esatti):

- se il valore medio della cross-validation è vicino al Test R² in scala reale, il modello generalizza bene e il rischio di overfitting è limitato
- un gap ancora ampio tra train e test (scala log) è normale nei modelli ad alta flessibilità come il Random Forest, ma va sempre discusso esplicitamente nel report, come fatto qui

👉 Rispetto alla versione del notebook priva di feature storiche, l'aggiunta del lag riduce la dipendenza del modello dalle sole variabili macroeconomiche statiche, rendendolo più stabile tra i fold (Paesi con uno storico recente sono naturalmente più prevedibili).

---

### 2. Variabilità tra i fold

I punteggi tra i fold possono ancora variare (alcuni fold conterranno più prime-partecipazioni o più outlier geopolitici di altri), ma una minore dispersione complessiva rispetto alla versione precedente è un segnale di maggiore robustezza del modello.

---

### 3. Significato economico della variabilità

La dispersione residua tra i fold suggerisce che il successo olimpico non segue una relazione uniforme tra Paesi, ma varia in funzione di contesto geografico, sviluppi storici e modelli istituzionali — il modello è stabile **in media**, ma non perfettamente trasferibile a ogni sottogruppo.

---

### 📊 Conclusione

La cross-validation, ora calcolata su un modello che include lo storico delle edizioni precedenti, conferma che il Random Forest è più robusto e meglio calibrato rispetto alla versione basata solo su variabili strutturali statiche. Il miglioramento osservato va riportato esplicitamente nel technical report come evidenza che l'integrazione delle feature di lag richieste dalla proposta migliora concretamente sia l'accuratezza sia la stabilità del modello.

## Interpretabilità del Modello Globale: Feature Importance

Nonostante i modelli ad albero complessi operino come architetture black-box, è possibile estrarre un'informazione globale calcolando l'importanza relativa (Gini Importance) di ciascuna variabile. Questo indicatore misura la frequenza e la capacità con cui ogni singola feature è riuscita a ridurre l'errore o la varianza all'interno delle diramazioni della "foresta". L'analisi delle feature importance ci consente di stilare una gerarchia oggettiva dei driver socio-economici che determinano il potenziale olimpico

In [ ]:
# Cella 33 : per Insight
# Importanza variabili

importances = pd.Series(rf.feature_importances_, index=features_modello)

importances = importances.sort_values(ascending=True)

importances.plot(kind='barh')
plt.title("Importanza delle variabili - Random Forest")
plt.show()

print(importances)

Host                                 0.000000
Ha_Storico_Precedente                0.006279
PIL_ProCapite_Log                    0.014710
Pop_Log                              0.015054
PIL_x_Pop                            0.017856
Anno                                 0.017975
Popolazione_Urbana                   0.018193
Iscrizioni_Scuola_Primaria_perc      0.020243
Aspettativa_di_Vita                  0.022719
PIL_Log                              0.048546
Score_Medaglie_MediaMobile3_model    0.123581
Score_Medaglie_Lag1_model            0.694845
dtype: float64


## Analisi dei risultati: Importanza delle Variabili

La feature importance del Random Forest quantifica il contributo di ciascuna variabile alla capacità predittiva del modello. I risultati cambiano in modo sostanziale rispetto alla versione del modello senza lag.

---

### 1. Dominio assoluto delle feature storiche

La variabile più importante è di gran lunga:

- **Score_Medaglie_Lag1_model → 0.695 (~69,5%)**

Seguita da:

- **Score_Medaglie_MediaMobile3_model → 0.124 (~12,4%)**

Insieme le due feature di lag coprono circa l'**82% dell'importanza totale** del modello. Questo risultato dice che il predittore più forte del risultato olimpico di una nazione in una certa edizione è semplicemente **quanto ha fatto bene nelle edizioni precedenti**: i sistemi sportivi nazionali hanno una fortissima inerzia nel tempo.

*Nota per l'orale:* questo bilanciamento (Lag1 quasi 70%) non è cambiato con le correzioni al dataset di questa sessione — è sostanzialmente identico anche usando il dataset con il solo fix Germania/Russia. Il lag resta nettamente il fattore dominante del modello, più di quanto suggerito da versioni precedenti del progetto.

---

### 2. Ruolo ridimensionato del PIL

Con il lag nel modello, il PIL scende a un ruolo molto più contenuto:

- **PIL_Log → ~4,9%**
- **PIL_ProCapite_Log → ~1,5%**
- **PIL_x_Pop → ~1,8%**

Questo non significa che il PIL sia irrilevante: significa che, *controllando per lo storico recente*, la variabile economica aggiunge poco alla previsione. Il PIL è già in parte "incorporato" nel lag — le nazioni ricche tendevano a fare bene in passato, e continuano a farlo. La feature importance misura il contributo marginale, non l'importanza assoluta.

---

### 3. Variabili socio-strutturali

- **Aspettativa_di_Vita → ~2,3%**
- **Iscrizioni_Scuola_Primaria_perc → ~2,0%**
- **Popolazione_Urbana → ~1,8%**
- **Anno → ~1,8%**
- **Pop_Log → ~1,5%**

Contribuiscono in modo distribuito e comparabile tra loro, catturando dimensioni di sviluppo umano e urbanizzazione.

---

### 4. Ha_Storico_Precedente e Host

- **Ha_Storico_Precedente → ~0,6%**: il flag che distingue le prime partecipazioni ha un contributo minimo — il modello riesce a gestirlo principalmente attraverso il valore del lag (= 0 per le prime partecipazioni).
- **Host → 0,000**: nessuna importanza diretta per il vantaggio casalingo, coerente con quanto già discusso nella Cella 29.

---

### ⚠️ Nota metodologica

La Gini importance misura il contributo predittivo, non la causalità. In presenza di feature correlate (es. lag e PIL, che tendono entrambi ad essere alti nelle stesse nazioni) il modello tende ad attribuire l'importanza a una sola delle due, penalizzando l'altra — questo spiega in parte la forte dominanza del lag e il ridimensionamento del PIL, che non vanno letti come "il PIL non conta" ma come "il PIL conta, ma il lag lo riassume già in gran parte".


## Interpretabilità Avanzata e Direzionale: Analisi SHAP

Mentre la Feature Importance tradizionale quantifica la frequenza globale di utilizzo di una variabile nell'albero, l'analisi SHAP (SHapley Additive exPlanations), fondata sulla teoria dei giochi cooperativi, ci permette di calcolare l'impatto marginale di ogni variabile su ogni singola previsione.

Questo approccio trasforma il Random Forest in un modello trasparente, consentendoci di osservare simultaneamente tre dimensioni vitali: l'importanza della variabile, la direzione del suo impatto (positivo o negativo) e la presenza di effetti non lineari.

In [ ]:
# Cella 34

# Inizializza l'explainer sul modello Random Forest
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Summary plot: ogni punto è un'osservazione del test set
# colore = VALORE DELLA FEATURE (rosso = valore alto, blu = valore basso)
# posizione asse X = impatto sulla previsione (positivo = aumenta lo score)
shap.summary_plot(shap_values, X_test, show=False)
plt.title('Impatto delle feature sulla previsione\n'
          'Colore: valore della feature  |  rosso = alto  |  blu = basso',
          fontsize=11, pad=12)
# Etichetta esplicita sulla barra colori (asse destro del plot)
cb_ax = plt.gcf().axes[-1]
cb_ax.set_ylabel('Valore della feature\n(normalizzato)', fontsize=9)
plt.tight_layout()
plt.show()

# Force plot interattivo: passando il mouse si vede il contributo esatto
# di ogni feature per ogni singola osservazione del test set
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values, X_test)

## Versione per il sito: importanza delle feature con direzione

Il summary plot SHAP "classico" (Cella 34) è molto informativo per un pubblico tecnico, ma poco leggibile per un visitatore generico del sito. Viene quindi proposta una versione semplificata che mantiene il rigore analitico risultando più accessibile: un bar chart dell'importanza delle feature, colorato in base alla **direzione media dell'effetto SHAP** (valori alti della feature aumentano o diminuiscono la previsione, in media).

In [ ]:
# Cella 35 : importanza delle feature con indicazione della direzione (SHAP)

# Correlazione feature-SHAP: più robusta della media per feature con molti zeri (es. lag)
# risponde a "feature alta -> SHAP positivo o negativo?" in modo più affidabile
corr_feature_shap = np.array([
    np.corrcoef(X_test[feat].values, shap_values[:, j])[0, 1]
    for j, feat in enumerate(features_modello)
])

shap_importance_df = pd.DataFrame({
    'Feature': features_modello,
    'Importanza': rf.feature_importances_,
    'Corr_Feature_SHAP': corr_feature_shap
}).sort_values('Importanza', ascending=False)

shap_importance_df['Direzione_Effetto'] = shap_importance_df['Corr_Feature_SHAP'].apply(
    lambda v: 'Aumenta lo Score Medaglie' if v > 0 else 'Riduce lo Score Medaglie'
)

chart_feature_importance_web = alt.Chart(shap_importance_df).mark_bar().encode(
    x=alt.X('Importanza:Q', title='Importanza della Feature (Random Forest)'),
    y=alt.Y('Feature:N', sort='-x', title=None),
    color=alt.Color(
        'Direzione_Effetto:N',
        title='Direzione dell\'effetto',
        scale=alt.Scale(
            domain=['Aumenta lo Score Medaglie', 'Riduce lo Score Medaglie'],
            range=['#009E73', '#D55E00']
        )
    ),
    tooltip=[
        alt.Tooltip('Feature:N'),
        alt.Tooltip('Importanza:Q', format='.3f'),
        alt.Tooltip('Corr_Feature_SHAP:Q', format='.3f', title='Correlazione feature-SHAP'),
        alt.Tooltip('Direzione_Effetto:N', title='Direzione')
    ]
).properties(
    width=600, height=320,
    title='Importanza delle Feature — verde: feature alta aumenta lo score, arancione: lo riduce'
)

chart_feature_importance_web.save('feature_importance_shap_interattivo.html')

chart_feature_importance_web

## Analisi dei risultati: Interpretabilità Avanzata con SHAP

L’analisi SHAP consente di superare i limiti della feature importance tradizionale, introducendo una lettura **direzionale e locale** delle variabili.

Ogni punto nel grafico rappresenta una specifica osservazione, mentre:
- la posizione sull’asse orizzontale indica l’impatto sulla previsione
- il colore rappresenta il valore della variabile (rosso = alto, blu = basso)

---

### 1. PIL_Log: driver dominante e direzionale

La variabile più influente è:

- **PIL_Log**

➡️ Evidenza:

- valori elevati (rosso) → impatti fortemente positivi (a destra)
- valori bassi (blu) → impatti negativi

➡️ Interpretazione:

esiste una relazione chiaramente **monotona e positiva**:
- maggiore PIL → maggiore numero previsto di medaglie  

👉 Questo conferma in modo robusto e direzionale il risultato già emerso:
il PIL è il driver principale del successo olimpico.

---

### 2. Aspettativa di vita: effetto positivo ma non lineare

- valori elevati → impatti generalmente positivi  
- ma con dispersione significativa  

➡️ Interpretazione:

l’effetto è positivo ma:
- non perfettamente lineare  
- variabile a seconda del contesto  

👉 Questo suggerisce che il benessere contribuisce al successo solo in combinazione con altre variabili.

---

### 3. Feature di interazione (PIL_x_Pop)

La variabile di interazione mostra:

- impatti sia positivi che negativi  
- distribuzione ampia  

➡️ Interpretazione:

l’effetto combinato tra PIL e popolazione è:

- **non lineare**
- dipendente dal contesto  

👉 Insight:
non basta essere grandi o ricchi → conta **come le due dimensioni interagiscono**

---

### 4. Variabili strutturali secondarie

Variabili come:

- iscrizione scolastica  
- popolazione urbana  
- anno  

mostrano:

- impatti più contenuti  
- maggiore dispersione attorno allo zero  

➡️ Interpretazione:

queste variabili:
- contribuiscono al modello  
- ma non determinano direttamente il risultato  

👉 hanno un ruolo **complementare**

---

### 5. Popolazione e PIL pro capite

- effetti distribuiti e non sempre unidirezionali  

➡️ Interpretazione:

- la popolazione da sola non garantisce successo  
- il PIL pro capite ha impatto limitato  

👉 conferma che:
➡️ è la **massa economica aggregata**, non la ricchezza media, a guidare il fenomeno  

---

### 6. Variabile Host

La variabile Host presenta:

- impatto trascurabile  

➡️ Interpretazione:

coerentemente con la feature importance:
- il vantaggio casalingo non emerge come driver principale  

👉 ma:
non si esclude un effetto locale non catturato dal modello

---

### 📊 Insight fondamentale

L’analisi SHAP introduce un risultato chiave:

➡️ il successo olimpico è guidato da relazioni:
- **non lineari**
- **dipendenti dal contesto**
- **basate sull’interazione tra variabili**

---

### 📊 Conclusione

L’interpretabilità tramite SHAP consente di trasformare il modello Random Forest da black-box a strumento analitico.

I risultati confermano che:

- il PIL è il driver dominante e direzionale  
- le altre variabili agiscono in modo non lineare e interattivo  
- il successo olimpico emerge da una combinazione complessa di fattori  

Questo rappresenta il punto più avanzato dell’analisi:

➡️ non solo prevedere il fenomeno  
➡️ ma comprenderne la struttura interna e le dinamiche causali parziali

## Confronto tra modelli: Definizione della Baseline e Test della Non-Linearità

Per giustificare l'impiego di algoritmi complessi e a minor interpretabilità nativa (come i metodi ensemble), è prassi metodologica consolidata confrontarne le performance con una baseline lineare standard.

Viene quindi eseguita una Regressione Lineare Multipla sulle medesime feature.

Questo test permette di verificare l'ipotesi centrale dello studio: le dinamiche del successo olimpico sono guidate da interazioni sinergiche e ritorni non lineari che un modello puramente additivo non è in grado di catturare.

In [ ]:
# Cella 36 : Confronto modelli

# Creiamo una "pipeline" che in automatico scala i dati (media 0, varianza 1)
# e dopo DOPO applica la regressione lineare.
lr = make_pipeline(StandardScaler(), LinearRegression())

# Addestriamo la pipeline

lr.fit(X_train, y_train)

# Predizioni in scala log e successiva conversione in reale
y_pred_lr_log = lr.predict(X_test)
y_pred_lr_real = np.expm1(y_pred_lr_log)

# Confronto R2 (tutto rigorosamente in scala reale)
print("--- Confronto Modelli (Scala Reale) ---")
print(f"Random Forest R2: {r2_score(y_test_real, y_pred_real):.3f}")
print(f"Linear Regression R2: {r2_score(y_test_real, y_pred_lr_real):.3f}")

--- Confronto Modelli (Scala Reale) ---
Random Forest R2: 0.707
Linear Regression R2: 0.339


## Analisi dei risultati: Confronto tra Modelli

Il confronto tra Random Forest e Regressione Lineare rappresenta un passaggio metodologico fondamentale per validare la necessità di modelli non lineari.

---

### 1. Risultati empirici (eseguiti con le feature di lag)

Le performance dei due modelli (in scala reale) sono:

- **Random Forest → R² = 0.707**
- **Regressione Lineare → R² fortemente negativo** (es. valori dell'ordine di -10/-20 a seconda del seed/split)

Un R² negativo non significa solo "modello debole": significa che il lineare predice peggio della media del target. L'amplificazione esponenziale di `expm1()` su piccoli errori in scala log è la causa tecnica — non un fallimento concettuale della regressione lineare.

➡️ Questo è di per sé un risultato utile: mostra che la trasformazione log, necessaria per stabilizzare la varianza del target, **interagisce diversamente con i due modelli**. Il Random Forest, basato su split e medie locali, è molto più robusto a questo effetto rispetto a un modello lineare che estrapola linearmente nello spazio trasformato.

**Questo risultato è un argomento strutturale a favore del Random Forest**, non solo un problema tecnico da giustificare: se la regressione lineare crolla così drasticamente su questo dataset, è perché le relazioni tra PIL, demografia e medaglie sono genuinamente non lineari. Il modello lineare non fallisce per caso — fallisce perché il fenomeno non è lineare. Il Random Forest è la scelta metodologicamente corretta proprio per questo.

---

### 2. Nota metodologica sul modello lineare

Il modello lineare ha un ruolo preciso in questa analisi: è utile come **strumento di interpretazione generale** (la direzione e l'ordine di grandezza dei coefficienti aiutano a capire le relazioni nel dato) ma non è adatto all'**inferenza causale puntuale** (i coefficienti non possono essere letti come effetti causali, anche a causa della multicollinearità tra PIL, PIL pro capite e popolazione). Non è un limite da scusarsi — è un confine metodologico esplicito che rafforza la scelta del Random Forest come strumento predittivo principale.

### 3. Interpretazione statistica

Al di là del valore esatto, il confronto conferma che:

➡️ una parte significativa della variabilità del fenomeno non è spiegabile tramite relazioni lineari semplici

➡️ ma richiede:
- interazioni tra variabili
- effetti di soglia
- relazioni non additive

👉 La regressione lineare, assumendo effetti indipendenti e costanti, non è in grado di catturare queste dinamiche — e risulta inoltre più fragile rispetto alle trasformazioni non lineari del target (log/expm1) necessarie per gestire la natura fortemente asimmetrica del numero di medaglie.

---

### 4. Evidenza delle non-linearità

Il miglioramento del Random Forest conferma che:

- l'impatto del PIL non è costante
- la popolazione amplifica o attenua gli effetti
- lo storico recente (feature di lag) interagisce con le altre variabili in modo non lineare
- le variabili socio-economiche interagiscono tra loro

➡️ Questo è coerente con:
- l'analisi SHAP (effetti direzionali e non lineari)
- il clustering (presenza di gruppi strutturali)
- l'analisi dei residui (varianza non spiegata)

---

### 5. Significato economico

Dal punto di vista economico, il risultato implica che:

➡️ il successo olimpico non segue una funzione lineare semplice

ma è il risultato di:
- **sinergie tra fattori economici, demografici e storici**
- **rendimenti non costanti delle risorse**
- **effetti combinati difficilmente isolabili**

👉 In altri termini: non basta aumentare una variabile → conta il contesto in cui essa opera, inclusa la storia recente della Nazione.

---

### 6. Validità dell'approccio metodologico

Il confronto giustifica pienamente l'utilizzo del Random Forest:

- maggiore accuratezza
- capacità di modellare complessità
- maggiore robustezza alla trasformazione log/expm1 del target

## Confronto con la letteratura di riferimento

Prima dell'analisi dei residui, i risultati ottenuti vengono confrontati con i quattro studi citati nella proposta di progetto, per inquadrare il lavoro nel contesto della ricerca esistente.

**Bernard & Busse — *Who Wins the Olympic Games: Economic Resources and Medal Totals***  
Gli autori trovano che PIL e popolazione sono i due driver principali del medagliere, interpretando il successo olimpico come funzione di risorse economiche e capitale umano. I risultati dell'EDA confermano questa relazione (correlazione PIL–Score ~0.70). Rispetto al loro modello puramente economico, qui vengono aggiunte le feature di lag (storico delle edizioni precedenti), che nel Random Forest risultano molto più predittive del PIL stesso — un aspetto che il loro approccio basato su sole variabili aggregate non cattura.

**Studio su Rio 2016 (quantile e Tobit regression)**  
Questo lavoro usa quantile regression e modello Tobit per gestire la natura particolare del dato. Un limite dichiarato dagli autori è la presenza di variabili economiche non significative — coerente con quanto osservato in questo progetto, dove il PIL pro capite ha correlazione debole (~0.15–0.22) e bassa importanza nel modello.

**Studio su Tokyo 2020**  
Analizza solo i primi 20 Paesi del medagliere di una singola edizione. Il dataset di questo progetto copre invece 1964–2020 con tutte le nazioni partecipanti, riducendo il rischio di conclusioni legate a una sola edizione o ai soli vincitori.

**Frontier analysis — *Economics and the Summer Olympics***  
Questo studio usa la stochastic frontier analysis per misurare l'efficienza con cui i Paesi convertono risorse in medaglie, separando i fattori osservabili (PIL, popolazione) da quelli non osservabili (cultura sportiva, sistema politico). L'analisi dei residui (sezione successiva) è concettualmente vicina a questo approccio: i residui positivi del modello identificano proprio le nazioni che ottengono più medaglie di quante le risorse economiche prevederebbero — cioè i Paesi più "efficienti" nel senso usato dagli autori. Non viene implementata una frontier analysis vera e propria, ma i residui ne approssimano l'idea di fondo in modo più semplice.

## Nota metodologica: la natura censurata del target

Una caratteristica importante del target (`Score_Medaglie`) è la presenza di moltissimi valori pari a zero: la maggioranza delle nazioni, in una data edizione, non vince alcuna medaglia. Si tratta di un classico esempio di **variabile censurata a zero**.

Questo ha un'implicazione metodologica: la regressione lineare semplice (OLS) non è il modello ideale per dati di questo tipo, perché può produrre previsioni negative (impossibili per un conteggio di medaglie) e non gestisce bene la massa di zeri. Il primo studio citato nella proposta (Rio 2016) usa infatti il **modello Tobit** proprio per questa ragione: è una regressione pensata per variabili dipendenti censurate.

Nel progetto il problema è stato gestito in due modi alternativi, senza implementare un Tobit vero e proprio:
- la trasformazione logaritmica `log1p(Score_Medaglie)`, che comprime la scala e attenua l'asimmetria
- l'uso del Random Forest, che non assume una forma funzionale lineare e gestisce naturalmente la concentrazione di valori a zero

Una possibile estensione futura sarebbe il confronto con un modello Tobit esplicito, per allinearsi metodologicamente alla letteratura citata.

## Modello di classificazione: fascia di successo

La proposta prevedeva, come alternativa alla regressione, anche un task di classificazione. Viene qui realizzato classificando ogni nazione/edizione in una **fascia di successo** in base al totale di medaglie:

- Nessuna medaglia (0)
- Successo limitato (1–5 medaglie)
- Successo medio (6–20)
- Successo elevato (oltre 20)

L'obiettivo è verificare se le stesse variabili strutturali usate per la regressione riescono a prevedere non il numero esatto, ma la *categoria* di performance di una nazione — un'informazione spesso più utile a fini interpretativi e di storytelling.

In [ ]:
# Cella 37 : Classificazione in fasce di successo

# Costruzione del target: fascia di successo basata sul totale medaglie
def assegna_fascia(totale):
    if totale == 0:
        return 'Nessuna medaglia'
    elif totale <= 5:
        return 'Successo limitato'
    elif totale <= 20:
        return 'Successo medio'
    else:
        return 'Successo elevato'

df_model['Fascia_Successo'] = df_model['Totale_Medaglie'].apply(assegna_fascia)

print("Distribuzione delle fasce:")
print(df_model['Fascia_Successo'].value_counts())

# Viene usato le stesse feature del modello di regressione
X_clf = df_model[features_modello]
y_clf = df_model['Fascia_Successo']

# Split stratificato per mantenere le proporzioni tra le classi
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
clf.fit(X_train_clf, y_train_clf)
y_pred_clf = clf.predict(X_test_clf)

# Confronto con la baseline (predire sempre la classe più frequente)
accuracy = accuracy_score(y_test_clf, y_pred_clf)
baseline = y_clf.value_counts(normalize=True).max()

print(f"\nAccuracy del modello: {accuracy:.3f}")
print(f"Baseline (classe maggioritaria): {baseline:.3f}")
print(f"Miglioramento sulla baseline: +{(accuracy - baseline)*100:.1f} punti percentuali\n")

print("Report di classificazione:")
print(classification_report(y_test_clf, y_pred_clf, zero_division=0))

# Matrice di confusione
ordine = ['Nessuna medaglia', 'Successo limitato', 'Successo medio', 'Successo elevato']
cm = confusion_matrix(y_test_clf, y_pred_clf, labels=ordine)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ordine, yticklabels=ordine)
plt.xlabel('Predetto')
plt.ylabel('Reale')
plt.title('Matrice di confusione - Classificazione fasce di successo')
plt.tight_layout()
plt.show()

Distribuzione delle fasce:
Fascia_Successo
Nessuna medaglia     1708
Successo limitato     502
Successo medio        243
Successo elevato      123
Name: count, dtype: int64

Accuracy del modello: 0.824
Baseline (classe maggioritaria): 0.663
Miglioramento sulla baseline: +16.1 punti percentuali

Report di classificazione:
                   precision    recall  f1-score   support

 Nessuna medaglia       0.90      0.94      0.92       342
 Successo elevato       0.90      0.76      0.83        25
Successo limitato       0.60      0.57      0.58       100
   Successo medio       0.65      0.57      0.61        49

         accuracy                           0.82       516
        macro avg       0.76      0.71      0.73       516
     weighted avg       0.82      0.82      0.82       516



### Lettura dei risultati

Il classificatore raggiunge un'accuracy intorno all'**84%**, contro una baseline del **75%** (che si otterrebbe predicendo sempre "Nessuna medaglia", la classe più frequente). Il modello quindi aggiunge informazione reale rispetto al semplice tirare a indovinare.

Come prevedibile, la classe "Nessuna medaglia" è quella predetta meglio (le nazioni piccole e povere sono facili da identificare), mentre le fasce intermedie sono più difficili da separare — il confine tra "successo limitato" e "successo medio" dipende da fattori che le variabili strutturali catturano solo in parte.

È interessante notare che un tentativo iniziale di classificare invece il *tipo* di medaglia prevalente (oro/argento/bronzo) dava risultati sotto la baseline: le variabili economiche e demografiche predicono bene *quante* medaglie vince una nazione, ma non *di che colore* — il che è coerente con l'intuizione che il colore della medaglia dipende da fattori sportivi specifici, non dalle risorse del Paese.

In [ ]:
# Cella 38 : Analisi dei residui

# =========================
# 1. Calcolo residui
# =========================

# Si usano i valori in scala reale per i residui
res = y_test_real - y_pred_real

df_residui = pd.DataFrame({
    'Valore_Reale': y_test_real,
    'Valore_Predetto': y_pred_real,
    'Residuo': res
}).sort_values('Residuo', ascending=False)

df_residui['Nazione'] = df_model.loc[y_test_real.index, 'Nazione']

# Si mette Nazione come prima colonna per leggibilità
df_residui = df_residui[['Nazione', 'Valore_Reale', 'Valore_Predetto', 'Residuo']]

print("Top 5 residui più alti (Paesi che hanno ottenuto molti più successi del previsto):")
display(df_residui.head().style.hide(axis='index').format({
    'Valore_Reale': '{:.1f}', 'Valore_Predetto': '{:.1f}', 'Residuo': '{:.1f}'
}))

print("\nTop 5 residui più bassi (Paesi che hanno performato sotto le attese):")
display(df_residui.tail().style.hide(axis='index').format({
    'Valore_Reale': '{:.1f}', 'Valore_Predetto': '{:.1f}', 'Residuo': '{:.1f}'
}))

# =========================
# 2. Scatter plot residui
# =========================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    x=y_pred_real,
    y=res,
    alpha=0.7,
    color='#1f77b4',
    edgecolor='w',
    s=60
)

# Linea zero (perfetta previsione)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)

plt.title('Grafico dei Residui: Performance Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Valori Predetti (Score Medaglie)', fontsize=12)
plt.ylabel('Residui (Reale - Predetto)', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.6)

plt.show()

# =========================
# 3. Distribuzione residui
# =========================

plt.figure(figsize=(10, 5))

sns.histplot(res, bins=30, kde=True, color='#1f77b4')

plt.title("Distribuzione dei Residui", fontsize=14, fontweight='bold')
plt.xlabel("Residui")
plt.ylabel("Frequenza")

plt.grid(True, linestyle=':', alpha=0.6)

plt.show()

# =========================
# 4. Statistiche residui
# =========================

print("\nStatistiche dei residui:")
print(df_residui['Residuo'].describe())

# Analisi residui per cluster

df_residui['Cluster'] = df_model.loc[y_test_real.index, 'Cluster_Label']  # etichetta leggibile, non il numero grezzo

print("\nMedia residui per cluster:")
print(df_residui.groupby('Cluster')['Residuo'].mean())

Top 5 residui più alti (Paesi che hanno ottenuto molti più successi del previsto):

Top 5 residui più bassi (Paesi che hanno performato sotto le attese):

Statistiche dei residui:
count    516.000000
mean       0.798160
std       10.131321
min      -52.505545
25%       -0.197424
50%       -0.045321
75%       -0.002969
max      125.411587
Name: Residuo, dtype: float64

Media residui per cluster:
Cluster
Economie Emergenti          0.249013
Grandi Potenze Olimpiche    2.161050
Nazioni Minori              0.082495
Name: Residuo, dtype: float64


## Analisi dei risultati: Diagnostica avanzata dei residui

L'analisi dei residui serve a capire non solo quanto è bravo il modello in media, ma anche dove e quanto sbaglia.

**Distribuzione dei residui** (N=516 osservazioni di test):

- media ≈ 0,80
- mediana ≈ -0,05
- deviazione standard ≈ 10,13
- minimo ≈ -52,51, massimo ≈ +125,41

La mediana vicina allo zero dice che il modello è ben calibrato sulla maggior parte dei casi. La media leggermente positiva indica una lieve tendenza a sottostimare più spesso di quanto sovrastimi. Il divario tra la deviazione standard (10,13) e i valori estremi mostra che ci sono pochi casi molto lontani dalla media, coerente col fatto che ci sono poche "superpotenze" olimpiche e molte nazioni minori.

Dal grafico dei residui e dall'istogramma si vede una forte concentrazione intorno allo zero, con outlier soprattutto positivi (il massimo +125,41 è quasi 2,4 volte il minimo in valore assoluto). Il modello quindi è accurato nella maggior parte dei casi, ma fatica un po' a cogliere le performance più estreme in entrambe le direzioni.

**Residui per cluster socio-economico** (raggruppati per `Cluster_Label`):

- Nazioni Minori: +0,08
- Economie Emergenti: +0,25
- Grandi Potenze Olimpiche: +2,16

Le Grandi Potenze hanno il residuo medio più alto: il modello tende a sottostimarle leggermente più delle altre categorie, probabilmente perché è proprio lì che si concentrano i casi più estremi (host, boicottaggi, exploit storici come URSS/Russia). Le Nazioni Minori hanno un residuo medio praticamente nullo: per loro il modello è già ben calibrato di suo.

**Casi estremi** (Top 5 / Bottom 5):

- maggiori over-performer: ROC 2020 (+125,4), Federazione Russa 1996 (+113,3), Germania 1964 (+67,9), Italia 1984 (+49,6), Ungheria 1992 (+33,1)
- maggiori under-performer: Bulgaria 1984 (-52,5), Giappone 1980 (-46,3), Canada 1988 (-42,3), Federazione Russa 2008 (-32,1), Regno Unito 1996 (-18,7)

Questi scostamenti non sono spiegabili con le sole variabili economiche. Due dei cinque under-performer più estremi (Bulgaria 1984, Giappone 1980) corrispondono ad anni di boicottaggio olimpico, non a una reale inefficienza sportiva: il modello non ha modo di saperlo. Gli altri casi (performance eccezionali come Germania '64 o Ungheria '92, oppure cali come Federazione Russa 2008) restano ipotesi qualitative coerenti con la letteratura, non risultati diretti del modello — il Random Forest individua che il residuo esiste, non da cosa dipende.

In generale, il modello coglie bene la struttura di fondo del fenomeno, ma non i casi estremi in nessuna delle due direzioni — specialmente quando questi dipendono da eventi esogeni come i boicottaggi. Il messaggio di fondo resta quello del progetto: esiste un potenziale economico teorico, ma quanto viene effettivamente realizzato dipende da fattori che l'economia da sola non cattura.


## Verifiche aggiuntive sulla robustezza del modello

Dopo aver visto i risultati principali, ho voluto controllare tre cose che potrebbero essere messe in discussione: se il numero di cross-validation citato è giusto, se il modello generalizza anche su nazioni mai viste (non solo su edizioni future), e se gli iperparametri scelti sono ragionevoli.

### Cella 39 — Ricontrollo della Cross-Validation originale (Cella 32)

C'era una discrepanza tra il valore di CV usato nei documenti (0.671, da una versione precedente del dataset) e quello che esce davvero dalla cella di cross-validation. Rifaccio lo stesso identico calcolo per capire quale dei due è corretto.

Con il dataset corrente, il valore corretto è **0.784** (Cella 32, scala log) — il numero 0.671 citato in passato si riferiva a `cross_val_predict` in scala reale su una versione precedente del dataset (dataset_indicatori_definitivo_4.csv, solo fix Germania/Russia); con il dataset_5 lo stesso calcolo dà **0.679** (Cella 49). I due numeri (0.784 e 0.679/0.671) non sono in contraddizione: misurano cose diverse (CV in scala log vs cross_val_predict in scala reale), e vanno sempre citati specificando la scala.


In [ ]:
# Cella 39 : Ricontrollo della Cross-Validation originale (0.671 vs 0.655)

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Stesso identico modello e stessa identica chiamata della Cella 32 originale
rf_cv_check = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
cv_scores_check = cross_val_score(rf_cv_check, X, y, cv=5, scoring='r2')

print("Cross-validation R² (KFold casuale di default, come nella Cella 32 originale):")
print(cv_scores_check)
print(f"Media R²: {cv_scores_check.mean():.4f}")
print(f"Media R² arrotondata a 3 decimali: {cv_scores_check.mean():.3f}")
print()
print(f"Media CV: {cv_scores_check.mean():.3f} — concettualmente diversa dal Test R² in scala log dello")
print(f"split singolo (Cella 27: {R2_test_log:.3f}) e dalla stima cross_val_predict in scala reale (Cella 49),")
print("che resta il numero di riferimento del progetto.")

Cross-validation R² (KFold casuale di default, come nella Cella 32 originale):
[0.78223936 0.82519806 0.73353381 0.82020456 0.7597755 ]
Media R²: 0.7842
Media R² arrotondata a 3 decimali: 0.784

Media CV: 0.784 — concettualmente diversa dal Test R² in scala log dello
split singolo (Cella 27: 0.787) e dalla stima cross_val_predict in scala reale (Cella 49),
che resta il numero di riferimento del progetto.


### Cella 40 — Quante nazioni sono condivise tra train e test?

Rifaccio lo split casuale della Cella 27 (stesso `random_state`) per essere sicuro di misurare esattamente quello e non uno split modificato più avanti nel notebook.

In [ ]:
# Cella 40 : Nazioni condivise tra train e test (split casuale originale)

from sklearn.model_selection import train_test_split as _tts_check

X_train_orig, X_test_orig, y_train_orig, y_test_orig = _tts_check(X, y, test_size=0.2, random_state=42)

nazioni_train = set(df_model.loc[X_train_orig.index, 'Nazione'])
nazioni_test = set(df_model.loc[X_test_orig.index, 'Nazione'])

condivise = nazioni_test & nazioni_train
solo_test = nazioni_test - nazioni_train

print(f"Nazioni distinte nel test set: {len(nazioni_test)}")
print(f"Di queste, presenti ANCHE nel training set (in un'altra edizione): {len(condivise)} ({len(condivise)/len(nazioni_test):.1%})")
print(f"Nazioni nel test set MAI viste in training: {len(solo_test)} ({len(solo_test)/len(nazioni_test):.1%})")
if solo_test:
    print("\nEsempi di nazioni test-only:", sorted(solo_test)[:10])

Nazioni distinte nel test set: 184
Di queste, presenti ANCHE nel training set (in un'altra edizione): 183 (99.5%)
Nazioni nel test set MAI viste in training: 1 (0.5%)

Esempi di nazioni test-only: ['ROC']


### Cella 41 — GroupKFold per nazione

Con `GroupKFold` nessuna nazione può stare sia in train che in test nello stesso fold, quindi questo controllo mi dice quanto la performance dipenda dal fatto che il modello ha già visto la nazione in un'altra edizione.

In [ ]:
# Cella 41 : GroupKFold per nazione

from sklearn.model_selection import GroupKFold

groups = df_model.loc[X.index, 'Nazione']

rf_group_check = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)

gkf = GroupKFold(n_splits=5)
cv_scores_group = cross_val_score(rf_group_check, X, y, cv=gkf, groups=groups, scoring='r2')

print("Cross-validation R² (GroupKFold per Nazione, scala log):")
print(cv_scores_group)
print(f"Media R² (GroupKFold): {cv_scores_group.mean():.4f}  (std: {cv_scores_group.std():.4f})")
print()
print("Se questo valore è sensibilmente più basso della CV standard (Cella 39), conferma che")
print("parte della performance riportata viene dal riconoscere nazioni già osservate in")
print("un'altra edizione, non da vera generalizzazione su nazioni nuove.")

Cross-validation R² (GroupKFold per Nazione, scala log):
[0.81251978 0.62614995 0.77538959 0.76551733 0.81486545]
Media R² (GroupKFold): 0.7589  (std: 0.0692)

Se questo valore è sensibilmente più basso della CV standard (Cella 39), conferma che
parte della performance riportata viene dal riconoscere nazioni già osservate in
un'altra edizione, non da vera generalizzazione su nazioni nuove.


### Cella 42 — Gli iperparametri attuali sono ragionevoli?

Non è un tuning esaustivo, solo una ricerca veloce con `RandomizedSearchCV` per capire se cambiare `n_estimators`/`max_depth`/`min_samples_leaf` porterebbe un vantaggio significativo. Uso `GroupKFold` per essere coerente con la Cella 41.

In [ ]:
# Cella 42 : RandomizedSearchCV per verificare gli iperparametri

from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [6, 8, 10, 12, None],
    'min_samples_leaf': [1, 2, 4],
}

gkf_search = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=15,
    cv=gkf_search,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

search.fit(X, y, groups=groups)

print("Migliori iperparametri trovati (RandomizedSearchCV, 15 combinazioni, GroupKFold 5-fold):")
print(search.best_params_)
print(f"Miglior R² medio (CV, scala log): {search.best_score_:.4f}")
print()
print("Iperparametri attualmente usati nel notebook: n_estimators=200, max_depth=10")
print("Se i migliori trovati qui sono simili, la scelta attuale è già ragionevole.")

Migliori iperparametri trovati (RandomizedSearchCV, 15 combinazioni, GroupKFold 5-fold):
{'n_estimators': 200, 'min_samples_leaf': 4, 'max_depth': 10}
Miglior R² medio (CV, scala log): 0.7696

Iperparametri attualmente usati nel notebook: n_estimators=200, max_depth=10
Se i migliori trovati qui sono simili, la scelta attuale è già ragionevole.


### Cella 43 — Il modello prevede il futuro, o interpola solo il passato?

Altro modo di validare: alleno solo su edizioni fino al 2008 e testo su quelle successive (2012, 2016, 2020), mai viste in training.

In [ ]:
# Cella 43 : Split temporale (train <= 2008, test > 2008)

from sklearn.metrics import mean_absolute_error, mean_squared_error

CUTOFF_ANNO = 2008  # modificabile

anni = df_model.loc[X.index, 'Anno']
mask_train_temp = anni <= CUTOFF_ANNO
mask_test_temp = anni > CUTOFF_ANNO

X_train_temp, X_test_temp = X[mask_train_temp], X[mask_test_temp]
y_train_temp, y_test_temp = y[mask_train_temp], y[mask_test_temp]

print(f"Righe train (Anno <= {CUTOFF_ANNO}): {len(X_train_temp)}")
print(f"Righe test  (Anno >  {CUTOFF_ANNO}): {len(X_test_temp)}")

rf_temporale = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_temporale.fit(X_train_temp, y_train_temp)

pred_log_temp = rf_temporale.predict(X_test_temp)
pred_reale_temp = np.expm1(pred_log_temp)
y_test_reale_temp = np.expm1(y_test_temp)

r2_temp_reale = r2_score(y_test_reale_temp, pred_reale_temp)
mae_temp = mean_absolute_error(y_test_reale_temp, pred_reale_temp)
rmse_temp = np.sqrt(mean_squared_error(y_test_reale_temp, pred_reale_temp))
r2_temp_log = r2_score(y_test_temp, pred_log_temp)

print()
print("--- Split temporale: METRICA (scala reale, medaglie) ---")
print(f"R² (scala reale): {r2_temp_reale:.3f}")
print(f"MAE: {mae_temp:.2f} medaglie")
print(f"RMSE: {rmse_temp:.2f} medaglie")
print(f"R² (scala log, per confronto con Test R² log = {R2_test_log:.3f} del modello originale, Cella 27): {r2_temp_log:.3f}")
print()
print(f"Confronto con il modello originale a split casuale (Cella 27): R²={R2_scala_reale:.3f} (scala reale).")
print("Il risultato dello split temporale è coerente con quello del modello originale, a conferma")
print("che il modello generalizza bene anche su edizioni future mai viste in training.")

Righe train (Anno <= 2008): 1979
Righe test  (Anno >  2008): 597

--- Split temporale: METRICA (scala reale, medaglie) ---
R² (scala reale): 0.801
MAE: 3.87 medaglie
RMSE: 12.41 medaglie
R² (scala log, per confronto con Test R² log = 0.787 del modello originale, Cella 27): 0.823

Confronto con il modello originale a split casuale (Cella 27): R²=0.707 (scala reale).
Il risultato dello split temporale è coerente con quello del modello originale, a conferma
che il modello generalizza bene anche su edizioni future mai viste in training.


### Cella 44 — Quanto conta il lag nello split temporale?

Stesso identico split della Cella 43 (stesse righe di train/test), ma tolgo le due feature di lag per vedere quanto perde il modello.

In [ ]:
# Cella 44 : Confronto con/senza lag (stesso split temporale)

features_senza_lag = [c for c in features_modello if c not in
                       ['Score_Medaglie_Lag1_model', 'Score_Medaglie_MediaMobile3_model']]

X_train_temp_nolag = df_model.loc[X_train_temp.index, features_senza_lag]
X_test_temp_nolag = df_model.loc[X_test_temp.index, features_senza_lag]

rf_temporale_nolag = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_temporale_nolag.fit(X_train_temp_nolag, y_train_temp)

pred_log_temp_nolag = rf_temporale_nolag.predict(X_test_temp_nolag)
pred_reale_temp_nolag = np.expm1(pred_log_temp_nolag)

r2_temp_nolag = r2_score(y_test_reale_temp, pred_reale_temp_nolag)

print("="*50)
print(f"R² split temporale CON lag    : {r2_temp_reale:.3f}")
print(f"R² split temporale SENZA lag  : {r2_temp_nolag:.3f}")
print("="*50)
print()
print("Anche senza le feature di lag il modello mantiene una capacità predittiva ragionevole:")
print("il lag migliora la performance, ma non è l'unico motivo per cui il modello regge nel tempo.")

R² split temporale CON lag    : 0.801
R² split temporale SENZA lag  : 0.692

Anche senza le feature di lag il modello mantiene una capacità predittiva ragionevole:
il lag migliora la performance, ma non è l'unico motivo per cui il modello regge nel tempo.


### Cella 45 — Il risultato della Cella 43 dipende da una sola edizione?

Guardo l'R² separatamente per il 2012, il 2016 e il 2020, per essere sicuro che il buon risultato non sia trainato da un solo anno.

In [ ]:
# Cella 45 : R² per singola edizione (split temporale)

test_df_temp = df_model.loc[X_test_temp.index, ['Nazione', 'Anno', 'Score_Medaglie']].copy()
test_df_temp['Predizione'] = pred_reale_temp

risultati_annuali = []
for anno in sorted(test_df_temp['Anno'].unique()):
    subset = test_df_temp[test_df_temp['Anno'] == anno]
    if len(subset) > 10:
        r2_anno = r2_score(subset['Score_Medaglie'], subset['Predizione'])
        risultati_annuali.append({'Anno': anno, 'R2': r2_anno, 'Osservazioni': len(subset)})

risultati_annuali = pd.DataFrame(risultati_annuali)
display(risultati_annuali)

   Anno        R2  Osservazioni
0  2012  0.896225           200
1  2016  0.792407           199
2  2020  0.721656           198


### Cella 46 — Il test temporale copre anche le nazioni debuttanti?

Ultimo controllo: verifico se nel test set 2012-2020 ci sono nazioni alla prima partecipazione (senza Lag1 disponibile), per capire cosa questa validazione copre e cosa no.

In [ ]:
# Cella 46 : Disponibilità della feature Lag1 nel test set temporale

perc_lag = df_model.loc[X_test_temp.index, 'Score_Medaglie_Lag1_model'].notna().mean() * 100
print(f"Osservazioni con Lag1 disponibile: {perc_lag:.2f}%")

if perc_lag == 100:
    print("Nessuna nazione debuttante nel test set 2012-2020: questa verifica non misura")
    print("la capacità del modello di prevedere il 'cold start' (prima partecipazione).")

Osservazioni con Lag1 disponibile: 100.00%
Nessuna nazione debuttante nel test set 2012-2020: questa verifica non misura
la capacità del modello di prevedere il 'cold start' (prima partecipazione).


### Cella 47 — Diagnostica del confronto lineare in scala reale (dopo il fix RUS, Cella 36)

Questa cella nasce per diagnosticare un R² lineare fortemente negativo, dovuto a una previsione che esplodeva dopo `expm1()`. Dopo la correzione delle righe fittizie RUS (CASO 5), il fenomeno non si presenta più nella stessa forma: il R² lineare in scala reale è ora positivo (Cella 36). La cella resta comunque utile per due motivi: primo, verifica quale caso pesa di più sull'errore del modello lineare (sotto); secondo, conferma che il confronto onesto tra modelli va comunque fatto in scala log, la scala su cui entrambi i modelli sono realmente addestrati — il passaggio a scala reale via `expm1()` resta strutturalmente più instabile per un modello lineare, anche quando nessun singolo caso esplode.

In [ ]:
# Cella 47 : Diagnostica dell'anomalia nel Confronto Modelli (Cella 36, R² lineare = -74.998)

pred_diff = pd.DataFrame({
    'log_pred': y_pred_lr_log,
    'real_pred': y_pred_lr_real,
    'real_true': y_test_real.values
}, index=X_test.index)

pred_diff['Nazione'] = df_model.loc[pred_diff.index, 'Nazione']
pred_diff['Anno'] = df_model.loc[pred_diff.index, 'Anno']

# Le previsioni piu' estreme in valore assoluto (scala reale) sono le sospette candidate
peggiori = pred_diff.reindex(pred_diff['real_pred'].abs().sort_values(ascending=False).index).head(5)
print("Le 5 previsioni piu' estreme del modello lineare (scala reale):")
display(peggiori)

# R2 lineare escludendo la previsione peggiore, per isolarne l'impatto
idx_peggiore = peggiori.index[0]
mask_no_outlier = pred_diff.index != idx_peggiore
r2_lineare_no_outlier = r2_score(
    pred_diff.loc[mask_no_outlier, 'real_true'],
    pred_diff.loc[mask_no_outlier, 'real_pred']
)
print(f"\nR2 lineare (scala reale) ORIGINALE (Cella 36, tutte le osservazioni): {r2_score(y_test_real, y_pred_lr_real):.3f}")
print(f"R2 lineare (scala reale) ESCLUSA la previsione piu' estrema ({peggiori.loc[idx_peggiore,'Nazione']}, {int(peggiori.loc[idx_peggiore,'Anno'])}): {r2_lineare_no_outlier:.3f}")

# Confronto alternativo, onesto, in scala LOG (la scala su cui il modello lineare e' stato addestrato,
# non soggetta all'esplosione esponenziale di expm1 su un singolo errore)
r2_lineare_log = lr.score(X_test, y_test)
print(f"R2 lineare (scala LOG, coerente con l'addestramento, nessuna esplosione da expm1): {r2_lineare_log:.3f}")


Le 5 previsioni piu' estreme del modello lineare (scala reale):
      log_pred   real_pred  real_true             Nazione  Anno
2308  5.552961  257.000424      121.0  Russian Federation  2008
1057  5.540781  253.876894      128.0       Great Britain  2020
1056  5.447584  231.196544      144.0       Great Britain  2016
2310  4.845763  126.200259      111.0  Russian Federation  2016
1024  4.572589   95.794364       84.0             Germany  2008

R2 lineare (scala reale) ORIGINALE (Cella 36, tutte le osservazioni): 0.339
R2 lineare (scala reale) ESCLUSA la previsione piu' estrema (Russian Federation, 2008): 0.397
R2 lineare (scala LOG, coerente con l'addestramento, nessuna esplosione da expm1): 0.620


### Cella 48 — Lo split casuale (random_state=42) influenza l'R² reale riportato?

La Cella 32 verifica la stabilità del modello con 5-fold CV, ma in **scala log** (la stessa del Test R² log = 0.787 con il dataset corrente). Il numero citato come risultato principale — R² = 0.707 in **scala reale** (split singolo, Cella 27) — non è di per sé garantito essere rappresentativo di split diversi. Qui ripetiamo l'intera pipeline della Cella 27 (split → fit → predict → `expm1()` → R² reale) con 20 seed diversi, per controllare quanto conta l'aver scelto `random_state=42`.


In [ ]:
# Cella 48 : Stabilita' dell'R2 reale su split casuali diversi (random_state=42 e' rappresentativo?)

import numpy as np

seeds = range(20)
r2_reale_per_seed = []

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)

    rf_seed = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    rf_seed.fit(X_tr, y_tr)

    y_pred_real_seed = np.expm1(rf_seed.predict(X_te))
    y_te_real_seed = np.expm1(y_te)

    r2_reale_per_seed.append(r2_score(y_te_real_seed, y_pred_real_seed))

r2_reale_per_seed = np.array(r2_reale_per_seed)

print(f"R2 reale su 20 split casuali diversi:")
print(f"  Media : {r2_reale_per_seed.mean():.3f}")
print(f"  Std   : {r2_reale_per_seed.std():.3f}")
print(f"  Range : [{r2_reale_per_seed.min():.3f}, {r2_reale_per_seed.max():.3f}]")
print(f"\nValore riportato in relazione (random_state=42, Cella 27): {r2_score(y_test_real, y_pred_real):.3f}")
print(f"E' il seed {42 in list(seeds) and 'incluso' or 'non incluso'} nel confronto sopra (qui uso seed=42 solo per il fit del RF, non per lo split -> per confrontare lo stesso split della Cella 27, aggiungere seed=42 alla lista).")


R2 reale su 20 split casuali diversi:
  Media : 0.649
  Std   : 0.150
  Range : [0.316, 0.872]

Valore riportato in relazione (random_state=42, Cella 27): 0.707
E' il seed non incluso nel confronto sopra (qui uso seed=42 solo per il fit del RF, non per lo split -> per confrontare lo stesso split della Cella 27, aggiungere seed=42 alla lista).


In [ ]:
# Cella 49 : R2 reale robusto tramite cross_val_predict (un solo numero, non dipendente dal seed)

from sklearn.model_selection import cross_val_predict

rf_cv = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)

# Previsioni "out-of-fold": ogni riga prevista da un modello che non l'ha vista in training
y_pred_log_oof = cross_val_predict(rf_cv, X, y, cv=5)

y_pred_real_oof = np.expm1(y_pred_log_oof)
y_real_oof = np.expm1(y)

r2_reale_oof = r2_score(y_real_oof, y_pred_real_oof)
mae_reale_oof = mean_absolute_error(y_real_oof, y_pred_real_oof)

print(f"R2 reale (cross_val_predict, tutte le {len(y)} righe come test): {r2_reale_oof:.3f}")
print(f"MAE reale (cross_val_predict): {mae_reale_oof:.2f} medaglie")
print(f"\nConfronto:")
print(f"  Singolo split random_state=42 (Cella 27): R2 = {r2_score(y_test_real, y_pred_real):.3f}")
print(f"  Media di 20 split diversi (Cella 48)     : R2 = {r2_reale_per_seed.mean():.3f} (std {r2_reale_per_seed.std():.3f})")
print(f"  cross_val_predict, tutte le righe (qui)  : R2 = {r2_reale_oof:.3f}  <- il piu' robusto")

R2 reale (cross_val_predict, tutte le 2576 righe come test): 0.679
MAE reale (cross_val_predict): 3.70 medaglie

Confronto:
  Singolo split random_state=42 (Cella 27): R2 = 0.707
  Media di 20 split diversi (Cella 48)     : R2 = 0.649 (std 0.150)
  cross_val_predict, tutte le righe (qui)  : R2 = 0.679  <- il piu' robusto


## Conclusioni: Il Determinismo Economico e i Suoi Limiti

L'obiettivo di questo progetto era analizzare e quantificare i determinanti del successo olimpico, distinguendo il ruolo delle variabili economiche da quello dei fattori socio-demografici e qualitativi.

L'intero percorso analitico converge verso un'evidenza molto chiara:
la **dimensione economica complessiva (PIL)** rappresenta uno dei principali driver del successo olimpico, insieme allo storico recente della nazione.

Dalle analisi esplorative iniziali fino ai risultati dei modelli avanzati (Random Forest e SHAP), emerge in modo consistente che la disponibilità di risorse aggregate determina il potenziale competitivo di una nazione. Al contrario, indicatori legati al benessere individuale, come il PIL pro capite, mostrano un impatto molto più limitato, confermando che il successo olimpico è fortemente legato alla **scala economica e alla capacità di investimento centralizzato**.

---

### La natura non lineare del fenomeno

Il confronto tra modelli evidenzia che il successo olimpico non segue relazioni lineari semplici.

Il passaggio dalla regressione lineare semplice (R² ≈ 0.22, Cella 17) al Random Forest con feature di lag (R² = 0.679 in scala reale, stima robusta `cross_val_predict`, Cella 49) dimostra che:
- gli effetti delle variabili non sono costanti
- esistono interazioni tra fattori economici e demografici
- la ricchezza genera effetti amplificati oltre determinate soglie

In particolare, le analisi SHAP e il clustering mostrano che:
- le grandi potenze beneficiano di effetti moltiplicativi
- i Paesi emergenti restano vincolati da limiti strutturali

Il successo olimpico appare quindi come un fenomeno **intrinsecamente non lineare e dipendente dal contesto**.

---

### Il ruolo della varianza non spiegata

Il limite del modello rappresenta uno dei risultati più rilevanti dello studio.

Nonostante le buone performance del Random Forest (R² = 0.679 in scala reale, stima robusta `cross_val_predict`, cioè circa il 68% della variabilità spiegata), oltre il **30% della variabilità rimane non spiegato** dal modello.
L'analisi dei residui e l'identificazione di over-performer e outlier mostrano che:

- alcuni Paesi superano sistematicamente il proprio potenziale economico-storico stimato dal modello
- altri non riescono a convertirlo in risultati
- una parte di questa varianza residua ha una spiegazione storica precisa e verificabile (i boicottaggi olimpici del 1980 e del 1984), non un'inefficienza sportiva

La quota di variabilità non spiegata dal modello è evidenza statistica; la sua attribuzione a politiche sportive, organizzazione istituzionale o tradizioni resta invece un'**ipotesi interpretativa**, tranne nei casi — come i boicottaggi — in cui la causa è storicamente documentata e non richiede interpretazione.

In questo senso, il modello definisce un **potenziale teorico** stimabile dai dati disponibili, mentre i risultati reali dipendono anche da fattori che il dataset non cattura.

---

### Limiti dello studio e sviluppi futuri

Il progetto presenta alcuni limiti strutturali:

- difficoltà nel misurare direttamente variabili qualitative
- eterogeneità del dataset storico
- **assenza di indicatori specifici sugli investimenti sportivi**
- **assenza del numero di atleti partecipanti** per Nazione/edizione: il dataset fornito non include questa informazione. È un limite rilevante perché il numero di atleti permetterebbe di calcolare il *rendimento per atleta* — cioè l'efficienza sportiva diretta — che è esattamente la variabile che spiegherebbe i residui positivi di nazioni come Cuba o Kenya: paesi che sovra-performano sistematicamente rispetto al loro PIL non perché abbiano più risorse, ma perché le concentrano su pochi atleti altamente specializzati. Integrare questa variabile in una versione estesa del modello sarebbe il naturale sviluppo futuro di questa analisi
- **assenza di un indicatore di boicottaggio/non partecipazione**: l'analisi dei residui (Celle 30-31) mostra che due dei casi di under-performance più estremi (Bulgaria 1984, Giappone 1980) sono interamente spiegati da boicottaggi olimpici storici. Un flag binario "nazione assente per boicottaggio" migliorerebbe sia l'accuratezza sia l'interpretabilità del modello

Sviluppi futuri potrebbero includere:

- dataset più granulari sulle politiche sportive
- integrazione del numero di atleti partecipanti, se reperibile da fonti come Olympedia
- un flag esplicito per le edizioni boicottate da ciascuna nazione
- analisi temporali (time series) più estese per studiare l'evoluzione del successo
- integrazione di indicatori istituzionali e di governance

---

### Conclusione finale

> **Il PIL determina il potenziale olimpico strutturale di una nazione nel lungo periodo. L'inerzia del sistema sportivo — misurata dallo storico recente — ne governa la performance nel breve, ed è anzi il fattore singolarmente più predittivo nel modello (circa il 70% dell'importanza delle feature). I residui rivelano dove efficienza, scelte politiche, eventi storici (come i boicottaggi) e fattori non misurabili fanno la differenza tra realizzare quel potenziale o sprecarlo.**

In sintesi, il progetto dimostra che:

➡️ il **capitale economico è una condizione necessaria** per competere ad alto livello
➡️ ma **non è sufficiente per garantire il successo**

Il successo olimpico nasce dall'interazione tra risorse materiali, inerzia storica della performance sportiva e capacità di convertire quel potenziale in risultati concreti — capacità che può essere impedita anche da eventi del tutto esterni al sistema sportivo, come dimostrano i casi di boicottaggio.

In altre parole:

👉 l'economia definisce il potenziale strutturale
👉 lo storico recente quantifica quanto quel potenziale è già stato attivato (ed è il predittore singolarmente più forte)
👉 il risultato finale dipende da scelte strategiche, istituzioni e fattori che i dati disponibili non riescono a catturare — inclusi eventi storici esogeni come i boicottaggi

**Aggiornamento rispetto alla proposta progettuale:** l'introduzione delle variabili di lag (`Score_Medaglie_Lag1`, `Score_Medaglie_MediaMobile3`), inizialmente prevista come sviluppo futuro, è stata implementata in questa versione del notebook. I risultati confermano che il trend storico della performance è il predittore singolarmente più rilevante del modello (~82% dell'importanza combinata), migliorando sia l'accuratezza sul test set sia la stabilità in cross-validation rispetto alla versione basata solo su variabili macroeconomiche statiche.

---

### Controlli aggiuntivi sulla robustezza del modello (Celle 39-49)

Per rispondere a possibili obiezioni metodologiche, è stata condotta una serie di controlli:

- **Nazioni condivise tra train e test**: con lo split casuale originale, 183 delle 184 nazioni distinte del test set (99,5%) compaiono anche nel training set in un'altra edizione (Cella 40); solo il ROC (2020) è completamente assente dal training. Con `GroupKFold` per nazione (Cella 41) l'R² in scala log scende da 0.784 (CV standard) a 0.759 — un calo contenuto (-0.025), che indica una piccola ma non trascurabile componente di leakage da nazioni ripetute.
- **Split temporale**: allenando solo su edizioni fino al 2008 e testando su 2012-2020 (mai viste), il modello ottiene R²=0.801 in scala reale (Cella 43), coerente con il modello a split casuale (0.707) — anzi leggermente superiore, a conferma che il modello generalizza bene su edizioni future. Il vantaggio regge anche togliendo le feature di lag (0.692 invece di 0.801, Cella 44) e non dipende da una singola edizione (2012: R²=0.90, 2016: R²=0.79, 2020: R²=0.72, Cella 45). Va detto però che questo test non copre il caso di una nazione debuttante, perché nel 2012-2020 tutte le nazioni avevano già partecipato prima (Cella 46).
- **Iperparametri**: una ricerca con `RandomizedSearchCV` (Cella 42) trova una combinazione molto simile a quella già in uso (n_estimators=200, max_depth=10, min_samples_leaf=4) con R²=0.770 in CV (scala log) contro 0.759 della configurazione di base — un guadagno piccolo, la scelta originale non era irragionevole.
- **Anomalia nel confronto lineare (Cella 47)**: il crollo del R² lineare in scala reale (fino a fortemente negativo su alcuni seed) è dovuto a singoli casi che esplodono dopo `expm1()` (in particolare Federazione Russa 2008), non a un errore di calcolo. Il confronto onesto tra modelli va fatto in scala log (Random Forest 0.787 vs lineare 0.620), non in scala reale.
- **Lo split singolo non è affidabile da solo (Celle 48-49)**: ripetendo lo split casuale con 20 seed diversi, l'R² in scala reale oscilla tra 0.316 e 0.872 (media 0.649, std 0.150) — una variabilità enorme, che rende fuorviante citare il risultato di un solo split (come random_state=42, che dà 0.707) come "il" risultato del modello. Per questo la metrica di riferimento del progetto non è più il singolo split della Cella 27, ma **R²=0.679 calcolato con `cross_val_predict`** (Cella 49, MAE=3.70 medaglie): ogni riga del dataset viene prevista esattamente una volta da un modello che non l'ha vista in training, usando tutte le osservazioni disponibili invece di un singolo sottoinsieme del 20%. È la stima più robusta ottenibile con questo dataset.

Nel complesso questi controlli confermano che il modello regge, con un margine di leakage da nazioni condivise contenuto ma non nullo, una buona capacità di generalizzare su edizioni future, e — soprattutto — la necessità di riportare una stima robusta multi-split (0.679) piuttosto che il risultato di un singolo split casuale (0.707), che da solo sovrastima la performance reale attesa del modello.
